# Auditoria Pedagógica — Planejado vs. Executado

> **Portfolio pessoal.** Este notebook foi extraído de um projeto real de automação de processos acadêmicos e passou por um processo de anonimização: nomes de instituições, pessoas e caminhos locais foram substituídos por termos genéricos, e todos os outputs de execução (que continham dados reais de alunos/professores) foram removidos. O código e a lógica de negócio foram mantidos como estavam em produção.

Cruza o que foi planejado (mapa pedagógico) com o que foi de fato executado (registro de professores), apontando divergências e gerando relatório executivo.

---


# Auditoria pedagógica — Planejado versus Executado



# SGA - Auditoria Pedagógica

Cruza o Mapa Pedagógico (planejado) com o Registro (executado) e a base de professores, e gera um único conjunto de saídas por execução:

- **1 Excel consolidado** — todas as abas analíticas (ETL, comparações, docentes, governança) em um único arquivo.
- **1 Dashboard HTML executivo** — visão interativa para navegação rápida.
- **1 Relatório PDF executivo** — capa com indicadores-chave, leitura textual e detalhamento por seção, com cabeçalho/rodapé padronizados em todas as páginas.

Entradas esperadas na pasta do notebook (o nome pode variar levemente, a busca é por padrão): Mapa Pedagógico (.xlsx), Registro de Professores (.xlsx) e base de Professores (.xlsx).


## Execução

Execute as células em ordem no JupyterLab.

In [ ]:
# ============================================================
# 0. Verificação e instalação automática de dependências
# ============================================================

import sys
import subprocess
import importlib.util

def pacote_instalado(nome_import: str) -> bool:
    """Verifica se um pacote está disponível para importação."""
    return importlib.util.find_spec(nome_import) is not None

def instalar_pacote(nome_pip: str) -> None:
    """Instala pacote via pip no kernel atual."""
    print(f"Instalando dependência ausente: {nome_pip}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", nome_pip])

DEPENDENCIAS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "xlsxwriter": "xlsxwriter",
    "plotly": "plotly",
    "reportlab": "reportlab",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
    "nbformat": "nbformat",
}

for import_name, pip_name in DEPENDENCIAS.items():
    if not pacote_instalado(import_name):
        try:
            instalar_pacote(pip_name)
        except Exception as e:
            print(f"Não foi possível instalar {pip_name} automaticamente: {e}")
            print("Se estiver em ambiente corporativo bloqueado, instale manualmente com:")
            print(f"    pip install {pip_name}")

print("Verificação de dependências concluída.")

# Importações-base validadas após a instalação
import pandas as pd
import numpy as np

In [ ]:
# ================================================================
# SGA — Auditoria Pedagógica End-to-End v52
# Versão v52 — heatmap apenas do realizado no PDF e detalhamento 3.1 de RPA consolidado em uma página
# ================================================================

from pathlib import Path
from datetime import datetime
import re
import unicodedata
import warnings
from html import escape as html_escape
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------------------------------------------
# CONFIGURAÇÕES
# ----------------------------------------------------------------
MAPA_PATH = None       # Ex.: "MAPA PEDAGÓGICO 53-2026.xlsx". Deixe None para detectar automaticamente.
REGISTRO_PATH = None   # Ex.: "REGISTRO_PROFESSORES_2026_INSTITUICAO.xlsx". Deixe None para detectar automaticamente.
PROFESSORES_PATH = None  # Ex.: "PROFESSORES_INSTITUICAO_FINAL_JUNHO.xlsx". Base atualizada de docentes/capacidade. Deixe None para detectar automaticamente.
MODULO_ALVO = None     # Ex.: 53. Deixe None para detectar pelo Mapa.
ANO_ALVO = None        # Ex.: 2026. Deixe None para detectar pelo Registro.
OUTPUT_DIR = Path("outputs_v52")
OUTPUT_DIR.mkdir(exist_ok=True)

# ----------------------------------------------------------------
# IDENTIFICAÇÃO EXECUTIVA DO RELATÓRIO
# ----------------------------------------------------------------
# Preencha os campos abaixo: eles alimentam a capa e o rodapé de todas as
# páginas do PDF. O relatório é um documento corporativo, por isso não carrega
# mais marca de ferramenta ("SGA - Auditoria Pedagógica") no cabeçalho.
RELATORIO_AUTOR_NOME = "Seu Nome"
RELATORIO_AUTOR_CARGO = "Analista de Planejamento Acadêmico"
RELATORIO_AUTOR_AREA = ""                      # Opcional. Ex.: "Diretoria de Ensino"
RELATORIO_ORGANIZACAO = "Instituicao EAD"
RELATORIO_FUSO = "America/Sao_Paulo"

# ----------------------------------------------------------------
# COMPOSIÇÃO DA MÉTRICA "DOCENTES CONSIDERADOS"
# ----------------------------------------------------------------
# Regra de negócio: docentes considerados é uma contagem de PESSOAS e é formada
# apenas pelos cargos docentes abaixo. RPA fica fora por natureza: cada linha de
# RPA é uma contratação, não uma pessoa do quadro — sua leitura operacional e
# financeira acontece na seção RPA do relatório. Ajuste as listas se a
# nomenclatura de cargos mudar na base; a comparação ignora acentos e caixa.
CARGOS_DOCENTES_CONSIDERADOS = [
    "PROFESSOR DE ENSINO SUPERIOR EAD",
    "PROFESSOR DE ENSINO SUPERIOR PRESENCIAL",
    "PROFESSOR REGENTE",
]
CARGOS_FORA_DA_METRICA_DOCENTE = ["RPA", "NF"]

def _cargo_normalizado_v53(valor) -> str:
    txt = "".join(ch for ch in unicodedata.normalize("NFKD", str(valor)) if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", txt).strip().upper()

def eh_cargo_docente_considerado(cargo) -> bool:
    """True quando o cargo compõe a métrica de docentes considerados."""
    c = _cargo_normalizado_v53(cargo)
    if not c or c in {"NAN", "NONE"}:
        return False
    for excluido in CARGOS_FORA_DA_METRICA_DOCENTE:
        e = _cargo_normalizado_v53(excluido)
        if e and e in c:
            return False
    alvos = [_cargo_normalizado_v53(x) for x in CARGOS_DOCENTES_CONSIDERADOS]
    return any(a and (a == c or a in c or c in a) for a in alvos)

def mascara_cargos_considerados(serie) -> "pd.Series":
    """Máscara booleana dos cargos que entram na métrica de docentes."""
    if serie is None:
        return pd.Series(dtype=bool)
    s = pd.Series(serie)
    if s.empty:
        return pd.Series(dtype=bool)
    return s.map(eh_cargo_docente_considerado).fillna(False).astype(bool)

def carimbo_atualizacao(fmt: str = "%d/%m/%Y %H:%M") -> str:
    """Data e hora da atualização do relatório, no fuso configurado."""
    try:
        from zoneinfo import ZoneInfo
        return datetime.now(ZoneInfo(RELATORIO_FUSO)).strftime(fmt)
    except Exception:
        return datetime.now().strftime(fmt)

def assinatura_relatorio() -> str:
    """Linha de responsabilidade usada no rodapé: nome, cargo e área."""
    partes = [str(p).strip() for p in (RELATORIO_AUTOR_NOME, RELATORIO_AUTOR_CARGO, RELATORIO_AUTOR_AREA) if str(p).strip()]
    return " | ".join(partes)

# ----------------------------------------------------------------
# ESCALA DOS MAPAS DE CALOR
# ----------------------------------------------------------------
# Problema resolvido aqui: as contagens por dia/horário têm amplitude pequena
# (ex.: de 1 a 5 disciplinas). Numa escala linear com paleta clara, todas as
# células saem quase no mesmo tom. A solução aplicada tem três partes:
#   1. limites calculados por percentil, ignorando células vazias (zeros);
#   2. realce de potência (gamma < 1), que abre o contraste na faixa baixa;
#   3. paletas com amplitude real, do quase branco ao tom escuro.
HEATMAP_GAMMA = 0.55                # < 1 abre contraste nos valores baixos
HEATMAP_PERCENTIS = (2, 98)         # recorte robusto contra células extremas
HEATMAP_PALETAS = {
    "planejado": ["#F8FAFC", "#DBEAFE", "#93C5FD", "#3B82F6", "#1D4ED8", "#172554"],
    "realizado": ["#F8FAFC", "#D1FAE5", "#6EE7B7", "#10B981", "#047857", "#064E3B"],
}

def limites_heatmap(valores, ignorar_zeros: bool = True):
    """Retorna (vmin, vmax) robustos para o mapa de calor.

    Zeros representam ausência de aula e distorcem a escala, por isso saem do
    cálculo por padrão. Se todos os valores forem iguais, abre 1 unidade para
    evitar divisão por zero na normalização.
    """
    arr = np.asarray(valores, dtype=float).ravel()
    arr = arr[np.isfinite(arr)]
    if ignorar_zeros:
        nao_zero = arr[arr > 0]
        if nao_zero.size:
            arr = nao_zero
    if arr.size == 0:
        return 0.0, 1.0
    vmin = float(np.percentile(arr, HEATMAP_PERCENTIS[0]))
    vmax = float(np.percentile(arr, HEATMAP_PERCENTIS[1]))
    if vmax <= vmin:
        vmin, vmax = float(arr.min()), float(arr.max())
    if vmax <= vmin:
        vmax = vmin + 1.0
    return vmin, vmax

def escala_heatmap_gamma(cores=None, gamma: float = None):
    """Colorscale Plotly equivalente ao PowerNorm(gamma) do matplotlib.

    Reposiciona cada parada de cor em t**(1/gamma): com gamma < 1 as cores
    médias e escuras passam a aparecer mais cedo, dando contraste visível
    mesmo quando a variação dos dados é pequena.
    """
    cores = list(cores or HEATMAP_PALETAS["planejado"])
    gamma = HEATMAP_GAMMA if gamma is None else gamma
    n = len(cores)
    escala = []
    for i, cor in enumerate(cores):
        t = i / (n - 1)
        pos = 0.0 if i == 0 else (1.0 if i == n - 1 else round(t ** (1.0 / gamma), 6))
        escala.append([pos, cor])
    return escala

# ----------------------------------------------------------------
# PALETA FIXA DE CORES
# ----------------------------------------------------------------
COLORS = {
    "planejado": "#1D4ED8",       # Azul: Mapa / Planejado
    "realizado": "#059669",       # Verde: Registro / Realizado
    "rpa": "#9333EA",             # Roxo: RPA/NF
    "horas": "#D97706",           # Âmbar: horas calculadas
    "gap": "#DC2626",             # Vermelho: gaps críticos
    "alerta": "#F59E0B",          # Laranja: pontos de atenção
    "neutro": "#64748B",          # Cinza: apoio
    "ok": "#16A34A",              # Verde: conformidade
    "texto": "#0F172A",           # Texto principal
}

ORDER_COLORS = {
    "1ª": "#2563EB",              # Azul fixo para 1ª ordem
    "2ª": "#16A34A",              # Verde fixo para 2ª ordem
    "ESTENDIDA": "#7C3AED",       # Roxo fixo para ESTENDIDA
}

ORDER_SORT = ["1ª", "2ª", "ESTENDIDA"]
DAY_SORT = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
TIME_SORT = ["19:00 às 20:30", "20:40 às 22:10"]
CONTRATOS_VALIDOS = ["CLT EAD", "CLT STRICTO", "RPA/NF"]

pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 220)


def is_notebook() -> bool:
    try:
        from IPython import get_ipython
        ip = get_ipython()
        return ip is not None and "IPKernelApp" in ip.config
    except Exception:
        return False



# Em execução via script (validação automática), suprime displays extensos para acelerar.
# Em JupyterLab, a função display original permanece ativa e o notebook exibe todas as saídas.
if not is_notebook():
    def display(*args, **kwargs):
        return None

def show_fig(fig):
    if is_notebook():
        fig.show()


def aplicar_layout(fig, title: str = None, height: int = 520, legend_title: str = ""):
    fig.update_layout(
        title={"text": title or fig.layout.title.text, "x": 0.02, "xanchor": "left", "font": {"size": 22, "color": COLORS["texto"]}},
        height=height,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font={"family": "Arial, Helvetica, sans-serif", "size": 14, "color": COLORS["texto"]},
        margin={"l": 48, "r": 35, "t": 90, "b": 50},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "right", "x": 1, "title": {"text": legend_title}},
    )
    fig.update_xaxes(showgrid=False, tickfont={"size": 13})
    fig.update_yaxes(showgrid=True, gridcolor="#E2E8F0", tickfont={"size": 13})
    return fig


def br_int(x):
    try:
        if pd.isna(x):
            return "0"
        return f"{int(round(float(x))):,}".replace(",", ".")
    except Exception:
        return str(x)


def br_pct(x, casas=2):
    try:
        if pd.isna(x):
            return "0,0%"
        return f"{float(x):.{casas}f}%".replace(".", ",")
    except Exception:
        return str(x)


def br_money(x):
    try:
        if pd.isna(x):
            x = 0
        s = f"R$ {float(x):,.2f}"
        return s.replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return str(x)


def br_float(x, casas=1):
    try:
        if pd.isna(x):
            return "0"
        return f"{float(x):,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return str(x)




FRIENDLY_COLUMNS = {
    "periodo": "Período",
    "composicao": "Disciplinas planejadas",
    "disciplinas_planejadas": "Disciplinas planejadas",
    "demanda_planejada_horas": "Demanda planejada",
    "ch_letiva_bruta_horas": "CH letiva bruta",
    "ch_letiva_alocavel_horas": "CH letiva alocável",
    "capacidade_docente_horas": "CH letiva bruta",
    "capacidade_docente_alocavel_horas": "CH letiva alocável",
    "delta_demanda_ch_alocavel": "Delta demanda - CH alocável",
    "delta_demanda_capacidade": "Delta demanda - CH alocável",
    "delta_demanda_menos_capacidade": "Delta demanda - CH alocável",
    "horas_alocadas_registro": "Alocado no Registro",
    "delta_alocado_ch_alocavel": "Delta alocado - CH alocável",
    "delta_alocado_capacidade": "Delta alocado - CH alocável",
    "cobertura_capacidade_%": "Cobertura da capacidade",
    "utilizacao_realizada_%": "Utilização realizada",
    "aderencia_%": "Aderência",
    "qtd_rpa": "Quantidade RPA",
    "qtd_rpa_com_valor": "RPA com valor",
    "qtd_rpa_sem_valor": "RPA sem valor",
    "valor_rpa": "Valor RPA",
    "origem": "Origem",
    "professor": "Professor",
    "cargo": "Cargo",
    "ch_semanal": "CH semanal",
    "ch_letiva": "CH letiva",
    "ch_letiva_alocavel": "CH letiva alocável",
    "capacidade_periodo": "Capacidade por período",
    "horas_p1": "Alocado P1",
    "horas_p2": "Alocado P2",
    "saldo_p1": "Saldo P1",
    "saldo_p2": "Saldo P2",
    "utilizacao_p1_%": "Utilização P1",
    "utilizacao_p2_%": "Utilização P2",
    "status_geral": "Status",
}

def friendly_col(col):
    return FRIENDLY_COLUMNS.get(str(col), str(col).replace("_", " ").strip().capitalize())

def html_table(df, columns=None, int_cols=None, pct_cols=None, money_cols=None, hour_cols=None, max_rows=25, empty_msg="Sem registros para exibir."):
    """Gera uma tabela HTML limpa e formatada para o relatório final."""
    if df is None or df.empty:
        return f'<div class="empty">{html_escape(empty_msg)}</div>'
    d = df.copy()
    if columns:
        cols = [c for c in columns if c in d.columns]
        d = d[cols]
    d = d.head(max_rows).copy()
    int_cols = int_cols or []
    pct_cols = pct_cols or []
    money_cols = money_cols or []
    hour_cols = hour_cols or []
    for c in int_cols:
        if c in d.columns:
            d[c] = d[c].map(br_int)
    for c in pct_cols:
        if c in d.columns:
            d[c] = d[c].map(br_pct)
    for c in money_cols:
        if c in d.columns:
            d[c] = d[c].map(br_money)
    for c in hour_cols:
        if c in d.columns:
            d[c] = d[c].map(lambda v: f"{br_int(v)}h")
    d = d.rename(columns={c: friendly_col(c) for c in d.columns})
    return d.to_html(index=False, classes="data-table", border=0, escape=True)


def section(title: str, subtitle: str = ""):
    display(HTML(f"""
    <div style="margin-top:28px; margin-bottom:16px; padding:18px 22px; border-radius:16px; background:linear-gradient(90deg,#0F172A,#1E3A8A); color:white;">
        <div style="font-size:26px; font-weight:800; letter-spacing:.2px;">{title}</div>
        <div style="font-size:15px; opacity:.92; margin-top:6px;">{subtitle}</div>
    </div>
    """))


def insight_box(title: str, text: str, color: str = "#1D4ED8"):
    display(HTML(f"""
    <div style="border-left:7px solid {color}; background:#FFFFFF; padding:16px 18px; border-radius:12px; box-shadow:0 1px 8px rgba(15,23,42,.08); margin:12px 0;">
        <div style="font-weight:800; font-size:18px; color:#0F172A; margin-bottom:6px;">{title}</div>
        <div style="font-size:15px; color:#334155; line-height:1.45;">{text}</div>
    </div>
    """))


def kpi_cards(cards: List[Dict]):
    html = '<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(210px,1fr)); gap:14px; margin:16px 0 22px 0;">'
    for c in cards:
        color = c.get("color", COLORS["planejado"])
        html += f"""
        <div style="background:white; border-top:6px solid {color}; border-radius:16px; padding:16px 18px; box-shadow:0 2px 12px rgba(15,23,42,.10); min-height:122px;">
            <div style="font-size:13px; color:#64748B; font-weight:700; text-transform:uppercase; letter-spacing:.6px;">{c.get('label','')}</div>
            <div style="font-size:30px; line-height:1.1; font-weight:900; color:#0F172A; margin-top:8px; word-break:break-word;">{c.get('value','')}</div>
            <div style="font-size:14px; color:#475569; margin-top:8px;">{c.get('note','')}</div>
        </div>
        """
    html += '</div>'
    display(HTML(html))

section("SGA — Auditoria Pedagógica", "Notebook end-to-end com visual executivo, sinergias do Mapa, docentes ativos, CH letiva alocável validada, realizado alinhado ao Ato 2, dashboard HTML premium e relatório PDF completo.")

In [ ]:
# ================================================================
# NORMALIZAÇÃO, PARSING E DETECÇÃO DE ESTRUTURA
# ================================================================

def strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(text)) if not unicodedata.combining(ch))


def normalize_text(value) -> str:
    if pd.isna(value):
        return ""
    s = str(value).strip()
    return re.sub(r"\s+", " ", s)


def normalize_key(value) -> str:
    s = normalize_text(value).upper()
    s = strip_accents(s)
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def normalize_id(value) -> str:
    """Normaliza identificadores como CHAPA/CPF preservando dígitos quando Excel lê como número."""
    if pd.isna(value):
        return ""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        if np.isfinite(value):
            return str(int(round(float(value))))
        return ""
    s = normalize_text(value)
    try:
        if re.fullmatch(r"[0-9]+(\.[0-9]+)?[eE][+-]?[0-9]+", s):
            return str(int(round(float(s))))
    except Exception:
        pass
    digits = re.sub(r"\D", "", s)
    return digits if digits else s


def montar_chave_docente(chapa_normalizada: str, professor_key: str) -> str:
    """Chave de casamento entre Registro e Base de Professores.

    Regra: usar CHAPA (identificador estável e sem ambiguidade de grafia) sempre que
    disponível; usar o nome normalizado apenas como reserva quando a CHAPA está
    ausente, zerada ou inválida (ex.: linha "PROFESSOR RPA" ou linha sem professor).
    Isso evita que pequenas divergências de nome entre planilhas (sobrenome
    faltando, abreviação) façam um docente ativo cair incorretamente em governança.
    """
    chapa_valida = chapa_normalizada if chapa_normalizada not in ("", "0", "00", "000") else ""
    return f"CHAPA:{chapa_valida}" if chapa_valida else f"NOME:{professor_key}"


def clean_contract(value) -> str:
    s = normalize_key(value)
    if s in {"RPA NF", "RPA/NF", "RPA"}:
        return "RPA/NF"
    if s == "CLT EAD":
        return "CLT EAD"
    if s == "CLT STRICTO":
        return "CLT STRICTO"
    if s == "SOB DEMANDA":
        return "SOB DEMANDA"
    if s in {"NAO TEM AULA", "NAO TEM", "NSA", ""}:
        return s if s else "NÃO INFORMADO"
    return normalize_text(value).upper()


def normalize_order(value) -> str:
    s = normalize_key(value)
    if s in {"1", "1A", "1 ORDEM", "PRIMEIRA", "PRIMEIRA ORDEM"}:
        return "1ª"
    if s in {"2", "2A", "2 ORDEM", "SEGUNDA", "SEGUNDA ORDEM"}:
        return "2ª"
    if "ESTEND" in s:
        return "ESTENDIDA"
    return "" if s in {"", "NAN", "NONE", "NSA"} else normalize_text(value)


def normalize_day(value) -> str:
    s = normalize_key(value)
    if not s or s in {"NSA", "NAN", "NONE"}:
        return ""
    if "SEGUNDA" in s:
        return "Segunda"
    if "TERCA" in s:
        return "Terça"
    if "QUARTA" in s:
        return "Quarta"
    if "QUINTA" in s:
        return "Quinta"
    if "SEXTA" in s:
        return "Sexta"
    if "SABADO" in s:
        return "Sábado"
    if "DOMINGO" in s:
        return "Domingo"
    return ""


def normalize_time(value) -> str:
    if pd.isna(value):
        return ""
    if hasattr(value, "strftime"):
        try:
            h = value.strftime("%H:%M")
        except Exception:
            h = normalize_text(value)
    else:
        s = normalize_text(value)
        if not s or normalize_key(s) in {"NSA", "NAN", "NONE"}:
            return ""
        m = re.search(r"(\d{1,2}:\d{2})", s)
        if not m:
            return ""
        h = m.group(1).zfill(5)
    if h == "19:00":
        return "19:00 às 20:30"
    if h == "20:40":
        return "20:40 às 22:10"
    return h


def parse_money(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    s = normalize_text(value)
    if not s or normalize_key(s) in {"NSA", "NAN", "NONE", "NAO INFORMADO"}:
        return np.nan
    s = s.replace("R$", "").strip()
    s = re.sub(r"[^0-9,\.\-]", "", s)
    if not s:
        return np.nan
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return np.nan


def locate_file(patterns: List[str], base_dir: Path = Path(".")) -> Path:
    files = []
    for p in patterns:
        files.extend(base_dir.glob(p))
    files = [f for f in files if f.is_file() and not f.name.startswith("~$")]
    if not files:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para os padrões: {patterns}")
    return sorted(files, key=lambda x: x.stat().st_mtime, reverse=True)[0]


def find_header_row(excel_path: Path, sheet_name: str, required_terms: List[str], max_scan_rows: int = 40) -> int:
    raw = pd.read_excel(excel_path, sheet_name=sheet_name, header=None, nrows=max_scan_rows)
    req = [normalize_key(t) for t in required_terms]
    for idx, row in raw.iterrows():
        vals = [normalize_key(v) for v in row.tolist() if normalize_key(v)]
        row_text = " | ".join(vals)
        if all(term in row_text for term in req):
            return int(idx)
    raise ValueError(f"Cabeçalho não encontrado na aba {sheet_name} com os termos {required_terms}")


def find_column_by_candidates(columns: List[str], candidates: List[str], must_contain_all: bool = False) -> Optional[str]:
    norm_cols = {c: normalize_key(c) for c in columns}
    cand_norm = [normalize_key(c) for c in candidates]
    for col, ncol in norm_cols.items():
        if must_contain_all:
            if all(c in ncol for c in cand_norm):
                return col
        else:
            for c in cand_norm:
                if c == ncol or c in ncol:
                    return col
    return None


def choose_format_col(df: pd.DataFrame) -> str:
    candidates = [c for c in df.columns if "FORMATO" in normalize_key(c)]
    for c in candidates:
        vals = df[c].dropna().astype(str).map(normalize_key)
        if vals.eq("AO VIVO").any():
            return c
    raise ValueError("Não encontrei coluna de formato com valor AO VIVO.")


def detect_map_sheet(path: Path) -> str:
    xl = pd.ExcelFile(path)
    scores = []
    for sheet in xl.sheet_names:
        try:
            sample = pd.read_excel(path, sheet_name=sheet, nrows=5)
            cols = [normalize_key(c) for c in sample.columns]
            score = 0
            for term in ["NOME DISCIPLINA", "MODELO CONTRATO", "ORDEM", "MOD", "DIA AULA", "HORARIO"]:
                if any(term in c for c in cols):
                    score += 1
            # Preferência para a aba operacional do módulo (ex.: 53.2026) e penalização de abas auxiliares.
            sheet_key = normalize_key(sheet)
            if re.match(r"^\d", sheet_key):
                score += 10
            if "INSTITUCIONAIS" in sheet_key or "TCC" in sheet_key or "ESTAGIO" in sheet_key or "MONITORAMENTO" in sheet_key:
                score -= 5
            scores.append((score, sheet))
        except Exception:
            pass
    scores = sorted(scores, reverse=True)
    if not scores or scores[0][0] < 3:
        raise ValueError("Não foi possível detectar a aba do Mapa Pedagógico.")
    return scores[0][1]


def detect_registro_sheet_and_header(path: Path) -> Tuple[str, int]:
    xl = pd.ExcelFile(path)
    for sheet in xl.sheet_names:
        try:
            header = find_header_row(path, sheet, ["PROFESSOR", "MÓDULO", "ORDEM", "DISCIPLINA"])
            return sheet, header
        except Exception:
            continue
    raise ValueError("Não foi possível detectar a aba/cabeçalho do Registro.")


def make_key(df: pd.DataFrame) -> pd.Series:
    return (
        df["disciplina_key"].fillna("") + "||" +
        df["ordem"].fillna("") + "||" +
        df["dia_semana"].fillna("") + "||" +
        df["horario"].fillna("")
    )


def safe_first(series: pd.Series):
    vals = [v for v in series.dropna().tolist() if normalize_text(v)]
    return vals[0] if vals else ""


def safe_join_unique(series: pd.Series, max_items: int = 5):
    vals = []
    seen = set()
    for v in series.dropna().tolist():
        txt = normalize_text(v)
        key = normalize_key(txt)
        if txt and key not in {"NSA", "NAN", "NONE", "-"} and key not in seen:
            vals.append(txt)
            seen.add(key)
    if len(vals) > max_items:
        return "; ".join(vals[:max_items]) + f"; +{len(vals)-max_items}"
    return "; ".join(vals)

In [ ]:
# ================================================================
# LEITURA DOS ARQUIVOS E ETL
# ================================================================

def carregar_mapa(path: Path) -> Tuple[pd.DataFrame, str, Dict[str, str]]:
    sheet = detect_map_sheet(path)
    df = pd.read_excel(path, sheet_name=sheet)
    df = df.dropna(how="all").copy()

    format_col = choose_format_col(df)
    contrato_col = find_column_by_candidates(df.columns, ["MODELO CONTRATO"], must_contain_all=True)
    ordem_col = find_column_by_candidates(df.columns, ["ORDEM"])
    disciplina_col = find_column_by_candidates(df.columns, ["NOME DISCIPLINA"], must_contain_all=True)
    curso_col = find_column_by_candidates(df.columns, ["NOME CURSO"], must_contain_all=True)
    codigo_curso_col = find_column_by_candidates(df.columns, ["CURSO"])
    cluster_col = find_column_by_candidates(df.columns, ["CLUSTER"])
    modulo_col = find_column_by_candidates(df.columns, ["MÓD", "MOD"])
    ano_col = find_column_by_candidates(df.columns, ["ANO"])
    dia_col = find_column_by_candidates(df.columns, ["DIA AULA"], must_contain_all=True)
    horario_col = find_column_by_candidates(df.columns, ["HORÁRIO", "HORARIO"])
    valor_col = find_column_by_candidates(df.columns, ["VALOR"])
    prof_clt_col = find_column_by_candidates(df.columns, ["PROFESSOR CLT"], must_contain_all=True)
    prof_rpa_col = find_column_by_candidates(df.columns, ["PROFESSOR RPA", "PROFESSOR RPA/NF"])
    codigo_disciplina_col = None
    for _col in df.columns:
        # No Mapa, a coluna DISCIPLINA costuma conter o código operacional da disciplina;
        # NOME DISCIPLINA fica em outra coluna e já é usado como descrição.
        if normalize_key(_col) == "DISCIPLINA":
            codigo_disciplina_col = _col
            break
    sinergia_col = find_column_by_candidates(df.columns, ["SINERGIA"])

    required = {"format_col": format_col, "contrato_col": contrato_col, "ordem_col": ordem_col, "disciplina_col": disciplina_col, "modulo_col": modulo_col, "dia_col": dia_col, "horario_col": horario_col}
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes no Mapa: {missing}")

    out = pd.DataFrame()
    out["fonte"] = pd.Series("Mapa", index=df.index)
    out["modulo"] = pd.to_numeric(df[modulo_col], errors="coerce").astype("Int64")
    out["ano"] = pd.to_numeric(df[ano_col], errors="coerce").astype("Int64") if ano_col else pd.Series([pd.NA] * len(df), dtype="Int64")
    out["curso_codigo"] = df[codigo_curso_col].map(normalize_text) if codigo_curso_col else ""
    out["curso_nome"] = df[curso_col].map(normalize_text) if curso_col else ""
    out["curso_key"] = (out["curso_codigo"].map(normalize_key) + "||" + out["curso_nome"].map(normalize_key)).str.strip("|")
    out["cluster"] = df[cluster_col].map(normalize_text) if cluster_col else ""
    out["cluster_key"] = out["cluster"].map(normalize_key)
    out["disciplina_codigo"] = df[codigo_disciplina_col].map(normalize_text) if codigo_disciplina_col else ""
    out["disciplina_codigo_key"] = df[codigo_disciplina_col].map(normalize_key) if codigo_disciplina_col else ""
    out["disciplina"] = df[disciplina_col].map(normalize_text)
    out["disciplina_key"] = df[disciplina_col].map(normalize_key)
    out["sinergia"] = df[sinergia_col].map(normalize_text) if sinergia_col else ""
    out["sinergia_key"] = out["sinergia"].map(normalize_key)
    out["ordem"] = df[ordem_col].map(normalize_order)
    out["dia_semana"] = df[dia_col].map(normalize_day)
    out["horario"] = df[horario_col].map(normalize_time)
    out["contrato"] = df[contrato_col].map(clean_contract)
    out["formato_ao_vivo"] = df[format_col].map(normalize_key).eq("AO VIVO")
    out["valor_rpa_mapa_raw"] = df[valor_col].map(normalize_text) if valor_col else ""
    out["valor_rpa_mapa"] = df[valor_col].map(parse_money) if valor_col else np.nan
    out["professor_clt"] = df[prof_clt_col].map(normalize_text) if prof_clt_col else ""
    out["professor_rpa"] = df[prof_rpa_col].map(normalize_text) if prof_rpa_col else ""
    out["professor_planejado"] = np.where(out["contrato"].eq("RPA/NF"), out["professor_rpa"], out["professor_clt"])
    out["professor_planejado_key"] = pd.Series(out["professor_planejado"]).map(normalize_key)
    out["linha_origem"] = np.arange(2, len(df) + 2)
    out["chave_conformidade"] = make_key(out)

    meta = {"sheet": sheet, "format_col": format_col, "contrato_col": contrato_col, "ordem_col": ordem_col, "disciplina_col": disciplina_col, "codigo_disciplina_col": codigo_disciplina_col, "curso_col": curso_col, "codigo_curso_col": codigo_curso_col, "cluster_col": cluster_col, "sinergia_col": sinergia_col, "dia_col": dia_col, "horario_col": horario_col, "valor_col": valor_col}
    return out, sheet, meta


def carregar_registro(path: Path) -> Tuple[pd.DataFrame, str, int, Dict[str, str]]:
    sheet, header = detect_registro_sheet_and_header(path)
    df = pd.read_excel(path, sheet_name=sheet, header=header)
    df = df[[c for c in df.columns if not str(c).startswith("Unnamed")]].dropna(how="all").copy()

    professor_col = find_column_by_candidates(df.columns, ["PROFESSOR"])
    modulo_col = find_column_by_candidates(df.columns, ["MÓDULO", "MODULO"])
    ordem_col = find_column_by_candidates(df.columns, ["ORDEM"])
    dia_col = find_column_by_candidates(df.columns, ["DIA DA TRANSMISSÃO", "DIA TRANSMISSAO", "DIA"])
    horario_col = find_column_by_candidates(df.columns, ["HORÁRIO", "HORARIO"])
    disciplina_col = find_column_by_candidates(df.columns, ["DISCIPLINA"])
    rpa_col = find_column_by_candidates(df.columns, ["R$ (RPA)", "RPA"])
    chapa_col = find_column_by_candidates(df.columns, ["CHAPA"])
    cargo_col = find_column_by_candidates(df.columns, ["CARGO"])
    ch_semanal_col = find_column_by_candidates(df.columns, ["CH SEMANAL", "CH_SEMANAL"])
    ch_letiva_col = find_column_by_candidates(df.columns, ["CH LETIVA", "CH_LETIVA"])
    observacao_col = find_column_by_candidates(df.columns, ["OBSERVAÇÃO", "OBSERVACAO"])

    required = {"professor_col": professor_col, "modulo_col": modulo_col, "ordem_col": ordem_col, "dia_col": dia_col, "horario_col": horario_col, "disciplina_col": disciplina_col}
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes no Registro: {missing}")

    modulo_raw = df[modulo_col].map(normalize_text)
    modulo_num = modulo_raw.str.extract(r"(\d+)", expand=False)
    ano_num = modulo_raw.str.extract(r"\|(\d{4})", expand=False)

    out = pd.DataFrame()
    out["fonte"] = pd.Series("Registro", index=df.index)
    out["modulo_raw"] = modulo_raw
    out["modulo"] = pd.to_numeric(modulo_num, errors="coerce").astype("Int64")
    out["ano"] = pd.to_numeric(ano_num, errors="coerce").astype("Int64")
    out["professor_realizado"] = df[professor_col].map(normalize_text)
    out["professor_realizado_key"] = df[professor_col].map(normalize_key)
    # CHAPA vem do Excel como número (às vezes float); normalize_id evita o sufixo ".0"
    # que quebraria o casamento com a CHAPA (também numérica) da base de professores.
    out["chapa"] = df[chapa_col].map(normalize_id) if chapa_col else ""
    out["chave_docente"] = [montar_chave_docente(c, k) for c, k in zip(out["chapa"], out["professor_realizado_key"])]
    out["cargo"] = df[cargo_col].map(normalize_text) if cargo_col else ""
    out["ch_semanal"] = pd.to_numeric(df[ch_semanal_col], errors="coerce") if ch_semanal_col else np.nan
    out["ch_letiva"] = pd.to_numeric(df[ch_letiva_col], errors="coerce") if ch_letiva_col else np.nan
    out["observacao"] = df[observacao_col].map(normalize_text) if observacao_col else ""
    out["disciplina"] = df[disciplina_col].map(normalize_text)
    out["disciplina_key"] = df[disciplina_col].map(normalize_key)
    out["ordem"] = df[ordem_col].map(normalize_order)
    out["dia_semana"] = df[dia_col].map(normalize_day)
    out["horario"] = df[horario_col].map(normalize_time)
    out["valor_rpa_registro_raw"] = df[rpa_col].map(normalize_text) if rpa_col else ""
    out["valor_rpa_registro"] = df[rpa_col].map(parse_money) if rpa_col else np.nan
    out["tem_rpa_registro"] = out["valor_rpa_registro"].fillna(0).gt(0) | out["valor_rpa_registro_raw"].map(normalize_key).ne("")
    # RPA no Registro não deve depender apenas de valor preenchido: quando CARGO = RPA ou professor = PROFESSOR RPA, a linha entra no total de RPA.
    out["is_rpa_registro"] = (
        out["cargo"].map(normalize_key).eq("RPA") |
        out["professor_realizado"].map(normalize_key).str.contains("PROFESSOR RPA", na=False) |
        out["tem_rpa_registro"].eq(True)
    )
    out["linha_origem"] = np.arange(header + 2, len(df) + header + 2)
    out["chave_conformidade"] = make_key(out)

    meta = {"sheet": sheet, "header_row_excel": header + 1, "professor_col": professor_col, "modulo_col": modulo_col, "ordem_col": ordem_col, "dia_col": dia_col, "horario_col": horario_col, "disciplina_col": disciplina_col, "rpa_col": rpa_col, "ch_semanal_col": ch_semanal_col, "ch_letiva_col": ch_letiva_col, "observacao_col": observacao_col}
    return out, sheet, header, meta


def deduplicar_registro_exato(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Remove duplicatas exatas do Registro antes dos cálculos.

    Regra de negócio v20:
    - se houver linhas totalmente iguais no Registro, apenas a primeira ocorrência entra nos cálculos;
    - as demais ocorrências são mantidas em uma base de governança para conferência;
    - a duplicidade é avaliada com os campos normalizados de negócio, sem usar `linha_origem`,
      porque a linha de origem é naturalmente diferente para cada repetição.
    """
    if df is None or df.empty:
        empty_cols = ["grupo_duplicata_exata", "qtd_linhas_grupo", "linha_origem", "professor_realizado", "cargo", "modulo", "ano", "ordem", "dia_semana", "horario", "disciplina", "observacao"]
        return df.copy() if isinstance(df, pd.DataFrame) else pd.DataFrame(), pd.DataFrame(columns=empty_cols), pd.DataFrame(columns=empty_cols)

    base = df.copy()
    duplicate_cols = [
        "modulo", "ano", "professor_realizado_key", "chapa", "cargo",
        "ch_semanal", "ch_letiva", "disciplina_key", "ordem", "dia_semana",
        "horario", "observacao", "valor_rpa_registro_raw"
    ]
    duplicate_cols = [c for c in duplicate_cols if c in base.columns]

    # Chave técnica para agrupar valores, tratando nulos de forma estável.
    chave = (
        base[duplicate_cols]
        .copy()
        .astype("string")
        .fillna("<NA>")
        .agg("||".join, axis=1)
    )
    base["grupo_duplicata_exata"] = pd.factorize(chave)[0] + 1
    base["duplicata_exata_registro"] = base.duplicated(subset=duplicate_cols, keep=False)
    base["removida_por_duplicata_exata"] = base.duplicated(subset=duplicate_cols, keep="first")

    dup_all = base[base["duplicata_exata_registro"].eq(True)].copy()
    dup_removed = base[base["removida_por_duplicata_exata"].eq(True)].copy()

    if not dup_all.empty:
        contagem = dup_all.groupby("grupo_duplicata_exata").size().rename("qtd_linhas_grupo").reset_index()
        primeiras = (
            base[~base["removida_por_duplicata_exata"] & base["duplicata_exata_registro"]]
            [["grupo_duplicata_exata", "linha_origem"]]
            .rename(columns={"linha_origem": "linha_mantida_calculo"})
        )
        dup_all = dup_all.merge(contagem, on="grupo_duplicata_exata", how="left").merge(primeiras, on="grupo_duplicata_exata", how="left")
        dup_removed = dup_removed.merge(contagem, on="grupo_duplicata_exata", how="left").merge(primeiras, on="grupo_duplicata_exata", how="left")
        dup_all["regra_tratamento"] = "Grupo com duplicata exata no Registro: primeira ocorrência mantida; repetições removidas dos cálculos."
        dup_removed["regra_tratamento"] = "Linha removida dos cálculos por ser duplicata exata de outra linha do Registro."

    limpo = base[~base["removida_por_duplicata_exata"]].copy()
    return limpo, dup_all, dup_removed



def calcular_ch_letiva_por_funcao_ch(cargo, ch_semanal):
    """Deriva CH letiva quando a base atualizada traz apenas a CH base semanal.

    Regra validada a partir da parametrização histórica:
    - PROFESSOR REGENTE 12h -> 9h letivas;
    - PROFESSOR REGENTE 24h -> 18h letivas;
    - PROFESSOR REGENTE 40h -> 20h letivas;
    - PROFESSOR DE ENSINO SUPERIOR EAD 12h/20h -> 2h letivas;
    - PROFESSOR DE ENSINO SUPERIOR EAD 40h -> 4h letivas;
    - Demais funções sem regra operacional conhecida recebem 0h letivas, para não inflar capacidade.
    """
    try:
        ch = float(ch_semanal) if pd.notna(ch_semanal) else 0.0
    except Exception:
        ch = 0.0
    cargo_key = normalize_key(cargo)
    if "REGENTE" in cargo_key:
        if abs(ch - 12) < 0.01:
            return 9.0
        if abs(ch - 24) < 0.01:
            return 18.0
        if abs(ch - 40) < 0.01:
            return 20.0
        # fallback conservador para novos contratos de regente: 75% limitado a 20h
        return min(np.floor((ch * 0.75) / 1) * 1, 20.0) if ch > 0 else 0.0
    if "ENSINO SUPERIOR EAD" in cargo_key:
        if ch >= 40:
            return 4.0
        if ch >= 12:
            return 2.0
        return 0.0
    # Professores presenciais/tutores não entram como capacidade letiva EAD nesta análise, salvo se futuramente houver regra de negócio específica.
    return 0.0


def locate_professores_file(base_dir: Path = Path(".")) -> Optional[Path]:
    """Localiza a base atualizada de docentes, evitando confundir com a planilha Registro."""
    patterns = ["*PROFESSORES*INSTITUICAO*.xlsx", "*PROFESSORES*.xlsx", "*professores*.xlsx"]
    files = []
    for ptn in patterns:
        files.extend(base_dir.glob(ptn))
    valid = []
    for f in files:
        name_key = normalize_key(f.name)
        if not f.is_file() or f.name.startswith("~$"):
            continue
        if any(x in name_key for x in ["REGISTRO", "SGA", "AUDITORIA", "INTELIGENCIA", "SPRINT", "PRESENCIAL"]):
            continue
        valid.append(f)
    if not valid:
        return None
    return sorted(valid, key=lambda x: x.stat().st_mtime, reverse=True)[0]


def locate_docentes_presencial_file(base_dir: Path = Path(".")) -> Optional[Path]:
    """Localiza a base de docentes presenciais (CH contratada), se o arquivo tiver sido fornecido neste mês.

    Esta base é opcional: nem todo mês ela estará disponível. Quando ausente, o notebook
    segue normalmente e só não gera a seção "Docentes Presencial (caso à parte)".
    """
    patterns = ["*DOCENTESPRESENCIAL*.xlsx", "*DOCENTES*PRESENCIAL*.xlsx", "*docentespresencial*.xlsx", "*docentes*presencial*.xlsx"]
    files = []
    for ptn in patterns:
        files.extend(base_dir.glob(ptn))
    valid = [f for f in files if f.is_file() and not f.name.startswith("~$")]
    if not valid:
        return None
    return sorted(valid, key=lambda x: x.stat().st_mtime, reverse=True)[0]


def _hhmm_para_horas(valor) -> float:
    """Converte um valor de CH no formato 'HH:MM' (texto) ou numérico para horas decimais."""
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float, np.integer, np.floating)):
        return float(valor)
    s = str(valor).strip()
    if ":" in s:
        partes = s.split(":")
        try:
            h = int(partes[0])
            m = int(partes[1]) if len(partes) > 1 else 0
            return h + m / 60.0
        except (ValueError, IndexError):
            return np.nan
    try:
        return float(s.replace(",", "."))
    except ValueError:
        return np.nan


def carregar_docentes_presencial(path: Path) -> Tuple[pd.DataFrame, str, Dict[str, str]]:
    """Carrega a base de docentes presenciais: CH contratada (semanal) e tipo de contrato.

    Esta base é usada SOMENTE para uma seção informativa à parte ("caso à parte"):
    professores de cargo "Professor de Ensino Superior Presencial" que aparecem alocados
    no Registro mas não constam na base de capacidade CLT EAD/Regente. Antes desta base,
    não havia nenhuma CH conhecida para eles; agora temos a CH contratada (semanal), mas
    ainda NÃO existe uma regra validada de CH LETIVA para presencial — por isso essa coluna
    não é calculada aqui. Nada desta base entra nos totais gerais de capacidade/delta docente.
    """
    xl = pd.ExcelFile(path)
    sheet = xl.sheet_names[0]
    df = pd.read_excel(path, sheet_name=sheet)
    df = df.dropna(how="all").copy()

    nome_col = find_column_by_candidates(df.columns, ["NOME"])
    chapa_col = find_column_by_candidates(df.columns, ["CHAPA"])
    funcao_col = find_column_by_candidates(df.columns, ["FUNÇÃO", "FUNCAO"])
    situacao_col = find_column_by_candidates(df.columns, ["SITUAÇÃO", "SITUACAO"])
    tipo_rec_col = find_column_by_candidates(df.columns, ["TIPO REC", "TIPO_REC"])
    ch_semanal_col = find_column_by_candidates(df.columns, ["CH.SEM", "CH SEM", "CH_SEM"])
    ch_ultima_col = find_column_by_candidates(df.columns, ["CH (ULTIMA)", "CH ULTIMA", "CH(ULTIMA)"])

    required = {"nome_col": nome_col, "chapa_col": chapa_col, "ch_semanal_col": ch_semanal_col}
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes na base de docentes presenciais: {missing}")

    out = pd.DataFrame()
    out["professor_presencial"] = df[nome_col].map(normalize_text)
    out["professor_key"] = df[nome_col].map(normalize_key)
    out["chapa"] = df[chapa_col].map(normalize_id)
    out["chave_docente"] = [montar_chave_docente(c, k) for c, k in zip(out["chapa"], out["professor_key"])]
    out["funcao_presencial"] = df[funcao_col].map(normalize_text) if funcao_col else ""
    out["situacao_rh"] = df[situacao_col].map(normalize_text) if situacao_col else ""
    out["tipo_contrato"] = df[tipo_rec_col].map(normalize_text) if tipo_rec_col else "NÃO INFORMADO"
    out["ch_contratada_semanal"] = df[ch_semanal_col].map(_hhmm_para_horas)
    out["ch_contratada_mensal"] = df[ch_ultima_col].map(_hhmm_para_horas) if ch_ultima_col else np.nan

    out = out[out["chave_docente"].ne("")].drop_duplicates("chave_docente", keep="first").copy()

    meta = {
        "sheet": sheet, "nome_col": nome_col, "chapa_col": chapa_col, "tipo_rec_col": tipo_rec_col,
        "ch_semanal_col": ch_semanal_col, "ch_ultima_col": ch_ultima_col,
        "total_linhas": int(len(out)),
    }
    return out, sheet, meta


def carregar_docentes_atualizados(path: Path) -> Tuple[pd.DataFrame, str, Dict[str, str]]:
    """Carrega a planilha atualizada de professores como banco oficial de capacidade docente.

    Esta base substitui a aba PARAMETROS do Registro para cálculo de CH docente.
    A aba PARAMETROS pode estar desatualizada, então permanece apenas como referência histórica/fallback.
    """
    xl = pd.ExcelFile(path)
    sheet = xl.sheet_names[0]
    # Procura uma aba que contenha NOME/CHAPA/FUNCAO se existir mais de uma.
    for s in xl.sheet_names:
        try:
            probe = pd.read_excel(path, sheet_name=s, nrows=5)
            cols_key = " | ".join(normalize_key(c) for c in probe.columns)
            if "NOME" in cols_key and ("FUNCAO" in cols_key or "CARGO" in cols_key) and ("CHAPA" in cols_key):
                sheet = s
                break
        except Exception:
            continue

    df = pd.read_excel(path, sheet_name=sheet)
    df = df[[c for c in df.columns if not str(c).startswith("Unnamed")]].dropna(how="all").copy()

    nome_col = find_column_by_candidates(df.columns, ["NOME", "NM_NOME", "PROFESSOR"])
    chapa_col = find_column_by_candidates(df.columns, ["CHAPA"])
    cargo_col = find_column_by_candidates(df.columns, ["NM_FUNCAO", "FUNÇÃO", "FUNCAO", "CARGO"])
    status_col = find_column_by_candidates(df.columns, ["DS_SITUACAO", "SITUAÇÃO", "SITUACAO", "STATUS"])
    ch_semanal_col = find_column_by_candidates(df.columns, ["VL_CH_BASE_SEMANAL", "CH BASE SEMANAL", "CH SEMANAL", "CH_SEMANAL"])
    ch_letiva_col = find_column_by_candidates(df.columns, ["CH LETIVA", "CH_LETIVA"])
    equipe_col = find_column_by_candidates(df.columns, ["NM_EQUIPE", "EQUIPE"])
    gestor_col = find_column_by_candidates(df.columns, ["NM_GESTOR", "GESTOR"])
    marca_col = find_column_by_candidates(df.columns, ["MARCA"])
    admissao_col = find_column_by_candidates(df.columns, ["DT_ADMISSAO", "ADMISSAO", "ADMISSÃO"])

    required = {"nome_col": nome_col, "ch_semanal_col": ch_semanal_col, "cargo_col": cargo_col, "status_col": status_col}
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes na base atualizada de professores: {missing}")

    out = pd.DataFrame()
    out["fonte"] = pd.Series("BASE_PROFESSORES_ATUALIZADA", index=df.index)
    out["professor_parametros"] = df[nome_col].map(normalize_text)
    out["professor_key"] = df[nome_col].map(normalize_key)
    out["chapa"] = df[chapa_col].map(normalize_id) if chapa_col else ""
    out["chave_docente"] = [montar_chave_docente(c, k) for c, k in zip(out["chapa"], out["professor_key"])]
    out["ch_semanal"] = pd.to_numeric(df[ch_semanal_col], errors="coerce").fillna(0)
    out["cargo"] = df[cargo_col].map(normalize_text)
    out["cargo_key"] = out["cargo"].map(normalize_key)
    out["status"] = df[status_col].map(normalize_text)
    out["status_key"] = out["status"].map(normalize_key)
    out["docente_ativo"] = out["status_key"].eq("ATIVO")
    out["equipe_rh"] = df[equipe_col].map(normalize_text) if equipe_col else ""
    out["gestor"] = df[gestor_col].map(normalize_text) if gestor_col else ""
    out["marca"] = df[marca_col].map(normalize_text) if marca_col else ""
    out["dt_admissao"] = df[admissao_col] if admissao_col else pd.NaT

    if ch_letiva_col:
        out["ch_letiva"] = pd.to_numeric(df[ch_letiva_col], errors="coerce").fillna(0)
        regra_ch = "CH LETIVA informada diretamente na base atualizada."
    else:
        out["ch_letiva"] = [calcular_ch_letiva_por_funcao_ch(c, ch) for c, ch in zip(out["cargo"], out["ch_semanal"])]
        regra_ch = "CH LETIVA derivada por cargo + CH base semanal, conforme regra de negócio validada."

    out["ch_nao_letiva"] = np.nan
    out["capacidade_periodo"] = np.floor(out["ch_letiva"].fillna(0) / 2) * 2
    out["ch_letiva_alocavel"] = out["capacidade_periodo"]
    out["capacidade_modulo_2_periodos"] = out["capacidade_periodo"] * 2

    base_docentes = out[out["professor_key"].ne("")].copy()
    base_ativos = base_docentes[base_docentes["docente_ativo"].eq(True)].copy()
    base_ativos["parametros_linha_id"] = np.arange(1, len(base_ativos) + 1)
    base_ativos["rank_professor_parametros"] = base_ativos.groupby("chave_docente").cumcount() + 1
    base_ativos["professor_duplicado_parametros"] = base_ativos.duplicated("chave_docente", keep=False)

    meta = {
        "sheet": sheet,
        "arquivo": path.name,
        "fonte_capacidade": "BASE_PROFESSORES_ATUALIZADA",
        "nome_col": nome_col,
        "chapa_col": chapa_col,
        "ch_semanal_col": ch_semanal_col,
        "ch_letiva_col": ch_letiva_col or "DERIVADA_CARGO_CH_SEMANAL",
        "status_col": status_col,
        "cargo_col": cargo_col,
        "total_docentes_parametros": int(len(base_docentes)),
        "total_docentes_parametros_unicos": int(base_docentes["professor_key"].nunique()),
        "total_docentes_ativos": int(len(base_ativos)),
        "total_docentes_ativos_unicos": int(base_ativos["professor_key"].nunique()),
        "total_docentes_excluidos_capacidade": int(len(base_docentes) - len(base_ativos)),
        "total_ch_letiva_ativos": float(base_ativos["ch_letiva"].fillna(0).sum()),
        "total_ch_letiva_alocavel_ativos": float(base_ativos["ch_letiva_alocavel"].fillna(0).sum()),
        "regra_capacidade": "Base atualizada de professores: somente DS_SITUACAO/STATUS = ATIVO; CH letiva alocável = piso(CH letiva/2) × 2; docentes alocados fora do ativo entram por regra operacional e permanecem em governança.",
        "regra_ch_letiva_derivada": regra_ch,
        "base_docentes_todos": base_docentes.copy(),
    }
    return base_ativos, sheet, meta


def carregar_base_docentes_capacidade(professores_path: Optional[Path], registro_path: Path) -> Tuple[pd.DataFrame, str, Dict[str, str]]:
    """Carrega a base oficial de capacidade docente.

    Prioridade:
    1. Planilha atualizada de professores (PROFESSORES_PATH ou detecção automática);
    2. Fallback para aba PARAMETROS do Registro, apenas para compatibilidade.
    """
    if professores_path is not None and Path(professores_path).exists():
        return carregar_docentes_atualizados(Path(professores_path))
    auto = locate_professores_file(Path("."))
    if auto is not None:
        return carregar_docentes_atualizados(auto)
    print("AVISO: base atualizada de professores não localizada. Usando aba PARAMETROS do Registro como fallback.")
    return carregar_parametros_registro(registro_path)

def carregar_parametros_registro(path: Path) -> Tuple[pd.DataFrame, str, Dict[str, str]]:
    """Carrega a aba PARAMETROS do Registro como banco oficial de docentes.

    Regra validada com a área de negócio:
    - o cálculo de capacidade usa somente linhas com STATUS = ATIVO;
    - a CH letiva bruta deve bater com a soma da aba PARAMETROS (ex.: 1057h no módulo validado);
    - a CH letiva alocável usa blocos de 2h: piso(CH letiva / 2) × 2;
    - não deduplicamos professores automaticamente, porque a aba PARAMETROS pode ter o mesmo docente em mais de uma equipe/célula operacional. Para transparência, o notebook exibe tanto registros ativos quanto professores únicos ativos.
    """
    xl = pd.ExcelFile(path)
    sheet = None
    for s in xl.sheet_names:
        if "PARAM" in normalize_key(s):
            sheet = s
            break
    if sheet is None:
        raise ValueError("Aba PARAMETROS não encontrada no arquivo de Registro.")

    df = pd.read_excel(path, sheet_name=sheet)
    df = df[[c for c in df.columns if not str(c).startswith("Unnamed")]].dropna(how="all").copy()

    nome_col = find_column_by_candidates(df.columns, ["NOME"])
    chapa_col = find_column_by_candidates(df.columns, ["CHAPA"])
    ch_semanal_col = find_column_by_candidates(df.columns, ["CH SEMANAL", "CH_SEMANAL"])
    ch_letiva_col = find_column_by_candidates(df.columns, ["CH LETIVA", "CH_LETIVA"])
    observacao_col = find_column_by_candidates(df.columns, ["OBSERVAÇÃO", "OBSERVACAO"])
    ch_nao_letiva_col = find_column_by_candidates(df.columns, ["CH NÃO-LETIVA", "CH NAO LETIVA", "CH NÃO LETIVA"])
    equipe_col = find_column_by_candidates(df.columns, ["EQUIPE"])
    status_col = find_column_by_candidates(df.columns, ["STATUS"])
    cargo_col = find_column_by_candidates(df.columns, ["CARGO"])

    required = {"nome_col": nome_col, "ch_semanal_col": ch_semanal_col, "ch_letiva_col": ch_letiva_col, "cargo_col": cargo_col}
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes na aba PARAMETROS: {missing}")

    out = pd.DataFrame()
    out["fonte"] = pd.Series("PARAMETROS", index=df.index)
    out["professor_parametros"] = df[nome_col].map(normalize_text)
    out["professor_key"] = df[nome_col].map(normalize_key)
    out["chapa"] = df[chapa_col].map(normalize_id) if chapa_col else ""
    out["chave_docente"] = [montar_chave_docente(c, k) for c, k in zip(out["chapa"], out["professor_key"])]
    out["ch_semanal"] = pd.to_numeric(df[ch_semanal_col], errors="coerce")
    out["ch_letiva"] = pd.to_numeric(df[ch_letiva_col], errors="coerce")
    out["ch_nao_letiva"] = pd.to_numeric(df[ch_nao_letiva_col], errors="coerce") if ch_nao_letiva_col else np.nan
    out["equipe_rh"] = df[equipe_col].map(normalize_text) if equipe_col else ""
    out["status"] = df[status_col].map(normalize_text) if status_col else "ATIVO"
    out["status_key"] = out["status"].map(normalize_key)
    out["docente_ativo"] = out["status_key"].eq("ATIVO") if status_col else True
    out["cargo"] = df[cargo_col].map(normalize_text) if cargo_col else ""
    out["cargo_key"] = out["cargo"].map(normalize_key)
    out["capacidade_periodo"] = np.floor(out["ch_letiva"].fillna(0) / 2) * 2
    out["ch_letiva_alocavel"] = out["capacidade_periodo"]
    out["capacidade_modulo_2_periodos"] = out["capacidade_periodo"] * 2

    base_docentes = out[out["professor_key"].ne("")].copy()
    base_ativos = base_docentes[base_docentes["docente_ativo"].eq(True)].copy()
    base_ativos["parametros_linha_id"] = np.arange(1, len(base_ativos) + 1)
    base_ativos["rank_professor_parametros"] = base_ativos.groupby("chave_docente").cumcount() + 1
    base_ativos["professor_duplicado_parametros"] = base_ativos.duplicated("chave_docente", keep=False)

    total_docentes_parametros_linhas = int(len(base_docentes))
    total_docentes_parametros_unicos = int(base_docentes["professor_key"].nunique())
    total_docentes_ativos_linhas = int(len(base_ativos))
    total_docentes_ativos_unicos = int(base_ativos["professor_key"].nunique())
    total_docentes_excluidos_linhas = int(len(base_docentes) - len(base_ativos))
    total_ch_letiva_ativos = float(base_ativos["ch_letiva"].fillna(0).sum())
    total_ch_letiva_alocavel_ativos = float(base_ativos["ch_letiva_alocavel"].fillna(0).sum())

    meta = {
        "sheet": sheet,
        "nome_col": nome_col,
        "chapa_col": chapa_col,
        "ch_semanal_col": ch_semanal_col,
        "ch_letiva_col": ch_letiva_col,
        "status_col": status_col,
        "cargo_col": cargo_col,
        "total_docentes_parametros": total_docentes_parametros_linhas,
        "total_docentes_parametros_unicos": total_docentes_parametros_unicos,
        "total_docentes_ativos": total_docentes_ativos_linhas,
        "total_docentes_ativos_unicos": total_docentes_ativos_unicos,
        "total_docentes_excluidos_capacidade": total_docentes_excluidos_linhas,
        "total_ch_letiva_ativos": total_ch_letiva_ativos,
        "total_ch_letiva_alocavel_ativos": total_ch_letiva_alocavel_ativos,
        "regra_capacidade": "Somente linhas com STATUS = ATIVO; CH letiva alocável = piso(CH letiva/2) × 2; sem deduplicar registros da aba PARAMETROS.",
        # Mantemos também a base completa em memória para governança cadastral.
        # A capacidade continua usando apenas base_ativos, mas a auditoria de alocações
        # consegue diferenciar: (1) docente sem cadastro em PARAMETROS e (2) docente
        # cadastrado em PARAMETROS com STATUS diferente de ATIVO.
        "base_docentes_todos": base_docentes.copy(),
    }
    return base_ativos, sheet, meta

base = Path(".")
mapa_path = Path(MAPA_PATH) if MAPA_PATH else locate_file(["*MAPA*.xlsx", "*mapa*.xlsx"], base)
registro_path = Path(REGISTRO_PATH) if REGISTRO_PATH else locate_file(["*REGISTRO*.xlsx", "*registro*.xlsx"], base)
professores_path = Path(PROFESSORES_PATH) if PROFESSORES_PATH else locate_professores_file(base)

mapa_raw, mapa_sheet, mapa_meta = carregar_mapa(mapa_path)
registro_raw, registro_sheet, registro_header, registro_meta = carregar_registro(registro_path)
registro_raw_bruto = registro_raw.copy()

# v52 - As duplicatas do Registro são apenas identificadas para governança.
# Nenhuma linha é retirada dos cálculos, pois a qualidade/correção do Registro deve refletir automaticamente nos resultados.
registro_raw_dedup_ref, registro_duplicatas_exatas, registro_duplicatas_removidas = deduplicar_registro_exato(registro_raw.copy())
registro_raw = registro_raw_bruto.copy()

registro_linhas_brutas = len(registro_raw_bruto)
registro_linhas_usadas = len(registro_raw)
registro_duplicatas_exatas_linhas = len(registro_duplicatas_exatas)
registro_duplicatas_removidas_linhas = len(registro_duplicatas_removidas)
registro_duplicatas_exatas_grupos = int(registro_duplicatas_exatas["grupo_duplicata_exata"].nunique()) if not registro_duplicatas_exatas.empty else 0
parametros_docentes, parametros_sheet, parametros_meta = carregar_base_docentes_capacidade(professores_path, registro_path)

# Base de docentes presenciais (CH contratada) — opcional, usada apenas na seção
# informativa "Docentes Presencial (caso à parte)". Não participa de nenhum total geral.
docentes_presencial_path = locate_docentes_presencial_file(base)
if docentes_presencial_path is not None:
    docentes_presencial_base, docentes_presencial_sheet, docentes_presencial_meta = carregar_docentes_presencial(docentes_presencial_path)
    print(f"Base de docentes presenciais localizada: {docentes_presencial_path.name} ({len(docentes_presencial_base)} registros)")
else:
    docentes_presencial_base = pd.DataFrame()
    docentes_presencial_sheet = None
    docentes_presencial_meta = {}
    print("Base de docentes presenciais não localizada neste mês — seção 'Docentes Presencial (caso à parte)' não será gerada.")
parametros_docentes_todos = parametros_meta.get("base_docentes_todos", parametros_docentes).copy()
parametros_total_docentes = int(parametros_meta.get("total_docentes_parametros", len(parametros_docentes)))
parametros_docentes_ativos = int(parametros_meta.get("total_docentes_ativos", len(parametros_docentes)))
parametros_docentes_ativos_unicos = int(parametros_meta.get("total_docentes_ativos_unicos", parametros_docentes_ativos))
parametros_docentes_excluidos = int(parametros_meta.get("total_docentes_excluidos_capacidade", 0))
parametros_ch_letiva_ativos = float(parametros_meta.get("total_ch_letiva_ativos", parametros_docentes["ch_letiva"].fillna(0).sum() if "ch_letiva" in parametros_docentes else 0))
parametros_ch_letiva_alocavel_ativos = float(parametros_meta.get("total_ch_letiva_alocavel_ativos", parametros_docentes["ch_letiva_alocavel"].fillna(0).sum() if "ch_letiva_alocavel" in parametros_docentes else 0))

if MODULO_ALVO is None:
    modulo_alvo = int(mapa_raw["modulo"].dropna().mode().iloc[0])
else:
    modulo_alvo = int(MODULO_ALVO)

if ANO_ALVO is None:
    anos_possiveis = registro_raw.loc[registro_raw["modulo"].eq(modulo_alvo), "ano"].dropna()
    if len(anos_possiveis):
        ano_alvo = int(anos_possiveis.mode().iloc[0])
    else:
        anos_mapa = mapa_raw["ano"].dropna()
        ano_alvo = int(anos_mapa.mode().iloc[0]) if len(anos_mapa) else None
else:
    ano_alvo = int(ANO_ALVO)

registro_duplicatas_exatas_modulo = registro_duplicatas_exatas[registro_duplicatas_exatas["modulo"].eq(modulo_alvo)].copy() if not registro_duplicatas_exatas.empty else registro_duplicatas_exatas.copy()
registro_duplicatas_removidas_modulo = registro_duplicatas_removidas[registro_duplicatas_removidas["modulo"].eq(modulo_alvo)].copy() if not registro_duplicatas_removidas.empty else registro_duplicatas_removidas.copy()
registro_duplicatas_exatas_linhas_modulo = len(registro_duplicatas_exatas_modulo)
registro_duplicatas_removidas_linhas_modulo = len(registro_duplicatas_removidas_modulo)
registro_duplicatas_exatas_grupos_modulo = int(registro_duplicatas_exatas_modulo["grupo_duplicata_exata"].nunique()) if not registro_duplicatas_exatas_modulo.empty else 0

planejado = mapa_raw[
    mapa_raw["modulo"].eq(modulo_alvo) &
    mapa_raw["formato_ao_vivo"].eq(True) &
    mapa_raw["contrato"].isin(CONTRATOS_VALIDOS) &
    mapa_raw["disciplina_key"].ne("") &
    mapa_raw["ordem"].isin(ORDER_SORT)
].copy()

realizado_linhas_modulo = registro_raw[
    registro_raw["modulo"].eq(modulo_alvo) &
    registro_raw["disciplina_key"].ne("") &
    registro_raw["ordem"].isin(ORDER_SORT)
].copy()

key_cols = ["disciplina_key", "ordem", "dia_semana", "horario"]
planejado_key = planejado[
    planejado["disciplina_key"].ne("") & planejado["ordem"].ne("") & planejado["dia_semana"].ne("") & planejado["horario"].ne("")
].drop_duplicates(key_cols).copy()
planejado_key["chave_conformidade"] = make_key(planejado_key)

# ----------------------------------------------------------------
# Regra v23 — deduplicação operacional do Registro por chave de aula
# ----------------------------------------------------------------
# Ato 2 já trabalha por chave operacional Disciplina + Ordem + Dia + Horário.
# A visão docente e todas as demais análises de realizado devem usar exatamente
# a mesma base para não inflar horas por linhas repetidas ou por duplicidades
# operacionais na aba Registro.
realizado_base_chave = realizado_linhas_modulo[
    realizado_linhas_modulo["disciplina_key"].ne("") &
    realizado_linhas_modulo["ordem"].ne("") &
    realizado_linhas_modulo["dia_semana"].ne("") &
    realizado_linhas_modulo["horario"].ne("")
].copy()
realizado_base_chave["chave_conformidade"] = make_key(realizado_base_chave)
realizado_base_chave["duplicata_chave_operacional_registro"] = realizado_base_chave.duplicated(subset=key_cols, keep=False)
realizado_base_chave["removida_por_duplicata_chave_operacional"] = realizado_base_chave.duplicated(subset=key_cols, keep="first")

registro_duplicatas_chave_modulo = realizado_base_chave[realizado_base_chave["duplicata_chave_operacional_registro"].eq(True)].copy()
registro_duplicatas_chave_removidas_modulo = realizado_base_chave[realizado_base_chave["removida_por_duplicata_chave_operacional"].eq(True)].copy()
if not registro_duplicatas_chave_modulo.empty:
    contagem_chave = (
        registro_duplicatas_chave_modulo
        .groupby("chave_conformidade")
        .size()
        .rename("qtd_linhas_chave")
        .reset_index()
    )
    mantidas_chave = (
        realizado_base_chave[
            realizado_base_chave["duplicata_chave_operacional_registro"].eq(True) &
            realizado_base_chave["removida_por_duplicata_chave_operacional"].eq(False)
        ][["chave_conformidade", "linha_origem"]]
        .rename(columns={"linha_origem": "linha_mantida_calculo"})
    )
    registro_duplicatas_chave_modulo = (
        registro_duplicatas_chave_modulo
        .merge(contagem_chave, on="chave_conformidade", how="left")
        .merge(mantidas_chave, on="chave_conformidade", how="left")
    )
    registro_duplicatas_chave_modulo["regra_tratamento"] = (
        "Duplicidade operacional por Disciplina + Ordem + Dia + Horário: "
        "primeira ocorrência mantida; repetições fora dos cálculos."
    )
    registro_duplicatas_chave_removidas_modulo = (
        registro_duplicatas_chave_removidas_modulo
        .merge(contagem_chave, on="chave_conformidade", how="left")
        .merge(mantidas_chave, on="chave_conformidade", how="left")
    )
    registro_duplicatas_chave_removidas_modulo["regra_tratamento"] = (
        "Linha removida dos cálculos por repetir a mesma chave operacional de aula no Registro."
    )

# v52 - Não removemos duplicidades operacionais do Registro.
# Todas as linhas registradas entram nos cálculos; as duplicidades ficam como alerta/governança.
realizado = realizado_base_chave.copy()
realizado_key = realizado.copy()
realizado_key["chave_conformidade"] = make_key(realizado_key)

registro_duplicatas_chave_linhas_modulo = len(registro_duplicatas_chave_modulo)
registro_duplicatas_chave_removidas_linhas_modulo = len(registro_duplicatas_chave_removidas_modulo)
registro_duplicatas_chave_grupos_modulo = int(registro_duplicatas_chave_modulo["chave_conformidade"].nunique()) if not registro_duplicatas_chave_modulo.empty else 0

section("Leitura e ETL concluídos", "Arquivos, abas e cabeçalhos detectados automaticamente para uso em módulos futuros.")
kpi_cards([
    {"label": "Arquivo Mapa", "value": mapa_path.name, "note": f"Aba: {mapa_sheet}", "color": COLORS["planejado"]},
    {"label": "Arquivo Registro", "value": registro_path.name, "note": f"Aba: {registro_sheet} | Cabeçalho linha {registro_header + 1}", "color": COLORS["realizado"]},
    {"label": "Docentes ativos", "value": br_int(parametros_docentes_ativos), "note": f"{br_int(parametros_docentes_ativos_unicos)} únicos | CH letiva {br_int(parametros_ch_letiva_ativos)}h | alocável {br_int(parametros_ch_letiva_alocavel_ativos)}h", "color": COLORS["horas"]},
    {"label": "Módulo analisado", "value": str(modulo_alvo), "note": f"Ano: {ano_alvo if ano_alvo else 'não identificado'}", "color": COLORS["neutro"]},
    {"label": "Chave de conferência", "value": "4 campos", "note": "Disciplina + Ordem + Dia + Horário", "color": COLORS["alerta"]},
    {"label": "Duplicatas removidas", "value": br_int(registro_duplicatas_removidas_linhas_modulo + registro_duplicatas_chave_removidas_linhas_modulo), "note": f"Exatas: {br_int(registro_duplicatas_removidas_linhas_modulo)} | Chave de aula: {br_int(registro_duplicatas_chave_removidas_linhas_modulo)}", "color": COLORS["gap"] if (registro_duplicatas_removidas_linhas_modulo + registro_duplicatas_chave_removidas_linhas_modulo) else COLORS["ok"]},
])

display(pd.DataFrame([
    {"Fonte": "Mapa", "Arquivo": mapa_path.name, "Aba": mapa_sheet, "Uso": "Planejado", "Coluna formato AO VIVO": mapa_meta.get("format_col"), "Coluna RPA/valor": mapa_meta.get("valor_col")},
    {"Fonte": "Registro", "Arquivo": registro_path.name, "Aba": registro_sheet, "Uso": "Realizado (todas as linhas do Registro; duplicidades apenas sinalizadas para governança)", "Coluna formato AO VIVO": "-", "Coluna RPA/valor": registro_meta.get("rpa_col")},
    {"Fonte": "Base docente", "Arquivo": (professores_path.name if professores_path else parametros_meta.get("arquivo", registro_path.name)), "Aba": parametros_sheet, "Uso": "Banco atualizado de CH letiva e CH alocável (somente ATIVOS + alocados por regra)", "Coluna formato AO VIVO": "-", "Coluna RPA/valor": "-"},
]))

In [ ]:
# ================================================================
# MÉTRICAS
# ================================================================

def summarize_order_counts(df_key: pd.DataFrame) -> pd.DataFrame:
    out = df_key.groupby("ordem", dropna=False).size().reindex(ORDER_SORT, fill_value=0).reset_index()
    out.columns = ["ordem", "disciplinas"]
    return out


def horas_por_periodo_from_order_counts(order_counts: pd.DataFrame) -> pd.DataFrame:
    vals = dict(zip(order_counts["ordem"], order_counts["disciplinas"]))
    p1_qtd = int(vals.get("1ª", 0) + vals.get("ESTENDIDA", 0))
    p2_qtd = int(vals.get("2ª", 0) + vals.get("ESTENDIDA", 0))
    return pd.DataFrame([
        {"periodo": "Período 1", "composicao": "1ª + ESTENDIDA", "disciplinas": p1_qtd, "horas": p1_qtd * 2},
        {"periodo": "Período 2", "composicao": "2ª + ESTENDIDA", "disciplinas": p2_qtd, "horas": p2_qtd * 2},
    ])


def aggregate_planned_rpa(df: pd.DataFrame) -> pd.DataFrame:
    rpa = df[df["contrato"].eq("RPA/NF")].copy()
    if rpa.empty:
        return pd.DataFrame(columns=["disciplina_key", "disciplina", "ordem", "dia_semana", "horario", "valor_rpa_mapa", "valor_informado", "valor_textual", "professor_planejado", "curso_nome"])
    agg = rpa.groupby(key_cols, dropna=False).agg(
        disciplina=("disciplina", safe_first),
        valor_rpa_mapa=("valor_rpa_mapa", "max"),
        valor_textual=("valor_rpa_mapa_raw", safe_join_unique),
        professor_planejado=("professor_planejado", safe_join_unique),
        curso_nome=("curso_nome", safe_join_unique),
        modulo=("modulo", "first"),
        qtd_linhas_mapa=("linha_origem", "count"),
    ).reset_index()
    agg["valor_informado"] = agg["valor_rpa_mapa"].notna()
    agg["chave_conformidade"] = make_key(agg)
    return agg


def aggregate_realizado_rpa(df: pd.DataFrame) -> pd.DataFrame:
    # Conta TODOS os RPAs realizados/sinalizados, com ou sem valor financeiro.
    # O valor financeiro continua sendo somado apenas quando preenchido.
    if "is_rpa_registro" in df.columns:
        rpa = df[df["is_rpa_registro"].eq(True)].copy()
    else:
        rpa = df[df["tem_rpa_registro"].eq(True)].copy()
    if rpa.empty:
        return pd.DataFrame(columns=["disciplina_key", "disciplina", "ordem", "dia_semana", "horario", "valor_rpa_registro", "valor_informado", "valor_textual", "professor_realizado", "cargo"])
    agg = rpa.groupby(key_cols, dropna=False).agg(
        disciplina=("disciplina", safe_first),
        valor_rpa_registro=("valor_rpa_registro", "max"),
        valor_textual=("valor_rpa_registro_raw", safe_join_unique),
        professor_realizado=("professor_realizado", safe_join_unique),
        cargo=("cargo", safe_join_unique),
        modulo=("modulo", "first"),
        qtd_linhas_registro=("linha_origem", "count"),
    ).reset_index()
    agg["valor_informado"] = agg["valor_rpa_registro"].notna() & agg["valor_rpa_registro"].fillna(0).gt(0)
    agg["chave_conformidade"] = make_key(agg)
    return agg

def compute_compliance(plan_keys: pd.DataFrame, real_keys: pd.DataFrame):
    p = plan_keys[key_cols + ["disciplina", "chave_conformidade"]].drop_duplicates("chave_conformidade").copy()
    r = real_keys[key_cols + ["disciplina", "chave_conformidade"]].drop_duplicates("chave_conformidade").copy()
    merged = p.merge(r[key_cols + ["chave_conformidade"]], on=key_cols, how="outer", indicator=True, suffixes=("_mapa", "_registro"))
    merged["status"] = np.select(
        [merged["_merge"].eq("both"), merged["_merge"].eq("left_only"), merged["_merge"].eq("right_only")],
        ["Conforme", "Planejado sem realizado", "Realizado sem planejado"],
        default="Verificar"
    )
    return merged, merged[merged["status"].eq("Conforme")].copy(), merged[merged["status"].eq("Planejado sem realizado")].copy(), merged[merged["status"].eq("Realizado sem planejado")].copy()


def compute_professor_divergences(plan: pd.DataFrame, real: pd.DataFrame, conforme_keys: pd.DataFrame) -> pd.DataFrame:
    if conforme_keys.empty:
        return pd.DataFrame()
    p_prof = plan.groupby(key_cols).agg(
        disciplina=("disciplina", safe_first),
        professor_planejado=("professor_planejado", safe_join_unique),
        professor_planejado_key=("professor_planejado_key", safe_join_unique),
        contrato=("contrato", safe_join_unique),
    ).reset_index()
    r_prof = real.groupby(key_cols).agg(
        professor_realizado=("professor_realizado", safe_join_unique),
        professor_realizado_key=("professor_realizado_key", safe_join_unique),
        cargo_realizado=("cargo", safe_join_unique),
    ).reset_index()
    base = conforme_keys[key_cols].drop_duplicates().merge(p_prof, on=key_cols, how="left").merge(r_prof, on=key_cols, how="left")
    invalid = {"", "NSA", "NAN", "NONE", "-", "A DEFINIR"}
    base["prof_plan_norm"] = base["professor_planejado"].map(normalize_key)
    base["prof_real_norm"] = base["professor_realizado"].map(normalize_key)
    base["divergencia_professor"] = base["prof_plan_norm"].ne(base["prof_real_norm"]) & ~base["prof_plan_norm"].isin(invalid) & ~base["prof_real_norm"].isin(invalid)
    return base[base["divergencia_professor"]].copy()


def compute_shift_differences(plan_keys: pd.DataFrame, real_keys: pd.DataFrame) -> pd.DataFrame:
    p = plan_keys[key_cols + ["disciplina"]].drop_duplicates().copy()
    r = real_keys[key_cols + ["disciplina"]].drop_duplicates().copy()
    base = p.merge(r, on=["disciplina_key", "ordem"], how="inner", suffixes=("_mapa", "_registro"))
    diff = base[(base["dia_semana_mapa"].ne(base["dia_semana_registro"])) | (base["horario_mapa"].ne(base["horario_registro"]))].copy()
    diff["key_mapa"] = diff["disciplina_key"] + "||" + diff["ordem"] + "||" + diff["dia_semana_mapa"] + "||" + diff["horario_mapa"]
    diff["key_reg"] = diff["disciplina_key"] + "||" + diff["ordem"] + "||" + diff["dia_semana_registro"] + "||" + diff["horario_registro"]
    diff = diff[diff["key_mapa"].ne(diff["key_reg"])].copy()
    return diff.drop_duplicates(["disciplina_key", "ordem", "dia_semana_mapa", "horario_mapa", "dia_semana_registro", "horario_registro"])


def compute_total_duplicates(df: pd.DataFrame, fonte: str) -> pd.DataFrame:
    """Identifica duplicata operacional respeitando sinergias do Mapa.

    Regras:
    - A chave de conformidade Mapa x Registro permanece: Disciplina + Ordem + Dia + Horário.
    - Para duplicidade no Mapa, a auditoria NÃO deve penalizar sinergias: uma mesma aula pode atender cursos diferentes.
    - Por isso, no Mapa o critério de duplicata inclui Curso e Código da Disciplina quando disponíveis.
    - No Registro, como a base não possui curso, o critério permanece Disciplina + Ordem + Dia + Horário.
    """
    base_cols = ["disciplina_key", "ordem", "dia_semana", "horario"]
    empty_cols = [
        "fonte", "criterio_duplicata", "disciplina", "disciplina_codigo", "ordem", "dia_semana", "horario",
        "curso_codigo", "curso_nome", "sinergia", "qtd_linhas", "linhas_origem",
        "professores_planejados", "professores_realizados", "cargos", "cursos"
    ]
    if df is None or df.empty:
        return pd.DataFrame(columns=empty_cols)

    d = df.copy()
    for c in base_cols:
        if c not in d.columns:
            d[c] = ""
    d = d[d["disciplina_key"].ne("") & d["ordem"].ne("") & d["dia_semana"].ne("") & d["horario"].ne("")].copy()
    if d.empty:
        return pd.DataFrame(columns=empty_cols)

    group_cols = base_cols.copy()
    criterio = "Disciplina + Ordem + Dia + Horário"

    # Ajuste de sinergia: no Mapa, cursos diferentes podem assistir a mesma aula.
    # Portanto, curso e código de disciplina entram no critério de duplicidade operacional quando existirem.
    if normalize_key(fonte) == "MAPA":
        if "curso_key" in d.columns and d["curso_key"].map(normalize_text).ne("").any():
            group_cols.append("curso_key")
            criterio += " + Curso"
        elif "curso_codigo" in d.columns and d["curso_codigo"].map(normalize_text).ne("").any():
            group_cols.append("curso_codigo")
            criterio += " + Curso"
        elif "curso_nome" in d.columns and d["curso_nome"].map(normalize_text).ne("").any():
            group_cols.append("curso_nome")
            criterio += " + Curso"
        if "disciplina_codigo_key" in d.columns and d["disciplina_codigo_key"].map(normalize_text).ne("").any():
            group_cols.append("disciplina_codigo_key")
            criterio += " + Código Disciplina"

    agg = {
        "disciplina": ("disciplina", safe_first),
        "qtd_linhas": ("linha_origem", "count"),
        "linhas_origem": ("linha_origem", lambda s: " | ".join(dict.fromkeys([str(int(x)) for x in s.dropna().tolist()]))),
    }
    optional_aggs = {
        "disciplina_codigo": ("disciplina_codigo", safe_join_unique),
        "curso_codigo": ("curso_codigo", safe_join_unique),
        "curso_nome": ("curso_nome", safe_join_unique),
        "sinergia": ("sinergia", safe_join_unique),
        "professores_planejados": ("professor_planejado", safe_join_unique),
        "professores_realizados": ("professor_realizado", safe_join_unique),
        "cargos": ("cargo", safe_join_unique),
        "cursos": ("curso_nome", safe_join_unique),
    }
    for out_col, spec in optional_aggs.items():
        if spec[0] in d.columns:
            agg[out_col] = spec

    out = d.groupby(group_cols, dropna=False).agg(**agg).reset_index()
    out = out[out["qtd_linhas"].gt(1)].copy()
    out.insert(0, "fonte", fonte)
    out.insert(1, "criterio_duplicata", criterio)
    for col in empty_cols:
        if col not in out.columns:
            out[col] = ""
    return out[empty_cols].sort_values(["qtd_linhas", "fonte", "ordem", "dia_semana", "horario", "disciplina"], ascending=[False, True, True, True, True, True])


def heatmap_data(df_key: pd.DataFrame) -> pd.DataFrame:
    valid = df_key[df_key["dia_semana"].isin(DAY_SORT) & df_key["ordem"].isin(ORDER_SORT)].copy()
    pivot = valid.pivot_table(index="ordem", columns="dia_semana", values="disciplina_key", aggfunc="count", fill_value=0)
    pivot = pivot.reindex(index=ORDER_SORT, columns=[d for d in DAY_SORT if d in pivot.columns], fill_value=0)
    return pivot


def rpa_by_module(rpa_df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    if rpa_df.empty or "modulo" not in rpa_df.columns:
        return pd.DataFrame(columns=["modulo", "qtd_rpa", "qtd_rpa_com_valor", "qtd_rpa_sem_valor", "valor_rpa"])
    out = rpa_df.groupby("modulo", dropna=False).agg(
        qtd_rpa=("chave_conformidade", "nunique"),
        qtd_rpa_com_valor=("valor_informado", "sum"),
        valor_rpa=(value_col, "sum"),
    ).reset_index()
    out["qtd_rpa_sem_valor"] = out["qtd_rpa"] - out["qtd_rpa_com_valor"]
    return out

planejado_ordem = summarize_order_counts(planejado_key)
realizado_ordem = summarize_order_counts(realizado_key)
horas_planejado = horas_por_periodo_from_order_counts(planejado_ordem)
horas_realizado = horas_por_periodo_from_order_counts(realizado_ordem)

# Base auditável do Mapa usada no processamento: exatamente as disciplinas AO VIVO consideradas no Planejado.
mapa_ao_vivo_cols = [
    "linha_origem", "modulo", "ano", "curso_codigo", "curso_nome", "disciplina_codigo", "disciplina", "ordem",
    "dia_semana", "horario", "contrato", "formato", "professor_planejado", "valor_rpa_mapa",
    "valor_rpa_mapa_raw", "sinergia", "disciplina_key", "curso_key", "chave_conformidade"
]
mapa_ao_vivo_usado = planejado.copy()
mapa_ao_vivo_usado["chave_conformidade"] = make_key(mapa_ao_vivo_usado)
mapa_ao_vivo_usado = mapa_ao_vivo_usado[[c for c in mapa_ao_vivo_cols if c in mapa_ao_vivo_usado.columns]].sort_values([c for c in ["ordem", "dia_semana", "horario", "disciplina", "curso_nome"] if c in mapa_ao_vivo_usado.columns])

rpa_mapa = aggregate_planned_rpa(planejado)
rpa_registro = aggregate_realizado_rpa(realizado)
# As bases de RPA já carregam o módulo a partir das planilhas originais.

comparativo_chaves, conformes, planejado_sem_realizado, realizado_sem_planejado = compute_compliance(planejado_key, realizado_key)
professor_divergencias = compute_professor_divergences(planejado, realizado, conformes)
# A mesma disciplina + ordem em dias/horários diferentes é uma possibilidade normal do processo.
# Portanto, a auditoria passa a tratar como alerta apenas a duplicata total da chave completa.
diferencas_dia_horario = compute_shift_differences(planejado_key, realizado_key)  # mantido somente para diagnóstico, não entra como gap.
duplicatas_mapa = compute_total_duplicates(planejado, "Mapa")
duplicatas_registro = compute_total_duplicates(realizado, "Registro")
duplicatas_totais = pd.concat([duplicatas_mapa, duplicatas_registro], ignore_index=True) if (not duplicatas_mapa.empty or not duplicatas_registro.empty) else pd.DataFrame(columns=["fonte", "criterio_duplicata", "disciplina", "disciplina_codigo", "ordem", "dia_semana", "horario", "curso_codigo", "curso_nome", "sinergia", "qtd_linhas", "linhas_origem", "professores_planejados", "professores_realizados", "cargos", "cursos"])

comparativo_horas = horas_planejado.merge(horas_realizado, on=["periodo", "composicao"], how="outer", suffixes=("_planejado", "_realizado")).fillna(0)
comparativo_horas["delta_horas"] = comparativo_horas["horas_realizado"] - comparativo_horas["horas_planejado"]
comparativo_horas["aderencia_%"] = np.where(comparativo_horas["horas_planejado"].gt(0), comparativo_horas["horas_realizado"] / comparativo_horas["horas_planejado"] * 100, np.nan)

qtd_rpa_mapa_total = int(rpa_mapa["chave_conformidade"].nunique()) if not rpa_mapa.empty else 0
qtd_rpa_mapa_com_valor = int(rpa_mapa["valor_informado"].sum()) if not rpa_mapa.empty else 0
qtd_rpa_mapa_sem_valor = max(qtd_rpa_mapa_total - qtd_rpa_mapa_com_valor, 0)
valor_rpa_mapa_total = float(rpa_mapa["valor_rpa_mapa"].sum(skipna=True)) if not rpa_mapa.empty else 0.0

qtd_rpa_reg_total = int(rpa_registro["chave_conformidade"].nunique()) if not rpa_registro.empty else 0
qtd_rpa_reg_com_valor = int(rpa_registro["valor_informado"].sum()) if not rpa_registro.empty else 0
qtd_rpa_reg_sem_valor = max(qtd_rpa_reg_total - qtd_rpa_reg_com_valor, 0)
valor_rpa_reg_total = float(rpa_registro["valor_rpa_registro"].sum(skipna=True)) if not rpa_registro.empty else 0.0

rpa_comp = pd.DataFrame([
    {"origem": "Mapa", "qtd_rpa": qtd_rpa_mapa_total, "qtd_rpa_com_valor": qtd_rpa_mapa_com_valor, "qtd_rpa_sem_valor": qtd_rpa_mapa_sem_valor, "valor_rpa": valor_rpa_mapa_total},
    {"origem": "Registro", "qtd_rpa": qtd_rpa_reg_total, "qtd_rpa_com_valor": qtd_rpa_reg_com_valor, "qtd_rpa_sem_valor": qtd_rpa_reg_sem_valor, "valor_rpa": valor_rpa_reg_total},
])

qtd_planejado = len(planejado_key)
qtd_realizado = len(realizado_key)
qtd_conforme = len(conformes)
qtd_faltantes = len(planejado_sem_realizado)
qtd_extras = len(realizado_sem_planejado)
qtd_duplicatas_mapa = len(duplicatas_mapa)
qtd_duplicatas_registro = len(duplicatas_registro)
qtd_duplicatas_total = len(duplicatas_totais)
conformidade_pct = (qtd_conforme / qtd_planejado * 100) if qtd_planejado else 0

# v24: a cor executiva passa a refletir apenas a conformidade operacional.
if conformidade_pct >= 95:
    conformidade_cor = COLORS["ok"]
elif conformidade_pct >= 85:
    conformidade_cor = COLORS["alerta"]
else:
    conformidade_cor = COLORS["gap"]

section("Métricas calculadas", "Base analítica pronta para Planejado, Realizado, Comparações e Resultados.")
kpi_cards([
    {"label": "Disciplinas planejadas", "value": br_int(qtd_planejado), "note": "Disciplina + Ordem + Dia + Horário", "color": COLORS["planejado"]},
    {"label": "Disciplinas realizadas", "value": br_int(qtd_realizado), "note": "Registro filtrado para o módulo", "color": COLORS["realizado"]},
    {"label": "Disciplinas previstas realizadas", "value": br_pct(conformidade_pct), "note": f"{br_int(qtd_conforme)} de {br_int(qtd_planejado)} disciplinas previstas", "color": conformidade_cor},
])

display(Markdown("### Contagem por ordem — base da visão de horas"))
contagem_ordem = planejado_ordem.merge(realizado_ordem, on="ordem", suffixes=("_planejado", "_realizado"))
display(contagem_ordem)

display(Markdown("### Horas por período — quantidade de disciplinas × 2"))
display(comparativo_horas)

In [ ]:
# ================================================================
# ATO 1 — PLANEJADO
# ================================================================
section("Ato 1 — Planejado", "Visão executiva do que o Mapa Pedagógico sinaliza para disciplinas AO VIVO.")

kpi_cards([
    {"label": "Planejado — 1ª ordem", "value": br_int(planejado_ordem.loc[planejado_ordem['ordem'].eq('1ª'), 'disciplinas'].sum()), "note": "Cor fixa: azul", "color": ORDER_COLORS["1ª"]},
    {"label": "Planejado — 2ª ordem", "value": br_int(planejado_ordem.loc[planejado_ordem['ordem'].eq('2ª'), 'disciplinas'].sum()), "note": "Cor fixa: verde", "color": ORDER_COLORS["2ª"]},
    {"label": "Planejado — Estendida", "value": br_int(planejado_ordem.loc[planejado_ordem['ordem'].eq('ESTENDIDA'), 'disciplinas'].sum()), "note": "Cor fixa: roxo", "color": ORDER_COLORS["ESTENDIDA"]},
    {"label": "Horas planejadas P1", "value": br_int(horas_planejado.loc[horas_planejado['periodo'].eq('Período 1'), 'horas'].sum()), "note": "1ª + ESTENDIDA × 2", "color": COLORS["horas"]},
    {"label": "Horas planejadas P2", "value": br_int(horas_planejado.loc[horas_planejado['periodo'].eq('Período 2'), 'horas'].sum()), "note": "2ª + ESTENDIDA × 2", "color": COLORS["horas"]},
    {"label": "RPA no Mapa", "value": br_int(qtd_rpa_mapa_total), "note": f"{br_money(valor_rpa_mapa_total)} • {br_int(qtd_rpa_mapa_com_valor)} com valor | {br_int(qtd_rpa_mapa_sem_valor)} sem valor", "color": COLORS["rpa"]},
])

fig = px.bar(planejado_ordem, x="ordem", y="disciplinas", text="disciplinas", color="ordem", color_discrete_map=ORDER_COLORS, category_orders={"ordem": ORDER_SORT}, labels={"ordem": "Ordem", "disciplinas": "Quantidade"})
fig.update_traces(textposition="outside", textfont_size=16, marker_line_width=0)
show_fig(aplicar_layout(fig, "Planejado por ordem — chave de conferência", height=460))

fig = px.bar(horas_planejado, x="periodo", y="horas", text="horas", color="periodo", color_discrete_map={"Período 1": COLORS["planejado"], "Período 2": COLORS["horas"]}, labels={"periodo": "Período", "horas": "Horas equivalentes (disciplinas × 2)", "composicao": "Disciplinas planejadas"}, hover_data={"composicao": True, "disciplinas": True})
fig.update_traces(textposition="outside", textfont_size=16)
show_fig(aplicar_layout(fig, "Horas planejadas por período", height=460))

hm_plan = heatmap_data(planejado_key)
_hm_plan_vmin, _hm_plan_vmax = limites_heatmap(hm_plan.values)
fig = go.Figure(data=go.Heatmap(z=hm_plan.values, x=hm_plan.columns.tolist(), y=hm_plan.index.tolist(), colorscale=escala_heatmap_gamma(HEATMAP_PALETAS["planejado"]), zmin=_hm_plan_vmin, zmax=_hm_plan_vmax, text=hm_plan.values, texttemplate="%{text}", textfont={"size": 16}, hovertemplate="Dia: %{x}<br>Ordem: %{y}<br>Quantidade: %{z}<extra></extra>"))
fig = aplicar_layout(fig, "Mapa de calor do Planejado — eixo X por dia da semana", height=420)
fig.update_xaxes(title="Dia da semana")
fig.update_yaxes(title="Ordem")
show_fig(fig)

contrato_dist = planejado.drop_duplicates(key_cols + ["contrato"]).groupby("contrato").size().reset_index(name="quantidade").sort_values("quantidade", ascending=True)
fig = px.bar(contrato_dist, x="quantidade", y="contrato", orientation="h", text="quantidade", color="contrato", color_discrete_map={"CLT EAD": COLORS["planejado"], "CLT STRICTO": COLORS["neutro"], "RPA/NF": COLORS["rpa"]}, labels={"quantidade": "Quantidade", "contrato": "Modelo de contrato"})
fig.update_traces(textposition="outside")
show_fig(aplicar_layout(fig, "Planejado por modelo de contrato", height=420))

if not rpa_mapa.empty:
    rpa_cursos = planejado[planejado["contrato"].eq("RPA/NF") & planejado["curso_nome"].ne("")].copy()
    rpa_cursos = rpa_cursos.drop_duplicates(key_cols + ["curso_nome"]).groupby("curso_nome").size().reset_index(name="qtd_rpa").sort_values("qtd_rpa", ascending=False).head(15).sort_values("qtd_rpa")
    fig = px.bar(rpa_cursos, x="qtd_rpa", y="curso_nome", orientation="h", text="qtd_rpa", color_discrete_sequence=[COLORS["rpa"]], labels={"qtd_rpa": "Quantidade RPA", "curso_nome": "Curso"})
    fig.update_traces(textposition="outside")
    show_fig(aplicar_layout(fig, "Top cursos com maior sinalização de RPA no Mapa", height=max(480, 28 * len(rpa_cursos))))
    display(Markdown("### Detalhamento RPA sinalizado no Mapa"))
    display(rpa_mapa[["disciplina", "ordem", "dia_semana", "horario", "professor_planejado", "valor_rpa_mapa", "valor_textual", "curso_nome"]].sort_values(["ordem", "dia_semana", "horario", "disciplina"]).head(100))
else:
    insight_box("RPA no Mapa", "Não há sinalização de RPA/NF no Mapa para o módulo analisado.", COLORS["rpa"])

In [ ]:
# ================================================================
# ATO 2 — REALIZADO
# ================================================================
section("Ato 2 — Realizado", "Visão do Registro mantendo as mesmas cores e a mesma lógica de horas.")

kpi_cards([
    {"label": "Realizado — 1ª ordem", "value": br_int(realizado_ordem.loc[realizado_ordem['ordem'].eq('1ª'), 'disciplinas'].sum()), "note": "Cor fixa: azul", "color": ORDER_COLORS["1ª"]},
    {"label": "Realizado — 2ª ordem", "value": br_int(realizado_ordem.loc[realizado_ordem['ordem'].eq('2ª'), 'disciplinas'].sum()), "note": "Cor fixa: verde", "color": ORDER_COLORS["2ª"]},
    {"label": "Realizado — Estendida", "value": br_int(realizado_ordem.loc[realizado_ordem['ordem'].eq('ESTENDIDA'), 'disciplinas'].sum()), "note": "Cor fixa: roxo", "color": ORDER_COLORS["ESTENDIDA"]},
    {"label": "Horas realizadas P1", "value": br_int(horas_realizado.loc[horas_realizado['periodo'].eq('Período 1'), 'horas'].sum()), "note": "1ª + ESTENDIDA × 2", "color": COLORS["horas"]},
    {"label": "Horas realizadas P2", "value": br_int(horas_realizado.loc[horas_realizado['periodo'].eq('Período 2'), 'horas'].sum()), "note": "2ª + ESTENDIDA × 2", "color": COLORS["horas"]},
    {"label": "RPA no Registro", "value": br_int(qtd_rpa_reg_total), "note": f"{br_money(valor_rpa_reg_total)} • {br_int(qtd_rpa_reg_com_valor)} com valor | {br_int(qtd_rpa_reg_sem_valor)} sem valor", "color": COLORS["rpa"]},
])

fig = px.bar(realizado_ordem, x="ordem", y="disciplinas", text="disciplinas", color="ordem", color_discrete_map=ORDER_COLORS, category_orders={"ordem": ORDER_SORT}, labels={"ordem": "Ordem", "disciplinas": "Quantidade"})
fig.update_traces(textposition="outside", textfont_size=16)
show_fig(aplicar_layout(fig, "Realizado por ordem", height=460))

hm_real = heatmap_data(realizado_key)
_hm_real_vmin, _hm_real_vmax = limites_heatmap(hm_real.values)
fig = go.Figure(data=go.Heatmap(z=hm_real.values, x=hm_real.columns.tolist(), y=hm_real.index.tolist(), colorscale=escala_heatmap_gamma(HEATMAP_PALETAS["realizado"]), zmin=_hm_real_vmin, zmax=_hm_real_vmax, text=hm_real.values, texttemplate="%{text}", textfont={"size": 16}, hovertemplate="Dia: %{x}<br>Ordem: %{y}<br>Quantidade: %{z}<extra></extra>"))
fig = aplicar_layout(fig, "Mapa de calor do Realizado — eixo X por dia da semana", height=420)
fig.update_xaxes(title="Dia da semana")
fig.update_yaxes(title="Ordem")
show_fig(fig)

prof_dist = realizado_key.copy()
prof_dist = prof_dist[prof_dist["professor_realizado"].map(normalize_key).ne("")].copy()
prof_top = prof_dist.groupby(["professor_realizado", "cargo"], dropna=False).size().reset_index(name="quantidade").sort_values("quantidade", ascending=False).head(15).sort_values("quantidade")
if not prof_top.empty:
    prof_top["docente_cargo"] = prof_top.apply(lambda r: f"{normalize_text(r['professor_realizado']).title()} — {normalize_text(r.get('cargo','')).title()}" if normalize_text(r.get('cargo','')) else normalize_text(r['professor_realizado']).title(), axis=1)
    fig = px.bar(prof_top, x="quantidade", y="docente_cargo", orientation="h", text="quantidade", color_discrete_sequence=[COLORS["realizado"]], labels={"quantidade": "Quantidade", "docente_cargo": "Professor(a) e cargo"}, hover_data={"professor_realizado": True, "cargo": True})
    fig.update_traces(textposition="outside")
    show_fig(aplicar_layout(fig, "Professores com maior volume no Registro — com cargo", height=max(520, 34 * len(prof_top))))

if not rpa_registro.empty:
    display(Markdown("### Detalhamento RPA sinalizado no Registro"))
    display(rpa_registro[["disciplina", "ordem", "dia_semana", "horario", "professor_realizado", "cargo", "valor_rpa_registro", "valor_informado", "valor_textual"]].sort_values(["ordem", "dia_semana", "horario", "disciplina"]))
else:
    insight_box("RPA no Registro", "Não há registros com valor de RPA para o módulo analisado.", COLORS["rpa"])

In [ ]:
# ================================================================
# ATO 3 — COMPARAÇÕES
# ================================================================
section("Ato 3 — Comparações", "Confronto Mapa × Registro pela chave Disciplina + Ordem + Dia + Horário.")

kpi_cards([
    {"label": "Conformes", "value": br_int(qtd_conforme), "note": "Encontrados no Mapa e no Registro", "color": COLORS["ok"]},
    {"label": "Planejado sem realizado", "value": br_int(qtd_faltantes), "note": "Existe no Mapa e não aparece no Registro", "color": COLORS["gap"]},
    {"label": "Realizado sem planejado", "value": br_int(qtd_extras), "note": "Existe no Registro e não aparece no Mapa", "color": COLORS["alerta"]},
    {"label": "Divergência professor", "value": br_int(len(professor_divergencias)), "note": "Chave confere, professor diferente", "color": COLORS["gap"]},
    {"label": "Duplicata operacional", "value": br_int(qtd_duplicatas_total), "note": "Mapa respeita sinergias por curso/código", "color": COLORS["alerta"]},
])

comp_horas_long = comparativo_horas.melt(id_vars=["periodo", "composicao"], value_vars=["horas_planejado", "horas_realizado"], var_name="origem", value_name="horas")
comp_horas_long["origem"] = comp_horas_long["origem"].map({"horas_planejado": "Planejado", "horas_realizado": "Realizado"})
fig = px.bar(comp_horas_long, x="periodo", y="horas", color="origem", barmode="group", text="horas", color_discrete_map={"Planejado": COLORS["planejado"], "Realizado": COLORS["realizado"]}, hover_data={"composicao": True}, labels={"periodo": "Período", "horas": "Horas equivalentes (disciplinas × 2)", "origem": "Origem", "composicao": "Disciplinas planejadas"})
fig.update_traces(textposition="outside", textfont_size=15)
show_fig(aplicar_layout(fig, "Planejado × Realizado — horas por período", height=500))

display(Markdown("### Tabela — Planejado × Realizado por período"))
display(comparativo_horas)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=rpa_comp["origem"], y=rpa_comp["qtd_rpa"], name="Quantidade RPA", marker_color=COLORS["rpa"], text=rpa_comp["qtd_rpa"], textposition="outside"), secondary_y=False)
fig.add_trace(go.Scatter(x=rpa_comp["origem"], y=rpa_comp["valor_rpa"], name="Valor RPA", mode="lines+markers+text", marker_color=COLORS["horas"], line={"width": 4}, text=[br_money(v) for v in rpa_comp["valor_rpa"]], textposition="top center"), secondary_y=True)
fig.update_yaxes(title_text="Quantidade RPA", secondary_y=False)
fig.update_yaxes(title_text="Valor RPA", secondary_y=True)
show_fig(aplicar_layout(fig, "RPA — quantidade e valor sinalizado no Mapa × Registro", height=500))

gap_df = pd.DataFrame([
    {"tipo": "Planejado sem realizado", "quantidade": qtd_faltantes, "cor": COLORS["gap"]},
    {"tipo": "Realizado sem planejado", "quantidade": qtd_extras, "cor": COLORS["alerta"]},
    {"tipo": "Divergência professor", "quantidade": len(professor_divergencias), "cor": COLORS["gap"]},
    {"tipo": "Duplicata total", "quantidade": qtd_duplicatas_total, "cor": COLORS["alerta"]},
]).sort_values("quantidade", ascending=True)
fig = px.bar(gap_df, x="quantidade", y="tipo", orientation="h", text="quantidade", color="tipo", color_discrete_map={row["tipo"]: row["cor"] for _, row in gap_df.iterrows()}, labels={"quantidade": "Quantidade", "tipo": "Tipo"})
fig.update_traces(textposition="outside")
show_fig(aplicar_layout(fig, "Mapa de gaps — prioridade de investigação", height=460))

if not planejado_sem_realizado.empty:
    display(Markdown("### Planejado sem realizado — investigar"))
    display(planejado_sem_realizado[["disciplina", "ordem", "dia_semana", "horario"]].sort_values(["ordem", "dia_semana", "horario", "disciplina"]).head(120))

if not realizado_sem_planejado.empty:
    display(Markdown("### Realizado sem planejado — investigar"))
    extras_display = realizado_sem_planejado[key_cols].merge(realizado_key[key_cols + ["disciplina"]].drop_duplicates(key_cols), on=key_cols, how="left")
    display(extras_display[["disciplina", "ordem", "dia_semana", "horario"]].sort_values(["ordem", "dia_semana", "horario", "disciplina"]).head(120))

if not professor_divergencias.empty:
    display(Markdown("### Divergências de professor — chave conforme, professor diferente"))
    display(professor_divergencias[["disciplina", "ordem", "dia_semana", "horario", "contrato", "professor_planejado", "professor_realizado", "cargo_realizado"]].sort_values(["ordem", "dia_semana", "horario", "disciplina"]).head(120))

insight_box(
    "Regra de leitura — dia e horário",
    "A mesma disciplina com a mesma ordem pode aparecer em dias e horários diferentes. Isso é tratado como variação operacional normal e não como divergência. O alerta desta seção considera somente duplicata total da chave completa: Disciplina + Ordem + Dia + Horário.",
    COLORS["neutro"]
)

if not registro_duplicatas_removidas_modulo.empty:
    display(Markdown("### Governança — duplicatas exatas removidas do Registro"))
    insight_box(
        "Duplicatas exatas no Registro",
        f"Foram removidas {br_int(registro_duplicatas_removidas_linhas_modulo)} linhas duplicadas exatas do módulo antes dos cálculos. A primeira ocorrência de cada grupo foi mantida para preservar a sessão realizada, e as repetições ficam listadas para governança.",
        COLORS["gap"]
    )
    cols_dup_reg = [c for c in ["grupo_duplicata_exata", "qtd_linhas_grupo", "linha_mantida_calculo", "linha_origem", "professor_realizado", "cargo", "modulo", "ano", "ordem", "dia_semana", "horario", "disciplina", "observacao", "regra_tratamento"] if c in registro_duplicatas_removidas_modulo.columns]
    display(registro_duplicatas_removidas_modulo[cols_dup_reg].sort_values(["grupo_duplicata_exata", "linha_origem"]).head(80))
else:
    insight_box("Duplicatas exatas no Registro", "Não foram encontradas duplicatas exatas no Registro para o módulo analisado.", COLORS["ok"])

if not duplicatas_totais.empty:
    display(Markdown("### Duplicatas operacionais — sinergias entre cursos tratadas corretamente"))
    dup_view = duplicatas_totais[[c for c in ["fonte", "criterio_duplicata", "disciplina", "disciplina_codigo", "ordem", "dia_semana", "horario", "curso_codigo", "curso_nome", "sinergia", "qtd_linhas", "linhas_origem", "professores_planejados", "professores_realizados", "cargos"] if c in duplicatas_totais.columns]].sort_values(["fonte", "ordem", "dia_semana", "horario", "disciplina"]).head(60).copy()
    for c in ["professores_planejados", "professores_realizados", "cargos", "linhas_origem"]:
        if c in dup_view.columns:
            dup_view[c] = dup_view[c].astype(str).str.slice(0, 120)
    display(dup_view)
    insight_box("Observação", "A listagem acima é compacta. No Mapa, a duplicidade operacional usa Curso e Código da Disciplina para não marcar como erro as sinergias entre cursos.", COLORS["alerta"])
else:
    insight_box("Duplicata operacional", "Não foram encontradas duplicatas operacionais após respeitar sinergias do Mapa por curso/código da disciplina.", COLORS["ok"])

In [ ]:
# ================================================================
# ATO 4 — VISÃO DOCENTE
# ================================================================
section("Ato 4 — Visão Docente", "Capacidade, alocação e delta usando a planilha atualizada de professores como banco oficial de capacidade docente.")

# Regra operacional:
# - Cada disciplina ocupa 2h da carga horária letiva.
# - A capacidade alocável por período é calculada em blocos de 2h: piso(CH letiva / 2) × 2.
# - O módulo possui dois períodos de 5 semanas:
#   Período 1 = 1ª + ESTENDIDA
#   Período 2 = 2ª + ESTENDIDA
# - A disciplina ESTENDIDA consome capacidade nos dois períodos.
# - O banco de capacidade vem da aba PARAMETROS, pois ela contém também docentes sem alocação.


def capacidade_alocavel_periodo(ch_letiva):
    try:
        if pd.isna(ch_letiva):
            return np.nan
        return np.floor(float(ch_letiva) / 2) * 2
    except Exception:
        return np.nan


def status_periodo(utilizacao, horas, capacidade):
    if pd.isna(capacidade) or capacidade <= 0:
        return "Sem CH letiva informada"
    if horas > capacidade:
        return "Acima da capacidade"
    if utilizacao >= 90:
        return "Alta utilização"
    if utilizacao >= 70:
        return "Boa utilização"
    if utilizacao >= 40:
        return "Utilização moderada"
    if horas > 0:
        return "Baixa utilização"
    return "Sem alocação"


def status_docente_geral(row):
    statuses = [row.get("status_p1", ""), row.get("status_p2", "")]
    if "Acima da capacidade" in statuses:
        return "Acima da capacidade"
    if "Alta utilização" in statuses:
        return "Alta utilização"
    if "Boa utilização" in statuses:
        return "Boa utilização"
    if "Utilização moderada" in statuses:
        return "Utilização moderada"
    if "Baixa utilização" in statuses:
        return "Baixa utilização"
    if "Sem CH letiva informada" in statuses:
        return "Sem CH letiva informada"
    return "Sem alocação"


def classifica_cor_status(status):
    return {
        "Acima da capacidade": COLORS["gap"],
        "Alta utilização": COLORS["alerta"],
        "Boa utilização": COLORS["ok"],
        "Utilização moderada": COLORS["horas"],
        "Baixa utilização": COLORS["neutro"],
        "Sem CH letiva informada": COLORS["neutro"],
        "Sem alocação": COLORS["neutro"],
    }.get(status, COLORS["neutro"])


def build_docente_label(nome, cargo, max_nome=42, max_cargo=38):
    nome = normalize_text(nome).title()[:max_nome]
    cargo = normalize_text(cargo).title()[:max_cargo]
    return f"{nome} — {cargo}" if cargo else nome


def montar_docentes_periodos(parametros_df: pd.DataFrame, realizado_df: pd.DataFrame, horas_planejadas_df: pd.DataFrame, horas_realizadas_df: Optional[pd.DataFrame] = None, parametros_todos_df: Optional[pd.DataFrame] = None):
    """Monta a visão docente por período operacional.

    Regra de negócio v18:
    - A capacidade base continua partindo dos docentes com STATUS = ATIVO na aba PARAMETROS.
    - Se um docente foi alocado no módulo no Registro, mas não está no banco ATIVO de PARAMETROS,
      ele também entra nos cálculos de capacidade/alocação, para que a visão reflita a operação real.
    - Esses casos continuam sendo exportados em uma tabela de governança para atualização posterior no RM/PARAMETROS.
    - Para docentes alocados fora do ativo, a CH é obtida preferencialmente da aba PARAMETROS completa;
      se não houver cadastro em PARAMETROS, usa a CH informada no Registro.
    """
    base_real = realizado_df.copy()
    base_real = base_real[base_real["professor_realizado_key"].ne("")].copy()
    # Regra: casar docentes por CHAPA (identificador estável) e usar o nome normalizado
    # apenas quando a CHAPA não está disponível na linha do Registro.
    base_real["professor_key"] = base_real["chave_docente"] if "chave_docente" in base_real.columns else base_real["professor_realizado_key"]

    rows_alloc = []
    for professor_key, g in base_real.groupby("professor_key", dropna=False):
        qtd_1 = int((g["ordem"] == "1ª").sum())
        qtd_2 = int((g["ordem"] == "2ª").sum())
        qtd_est = int((g["ordem"] == "ESTENDIDA").sum())
        rows_alloc.append({
            "professor_key": professor_key,
            "professor_realizado": safe_first(g["professor_realizado"]),
            "chapa_realizado": safe_first(g["chapa"]),
            "cargo_realizado": safe_first(g["cargo"]),
            "ch_semanal_realizado": pd.to_numeric(g["ch_semanal"], errors="coerce").dropna().max() if "ch_semanal" in g else np.nan,
            "ch_letiva_realizado": pd.to_numeric(g["ch_letiva"], errors="coerce").dropna().max() if "ch_letiva" in g else np.nan,
            "qtd_1": qtd_1,
            "qtd_2": qtd_2,
            "qtd_estendida": qtd_est,
            "disciplinas_distintas": int(g["disciplina_key"].nunique()),
            "horas_p1": int((qtd_1 + qtd_est) * 2),
            "horas_p2": int((qtd_2 + qtd_est) * 2),
            "valor_rpa_registro": float(g["valor_rpa_registro"].sum(skipna=True)) if "valor_rpa_registro" in g else 0,
            "rpa_total_linhas": int(g.get("is_rpa_registro", pd.Series(False, index=g.index)).sum()) if "is_rpa_registro" in g else 0,
        })
    aloc = pd.DataFrame(rows_alloc)

    params = parametros_df.copy()
    params = params.rename(columns={
        "professor_parametros": "professor",
        "chapa": "chapa_parametros",
        "cargo": "cargo_parametros",
        "ch_semanal": "ch_semanal_parametros",
        "ch_letiva": "ch_letiva_parametros",
    })
    if "chave_docente" in params.columns:
        params["professor_key"] = params["chave_docente"]

    params_keys = set(params["professor_key"].dropna().astype(str)) if "professor_key" in params.columns else set()
    params_todos = parametros_todos_df.copy() if parametros_todos_df is not None else parametros_df.copy()
    if "chave_docente" in params_todos.columns:
        params_todos["professor_key"] = params_todos["chave_docente"]
    params_todos_lookup_cols = [c for c in [
        "professor_key", "professor_parametros", "chapa", "cargo", "ch_semanal", "ch_letiva",
        "ch_letiva_alocavel", "capacidade_periodo", "status", "status_key", "docente_ativo", "equipe_rh", "fonte"
    ] if c in params_todos.columns]
    params_todos_lookup = params_todos[params_todos_lookup_cols].drop_duplicates("professor_key", keep="first") if "professor_key" in params_todos.columns else pd.DataFrame()

    # Tabela de governança: docentes alocados no módulo que não aparecem no banco ATIVO.
    if aloc.empty:
        alocados_fora_parametros_ativos = pd.DataFrame()
    else:
        alocados_fora_parametros_ativos = aloc[~aloc["professor_key"].isin(params_keys)].copy()
        if not alocados_fora_parametros_ativos.empty and not params_todos_lookup.empty:
            alocados_fora_parametros_ativos = alocados_fora_parametros_ativos.merge(
                params_todos_lookup.add_suffix("_parametros_todos"),
                left_on="professor_key",
                right_on="professor_key_parametros_todos",
                how="left"
            )
            alocados_fora_parametros_ativos["encontrado_parametros"] = alocados_fora_parametros_ativos["professor_key_parametros_todos"].notna()
            alocados_fora_parametros_ativos["status_parametros"] = alocados_fora_parametros_ativos.get("status_parametros_todos", pd.Series("", index=alocados_fora_parametros_ativos.index)).fillna("")
            alocados_fora_parametros_ativos["cargo_parametros"] = alocados_fora_parametros_ativos.get("cargo_parametros_todos", pd.Series("", index=alocados_fora_parametros_ativos.index)).fillna("")
            alocados_fora_parametros_ativos["ch_semanal_parametros"] = pd.to_numeric(alocados_fora_parametros_ativos.get("ch_semanal_parametros_todos", pd.Series(np.nan, index=alocados_fora_parametros_ativos.index)), errors="coerce")
            alocados_fora_parametros_ativos["ch_letiva_parametros"] = pd.to_numeric(alocados_fora_parametros_ativos.get("ch_letiva_parametros_todos", pd.Series(np.nan, index=alocados_fora_parametros_ativos.index)), errors="coerce")
            alocados_fora_parametros_ativos["motivo"] = np.where(
                alocados_fora_parametros_ativos["encontrado_parametros"],
                "Alocado no Registro e cadastrado na base docente atualizada, mas sem STATUS = ATIVO",
                "Alocado no Registro, mas não localizado na base docente atualizada"
            )
        elif not alocados_fora_parametros_ativos.empty:
            alocados_fora_parametros_ativos["encontrado_parametros"] = False
            alocados_fora_parametros_ativos["status_parametros"] = ""
            alocados_fora_parametros_ativos["cargo_parametros"] = ""
            alocados_fora_parametros_ativos["ch_semanal_parametros"] = np.nan
            alocados_fora_parametros_ativos["ch_letiva_parametros"] = np.nan
            alocados_fora_parametros_ativos["motivo"] = "Alocado no Registro, mas não localizado na base docente atualizada"

        if not alocados_fora_parametros_ativos.empty:
            alocados_fora_parametros_ativos["incluido_nos_calculos"] = True
            alocados_fora_parametros_ativos["regra_v20"] = "Alocado no módulo: entra nos cálculos; permanece na governança para ajuste cadastral/status no RM."

    # Banco inicial: docentes da base de capacidade.
    docentes = params.merge(aloc, on="professor_key", how="left")

    # Quando o mesmo docente aparece mais de uma vez na base docente atualizada, a capacidade deve ser preservada
    # em todas as linhas ativas, mas a alocação realizada no Registro não pode ser duplicada.
    alloc_numeric_cols = ["qtd_1", "qtd_2", "qtd_estendida", "disciplinas_distintas", "horas_p1", "horas_p2", "valor_rpa_registro", "rpa_total_linhas"]
    alloc_text_cols = ["professor_realizado", "chapa_realizado", "cargo_realizado"]
    alloc_ch_cols = ["ch_semanal_realizado", "ch_letiva_realizado"]
    non_primary_alloc = docentes.get("rank_professor_parametros", pd.Series(1, index=docentes.index)).fillna(1).gt(1)
    for c in alloc_numeric_cols:
        if c in docentes.columns:
            docentes.loc[non_primary_alloc, c] = 0
    for c in alloc_text_cols + alloc_ch_cols:
        if c in docentes.columns:
            docentes.loc[non_primary_alloc, c] = np.nan

    # Inclusão operacional: docentes alocados que não estão no banco ativo entram nos cálculos.
    if not alocados_fora_parametros_ativos.empty:
        extras = alocados_fora_parametros_ativos.copy()
        extras_doc = pd.DataFrame(index=extras.index)
        extras_doc["fonte"] = extras.get("fonte_parametros_todos", pd.Series(np.nan, index=extras.index)).fillna("Registro")
        extras_doc["professor_key"] = extras["professor_key"]
        extras_doc["professor"] = extras.get("professor_parametros_todos", pd.Series(np.nan, index=extras.index)).combine_first(extras["professor_realizado"])
        extras_doc["chapa_parametros"] = extras.get("chapa_parametros_todos", pd.Series(np.nan, index=extras.index))
        extras_doc["cargo_parametros"] = extras.get("cargo_parametros_todos", pd.Series(np.nan, index=extras.index))
        extras_doc["ch_semanal_parametros"] = pd.to_numeric(extras.get("ch_semanal_parametros_todos", pd.Series(np.nan, index=extras.index)), errors="coerce")
        extras_doc["ch_letiva_parametros"] = pd.to_numeric(extras.get("ch_letiva_parametros_todos", pd.Series(np.nan, index=extras.index)), errors="coerce")
        extras_doc["ch_nao_letiva"] = np.nan
        extras_doc["equipe_rh"] = extras.get("equipe_rh_parametros_todos", pd.Series("", index=extras.index))
        extras_doc["status"] = extras.get("status_parametros_todos", pd.Series("NÃO LOCALIZADO", index=extras.index)).fillna("NÃO LOCALIZADO")
        extras_doc["status_key"] = extras.get("status_key_parametros_todos", pd.Series("NAO LOCALIZADO", index=extras.index)).fillna("NAO LOCALIZADO")
        extras_doc["docente_ativo"] = extras.get("docente_ativo_parametros_todos", pd.Series(False, index=extras.index)).fillna(False).astype(bool)
        extras_doc["cargo_key"] = extras.get("cargo_parametros_todos", extras.get("cargo_realizado", pd.Series("", index=extras.index))).map(normalize_key)
        extras_doc["parametros_linha_id"] = np.nan
        extras_doc["rank_professor_parametros"] = 1
        extras_doc["professor_duplicado_parametros"] = False
        for c in alloc_numeric_cols + alloc_text_cols + alloc_ch_cols:
            if c in extras.columns:
                extras_doc[c] = extras[c]
            else:
                extras_doc[c] = 0 if c in alloc_numeric_cols else np.nan
        extras_doc["incluido_por_alocacao"] = True
        extras_doc["motivo_governanca"] = extras.get("motivo", pd.Series("Alocado fora do banco ativo", index=extras.index))
        docentes["incluido_por_alocacao"] = False
        docentes["motivo_governanca"] = ""
        docentes = pd.concat([docentes, extras_doc], ignore_index=True, sort=False)
    else:
        docentes["incluido_por_alocacao"] = False
        docentes["motivo_governanca"] = ""

    docentes["professor"] = docentes["professor"].combine_first(docentes.get("professor_realizado"))
    docentes["cargo"] = docentes["cargo_parametros"].combine_first(docentes.get("cargo_realizado"))
    docentes["chapa"] = docentes["chapa_parametros"].combine_first(docentes.get("chapa_realizado"))
    docentes["ch_semanal"] = docentes["ch_semanal_parametros"].combine_first(docentes.get("ch_semanal_realizado"))
    docentes["ch_letiva"] = docentes["ch_letiva_parametros"].combine_first(docentes.get("ch_letiva_realizado"))
    fonte_norm = docentes["fonte"].fillna("").map(normalize_key)
    fonte_base_oficial = fonte_norm.isin(["PARAMETROS", "BASE PROFESSORES ATUALIZADA"])
    docentes["origem_capacidade"] = np.select(
        [
            docentes.get("incluido_por_alocacao", pd.Series(False, index=docentes.index)).fillna(False) & fonte_base_oficial,
            docentes.get("incluido_por_alocacao", pd.Series(False, index=docentes.index)).fillna(False) & ~fonte_base_oficial,
            fonte_base_oficial,
        ],
        [
            "Base docente atualizada - incluído por alocação",
            "Registro - incluído por alocação",
            "Base docente atualizada - ATIVO",
        ],
        default="Registro"
    )
    docentes["alocado_no_registro"] = docentes["horas_p1"].fillna(0).gt(0) | docentes["horas_p2"].fillna(0).gt(0)
    docentes["incluido_base_capacidade"] = True

    for col in ["qtd_1", "qtd_2", "qtd_estendida", "disciplinas_distintas", "horas_p1", "horas_p2", "valor_rpa_registro", "rpa_total_linhas"]:
        if col not in docentes.columns:
            docentes[col] = 0
        docentes[col] = docentes[col].fillna(0)

    docentes["ch_semanal"] = pd.to_numeric(docentes["ch_semanal"], errors="coerce").fillna(0)
    docentes["ch_letiva"] = pd.to_numeric(docentes["ch_letiva"], errors="coerce").fillna(0)
    docentes["capacidade_periodo"] = docentes["ch_letiva"].map(capacidade_alocavel_periodo)
    docentes["ch_letiva_alocavel"] = docentes["capacidade_periodo"]
    docentes["capacidade_modulo_2_periodos"] = docentes["capacidade_periodo"] * 2
    docentes["horas_total_periodos"] = docentes["horas_p1"] + docentes["horas_p2"]
    docentes["saldo_p1"] = docentes["capacidade_periodo"] - docentes["horas_p1"]
    docentes["saldo_p2"] = docentes["capacidade_periodo"] - docentes["horas_p2"]
    docentes["delta_alocacao_p1"] = docentes["horas_p1"] - docentes["capacidade_periodo"]
    docentes["delta_alocacao_p2"] = docentes["horas_p2"] - docentes["capacidade_periodo"]
    docentes["utilizacao_p1_%"] = np.where(docentes["capacidade_periodo"].gt(0), docentes["horas_p1"] / docentes["capacidade_periodo"] * 100, np.nan)
    docentes["utilizacao_p2_%"] = np.where(docentes["capacidade_periodo"].gt(0), docentes["horas_p2"] / docentes["capacidade_periodo"] * 100, np.nan)
    docentes["utilizacao_max_%"] = docentes[["utilizacao_p1_%", "utilizacao_p2_%"]].max(axis=1)
    docentes["status_p1"] = docentes.apply(lambda r: status_periodo(r["utilizacao_p1_%"], r["horas_p1"], r["capacidade_periodo"]), axis=1)
    docentes["status_p2"] = docentes.apply(lambda r: status_periodo(r["utilizacao_p2_%"], r["horas_p2"], r["capacidade_periodo"]), axis=1)
    docentes["status_geral"] = docentes.apply(status_docente_geral, axis=1)
    docentes["cor_status"] = docentes["status_geral"].map(classifica_cor_status)
    docentes["docente_cargo"] = docentes.apply(lambda r: build_docente_label(r["professor"], r["cargo"]), axis=1)

    agg_cols = ["cargo", "ch_semanal", "ch_letiva"]
    geral = docentes.groupby(agg_cols, dropna=False).agg(
        docentes_total=("professor_key", "size"),
        professores_unicos=("professor_key", "nunique"),
        docentes_alocados=("alocado_no_registro", "sum"),
        incluidos_por_alocacao=("incluido_por_alocacao", "sum"),
        capacidade_periodo=("capacidade_periodo", "sum"),
        capacidade_modulo_2_periodos=("capacidade_modulo_2_periodos", "sum"),
        horas_p1=("horas_p1", "sum"),
        horas_p2=("horas_p2", "sum"),
        horas_total_periodos=("horas_total_periodos", "sum"),
        qtd_1=("qtd_1", "sum"),
        qtd_2=("qtd_2", "sum"),
        qtd_estendida=("qtd_estendida", "sum"),
        docentes_acima_p1=("status_p1", lambda s: int((s == "Acima da capacidade").sum())),
        docentes_acima_p2=("status_p2", lambda s: int((s == "Acima da capacidade").sum())),
        docentes_acima_geral=("status_geral", lambda s: int((s == "Acima da capacidade").sum())),
        valor_rpa_registro=("valor_rpa_registro", "sum"),
        rpa_total_linhas=("rpa_total_linhas", "sum"),
    ).reset_index()
    geral["docentes_sem_alocacao"] = geral["docentes_total"] - geral["docentes_alocados"]
    geral["delta_alocacao_p1"] = geral["horas_p1"] - geral["capacidade_periodo"]
    geral["delta_alocacao_p2"] = geral["horas_p2"] - geral["capacidade_periodo"]
    geral["saldo_p1"] = geral["capacidade_periodo"] - geral["horas_p1"]
    geral["saldo_p2"] = geral["capacidade_periodo"] - geral["horas_p2"]
    geral["utilizacao_p1_%"] = np.where(geral["capacidade_periodo"].gt(0), geral["horas_p1"] / geral["capacidade_periodo"] * 100, np.nan)
    geral["utilizacao_p2_%"] = np.where(geral["capacidade_periodo"].gt(0), geral["horas_p2"] / geral["capacidade_periodo"] * 100, np.nan)
    geral["utilizacao_max_%"] = geral[["utilizacao_p1_%", "utilizacao_p2_%"]].max(axis=1)

    por_ch = docentes.groupby(["ch_semanal", "ch_letiva"], dropna=False).agg(
        docentes_total=("professor_key", "size"),
        professores_unicos=("professor_key", "nunique"),
        docentes_alocados=("alocado_no_registro", "sum"),
        incluidos_por_alocacao=("incluido_por_alocacao", "sum"),
        capacidade_periodo=("capacidade_periodo", "sum"),
        capacidade_modulo_2_periodos=("capacidade_modulo_2_periodos", "sum"),
        horas_p1=("horas_p1", "sum"),
        horas_p2=("horas_p2", "sum"),
        horas_total_periodos=("horas_total_periodos", "sum"),
        docentes_acima_geral=("status_geral", lambda s: int((s == "Acima da capacidade").sum())),
    ).reset_index()
    por_ch["docentes_sem_alocacao"] = por_ch["docentes_total"] - por_ch["docentes_alocados"]
    por_ch["delta_alocacao_p1"] = por_ch["horas_p1"] - por_ch["capacidade_periodo"]
    por_ch["delta_alocacao_p2"] = por_ch["horas_p2"] - por_ch["capacidade_periodo"]
    por_ch["utilizacao_p1_%"] = np.where(por_ch["capacidade_periodo"].gt(0), por_ch["horas_p1"] / por_ch["capacidade_periodo"] * 100, np.nan)
    por_ch["utilizacao_p2_%"] = np.where(por_ch["capacidade_periodo"].gt(0), por_ch["horas_p2"] / por_ch["capacidade_periodo"] * 100, np.nan)
    por_ch["utilizacao_max_%"] = por_ch[["utilizacao_p1_%", "utilizacao_p2_%"]].max(axis=1)

    demanda_p1 = float(horas_planejadas_df.loc[horas_planejadas_df["periodo"].eq("Período 1"), "horas"].sum())
    demanda_p2 = float(horas_planejadas_df.loc[horas_planejadas_df["periodo"].eq("Período 2"), "horas"].sum())
    ch_letiva_bruta_total = float(docentes["ch_letiva"].fillna(0).sum())
    ch_letiva_alocavel_total = float(docentes["capacidade_periodo"].fillna(0).sum())

    horas_alocadas_base_capacidade_p1 = float(docentes["horas_p1"].fillna(0).sum())
    horas_alocadas_base_capacidade_p2 = float(docentes["horas_p2"].fillna(0).sum())

    if horas_realizadas_df is not None and not horas_realizadas_df.empty and "periodo" in horas_realizadas_df.columns and "horas" in horas_realizadas_df.columns:
        horas_alocadas_p1 = float(horas_realizadas_df.loc[horas_realizadas_df["periodo"].eq("Período 1"), "horas"].sum())
        horas_alocadas_p2 = float(horas_realizadas_df.loc[horas_realizadas_df["periodo"].eq("Período 2"), "horas"].sum())
    else:
        horas_alocadas_p1 = horas_alocadas_base_capacidade_p1
        horas_alocadas_p2 = horas_alocadas_base_capacidade_p2

    delta_geral = pd.DataFrame([
        {
            "periodo": "Período 1",
            "disciplinas_planejadas": "1ª + ESTENDIDA",
            "demanda_planejada_horas": demanda_p1,
            "ch_letiva_bruta_horas": ch_letiva_bruta_total,
            "ch_letiva_alocavel_horas": ch_letiva_alocavel_total,
            "delta_demanda_ch_alocavel": demanda_p1 - ch_letiva_alocavel_total,
            "horas_alocadas_registro": horas_alocadas_p1,
            "delta_alocado_ch_alocavel": horas_alocadas_p1 - ch_letiva_alocavel_total,
        },
        {
            "periodo": "Período 2",
            "disciplinas_planejadas": "2ª + ESTENDIDA",
            "demanda_planejada_horas": demanda_p2,
            "ch_letiva_bruta_horas": ch_letiva_bruta_total,
            "ch_letiva_alocavel_horas": ch_letiva_alocavel_total,
            "delta_demanda_ch_alocavel": demanda_p2 - ch_letiva_alocavel_total,
            "horas_alocadas_registro": horas_alocadas_p2,
            "delta_alocado_ch_alocavel": horas_alocadas_p2 - ch_letiva_alocavel_total,
        },
    ])
    delta_geral["composicao"] = delta_geral["disciplinas_planejadas"]
    delta_geral["capacidade_docente_horas"] = delta_geral["ch_letiva_bruta_horas"]
    delta_geral["capacidade_docente_alocavel_horas"] = delta_geral["ch_letiva_alocavel_horas"]
    delta_geral["delta_demanda_capacidade"] = delta_geral["delta_demanda_ch_alocavel"]
    delta_geral["delta_demanda_menos_capacidade"] = delta_geral["delta_demanda_ch_alocavel"]
    delta_geral["cobertura_capacidade_%"] = np.where(delta_geral["demanda_planejada_horas"].gt(0), delta_geral["ch_letiva_alocavel_horas"] / delta_geral["demanda_planejada_horas"] * 100, np.nan)
    delta_geral["utilizacao_realizada_%"] = np.where(delta_geral["ch_letiva_alocavel_horas"].gt(0), delta_geral["horas_alocadas_registro"] / delta_geral["ch_letiva_alocavel_horas"] * 100, np.nan)

    docentes = docentes.sort_values(["alocado_no_registro", "incluido_por_alocacao", "utilizacao_max_%", "horas_total_periodos"], ascending=[False, False, False, False])
    return base_real, docentes, geral, por_ch, delta_geral, alocados_fora_parametros_ativos

def barra_progressao_docente_periodos(docentes_df: pd.DataFrame, top_n: int = 30):
    if docentes_df.empty:
        insight_box("Docentes", "Não há docentes no banco de parâmetros.", COLORS["neutro"])
        return
    d = docentes_df.copy().sort_values(["alocado_no_registro", "utilizacao_max_%"], ascending=[False, False]).head(top_n)
    html = '<div style="display:flex; flex-direction:column; gap:12px; margin:14px 0;">'
    for _, r in d.iterrows():
        nome = normalize_text(r["professor"]).title()[:64]
        cargo = normalize_text(r.get("cargo", "")).title()
        cap = r["capacidade_periodo"] if pd.notna(r["capacidade_periodo"]) else 0
        p1 = min(r["utilizacao_p1_%"], 140) if pd.notna(r["utilizacao_p1_%"]) else 0
        p2 = min(r["utilizacao_p2_%"], 140) if pd.notna(r["utilizacao_p2_%"]) else 0
        color = r.get("cor_status", COLORS["neutro"])
        origem = r.get("origem_capacidade", "")
        html += f'''
        <div style="background:white; border-radius:14px; padding:12px 14px; box-shadow:0 1px 8px rgba(15,23,42,.08); border-left:6px solid {color};">
            <div style="display:flex; justify-content:space-between; gap:8px; font-weight:800; color:#0F172A;">
                <span>{nome}</span><span>{r['status_geral']}</span>
            </div>
            <div style="font-size:13px; color:#334155; margin:4px 0 4px 0; font-weight:700;">{cargo}</div>
            <div style="font-size:13px; color:#475569; margin:4px 0 8px 0;">Status: {normalize_text(r.get('status','')) or 'não informado'} • Origem capacidade: {origem} • CH semanal {br_int(r['ch_semanal'])}h • CH letiva {br_int(r['ch_letiva'])}h • capacidade por período {br_int(cap)}h</div>
            <div style="font-size:13px; color:#334155; margin-bottom:3px;">P1: {br_int(r['horas_p1'])}h / {br_int(cap)}h • saldo {br_int(r['saldo_p1'])}h • {br_pct(r['utilizacao_p1_%'])}</div>
            <div style="background:#E2E8F0; height:10px; border-radius:999px; overflow:hidden; margin-bottom:7px;"><div style="width:{min(p1,100)}%; background:{ORDER_COLORS['1ª']}; height:10px;"></div></div>
            <div style="font-size:13px; color:#334155; margin-bottom:3px;">P2: {br_int(r['horas_p2'])}h / {br_int(cap)}h • saldo {br_int(r['saldo_p2'])}h • {br_pct(r['utilizacao_p2_%'])}</div>
            <div style="background:#E2E8F0; height:10px; border-radius:999px; overflow:hidden;"><div style="width:{min(p2,100)}%; background:{ORDER_COLORS['2ª']}; height:10px;"></div></div>
        </div>
        '''
    html += '</div>'
    display(HTML(html))


def fig_delta_demanda_capacidade(delta_df: pd.DataFrame):
    plot = delta_df.melt(
        id_vars=["periodo", "disciplinas_planejadas"],
        value_vars=["demanda_planejada_horas", "ch_letiva_alocavel_horas", "horas_alocadas_registro"],
        var_name="medida",
        value_name="horas",
    )
    plot["medida"] = plot["medida"].map({
        "demanda_planejada_horas": "Demanda planejada",
        "ch_letiva_alocavel_horas": "CH letiva alocável",
        "horas_alocadas_registro": "Alocado no Registro (Ato 2)",
    })
    fig = px.bar(
        plot,
        x="periodo",
        y="horas",
        color="medida",
        barmode="group",
        text="horas",
        color_discrete_map={"Demanda planejada": COLORS["planejado"], "CH letiva alocável": COLORS["horas"], "Alocado no Registro (Ato 2)": COLORS["realizado"]},
        labels={"periodo": "Período", "horas": "Horas", "medida": "Medida", "disciplinas_planejadas": "Disciplinas planejadas"},
        hover_data={"disciplinas_planejadas": True},
    )
    fig.update_traces(texttemplate="%{text:.0f}h", textposition="outside", textfont_size=14)
    return aplicar_layout(fig, "Visão docente — demanda planejada × CH letiva alocável × realizado (Ato 2)", height=540)


def fig_docentes_periodos_lado_a_lado(docentes_df: pd.DataFrame, top_n: int = 35):
    d = docentes_df.copy().sort_values(["alocado_no_registro", "utilizacao_max_%"], ascending=[False, False]).head(top_n)
    d["docente"] = d.apply(lambda r: build_docente_label(r["professor"], r["cargo"]), axis=1)
    m = d.melt(id_vars=["docente", "cargo", "capacidade_periodo", "status_geral"], value_vars=["horas_p1", "horas_p2"], var_name="periodo", value_name="horas")
    m["periodo"] = m["periodo"].map({"horas_p1": "P1: 1ª + ESTENDIDA", "horas_p2": "P2: 2ª + ESTENDIDA"})
    fig = px.bar(m, y="docente", x="horas", color="periodo", orientation="h", barmode="group", text="horas", color_discrete_map={"P1: 1ª + ESTENDIDA": ORDER_COLORS["1ª"], "P2: 2ª + ESTENDIDA": ORDER_COLORS["2ª"]}, labels={"docente": "Docente e cargo", "horas": "Horas equivalentes", "periodo": "Período"}, hover_data={"cargo": True, "capacidade_periodo": True, "status_geral": True})
    fig.add_vline(x=float(d["capacidade_periodo"].dropna().median()) if d["capacidade_periodo"].notna().any() else 0, line_dash="dash", line_color=COLORS["neutro"], annotation_text="Mediana capacidade", annotation_position="top right")
    fig.update_traces(textposition="outside")
    return aplicar_layout(fig, "Docentes — horas alocadas por período operacional", height=max(760, 30 * len(d)))


def fig_utilizacao_docente(docentes_df: pd.DataFrame, top_n: int = 35):
    d = docentes_df.copy().sort_values(["alocado_no_registro", "utilizacao_max_%"], ascending=[False, False]).head(top_n)
    d["docente"] = d.apply(lambda r: build_docente_label(r["professor"], r["cargo"]), axis=1)
    m = d.melt(id_vars=["docente", "cargo", "status_geral"], value_vars=["utilizacao_p1_%", "utilizacao_p2_%"], var_name="periodo", value_name="utilizacao")
    m["periodo"] = m["periodo"].map({"utilizacao_p1_%": "P1: 1ª + ESTENDIDA", "utilizacao_p2_%": "P2: 2ª + ESTENDIDA"})
    fig = px.bar(m, y="docente", x="utilizacao", color="periodo", orientation="h", barmode="group", text="utilizacao", color_discrete_map={"P1: 1ª + ESTENDIDA": ORDER_COLORS["1ª"], "P2: 2ª + ESTENDIDA": ORDER_COLORS["2ª"]}, labels={"docente": "Docente e cargo", "utilizacao": "Utilização da CH letiva (%)", "periodo": "Período"}, hover_data={"cargo": True, "status_geral": True})
    fig.add_vline(x=100, line_dash="dash", line_color=COLORS["gap"], annotation_text="Limite 100%", annotation_position="top right")
    fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside")
    return aplicar_layout(fig, "Docentes — utilização da capacidade por período", height=max(760, 30 * len(d)))


def fig_geral_cargo_ch_periodos(geral_df: pd.DataFrame):
    d = geral_df.copy()
    d["grupo"] = d["cargo"].replace("", "Não informado") + " | " + d["ch_semanal"].fillna(0).astype(int).astype(str) + "h sem. | " + d["ch_letiva"].fillna(0).astype(int).astype(str) + "h let. | " + d["capacidade_periodo"].fillna(0).astype(int).astype(str) + "h aloc."
    d = d.sort_values("capacidade_periodo", ascending=True)
    plot = d.melt(id_vars=["grupo", "docentes_total", "docentes_alocados", "capacidade_periodo"], value_vars=["capacidade_periodo", "horas_p1", "horas_p2"], var_name="medida", value_name="horas")
    plot["medida"] = plot["medida"].map({"capacidade_periodo": "CH letiva alocável", "horas_p1": "Alocado P1", "horas_p2": "Alocado P2"})
    fig = px.bar(plot, x="horas", y="grupo", color="medida", orientation="h", barmode="group", text="horas", color_discrete_map={"CH letiva alocável": COLORS["horas"], "Alocado P1": ORDER_COLORS["1ª"], "Alocado P2": ORDER_COLORS["2ª"]}, labels={"horas": "Horas", "grupo": "Cargo + CH", "medida": "Medida"}, hover_data={"docentes_total": True, "docentes_alocados": True})
    fig.update_traces(textposition="outside")
    return aplicar_layout(fig, "Visão geral — CH letiva alocável e alocação por cargo/CH", height=max(600, 58 * len(d)))


def fig_delta_cargo_ch(geral_df: pd.DataFrame):
    d = geral_df.copy()
    d["grupo"] = d["cargo"].replace("", "Não informado") + " | " + d["ch_semanal"].fillna(0).astype(int).astype(str) + "h sem. | " + d["ch_letiva"].fillna(0).astype(int).astype(str) + "h let. | " + d["capacidade_periodo"].fillna(0).astype(int).astype(str) + "h aloc."
    plot = d.melt(id_vars=["grupo", "docentes_total", "docentes_alocados"], value_vars=["delta_alocacao_p1", "delta_alocacao_p2"], var_name="periodo", value_name="delta")
    plot["periodo"] = plot["periodo"].map({"delta_alocacao_p1": "P1: alocado - alocável", "delta_alocacao_p2": "P2: alocado - alocável"})
    fig = px.bar(plot.sort_values("delta"), x="delta", y="grupo", color="periodo", orientation="h", barmode="group", text="delta", color_discrete_map={"P1: alocado - alocável": ORDER_COLORS["1ª"], "P2: alocado - alocável": ORDER_COLORS["2ª"]}, labels={"delta": "Delta de alocação (h)", "grupo": "Cargo + CH", "periodo": "Período"}, hover_data={"docentes_total": True, "docentes_alocados": True})
    fig.add_vline(x=0, line_color=COLORS["neutro"], line_dash="dash")
    fig.update_traces(textposition="outside")
    return aplicar_layout(fig, "Delta por cargo e CH — alocado no Registro menos CH letiva alocável", height=max(600, 58 * len(d)))


def fig_geral_ch_utilizacao(por_ch_df: pd.DataFrame):
    d = por_ch_df.copy()
    d["grupo_ch"] = d["ch_semanal"].fillna(0).astype(int).astype(str) + "h semanal / " + d["ch_letiva"].fillna(0).astype(int).astype(str) + "h letiva / " + d["capacidade_periodo"].fillna(0).astype(int).astype(str) + "h alocável"
    plot = d.melt(id_vars=["grupo_ch", "docentes_total", "docentes_alocados"], value_vars=["utilizacao_p1_%", "utilizacao_p2_%"], var_name="periodo", value_name="utilizacao")
    plot["periodo"] = plot["periodo"].map({"utilizacao_p1_%": "P1: 1ª + ESTENDIDA", "utilizacao_p2_%": "P2: 2ª + ESTENDIDA"})
    fig = px.bar(plot, x="grupo_ch", y="utilizacao", color="periodo", barmode="group", text="utilizacao", color_discrete_map={"P1: 1ª + ESTENDIDA": ORDER_COLORS["1ª"], "P2: 2ª + ESTENDIDA": ORDER_COLORS["2ª"]}, labels={"grupo_ch": "Grupo CH", "utilizacao": "Utilização (%)", "periodo": "Período"}, hover_data={"docentes_total": True, "docentes_alocados": True})
    fig.add_hline(y=100, line_dash="dash", line_color=COLORS["gap"], annotation_text="Limite 100%", annotation_position="top right")
    fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside")
    fig.update_xaxes(tickangle=-20)
    return aplicar_layout(fig, "Utilização geral por grupo de CH", height=560)


base_docente, docente_individual, docente_geral, docente_por_ch, docente_delta_geral, docente_fora_parametros_ativos = montar_docentes_periodos(parametros_docentes, realizado, horas_planejado, horas_realizado, parametros_docentes_todos)

total_docentes_banco = int(len(docente_individual)) if not docente_individual.empty else 0
# v53 — "Docentes considerados" é contagem de PESSOAS e cobre apenas os cargos
# docentes definidos em CARGOS_DOCENTES_CONSIDERADOS (professor de ensino superior
# EAD, professor de ensino superior presencial e professor regente). RPA não entra:
# cada linha de RPA é uma contratação, não uma pessoa do quadro, e sua leitura
# acontece na seção RPA. Sem esse recorte, a métrica do topo do relatório não
# fechava com o quadro de docentes por cargo.
total_docentes_banco_unicos_todos_cargos = int(docente_individual["professor_key"].nunique()) if not docente_individual.empty else 0
mask_docente_considerado_v53 = mascara_cargos_considerados(docente_individual["cargo"]) if not docente_individual.empty and "cargo" in docente_individual.columns else pd.Series(dtype=bool)
if not docente_individual.empty and len(mask_docente_considerado_v53) and bool(mask_docente_considerado_v53.any()):
    docente_considerado_v53 = docente_individual.loc[mask_docente_considerado_v53].copy()
    total_docentes_banco_unicos = int(docente_considerado_v53["professor_key"].nunique())
else:
    docente_considerado_v53 = docente_individual.copy()
    total_docentes_banco_unicos = total_docentes_banco_unicos_todos_cargos
    if not docente_individual.empty:
        print("Atenção: nenhum cargo da base bateu com CARGOS_DOCENTES_CONSIDERADOS. A métrica usou todos os cargos — revise a lista na configuração.")

# Recorte de ativos dentro do mesmo escopo de cargos, para que a leitura
# "considerados = ativos + alocados fora da base ativa" feche na validação.
if not docente_considerado_v53.empty and "docente_ativo" in docente_considerado_v53.columns:
    _mask_ativo_v53 = docente_considerado_v53["docente_ativo"].fillna(False).astype(bool)
    qtd_docentes_ativos_considerados_v53 = int(docente_considerado_v53.loc[_mask_ativo_v53, "professor_key"].nunique())
    docentes_considerados_fora_base_ativa_v53 = int(docente_considerado_v53.loc[~_mask_ativo_v53, "professor_key"].nunique())
else:
    qtd_docentes_ativos_considerados_v53 = total_docentes_banco_unicos
    docentes_considerados_fora_base_ativa_v53 = 0

# Diagnóstico da composição: mostra cargo a cargo quem entrou e quem ficou fora.
if not docente_individual.empty and "cargo" in docente_individual.columns:
    composicao_docentes_v53 = (
        docente_individual.assign(_entra=mascara_cargos_considerados(docente_individual["cargo"]))
        .groupby(["cargo", "_entra"], dropna=False)["professor_key"].nunique()
        .reset_index(name="pessoas")
        .rename(columns={"_entra": "entra_na_metrica"})
        .sort_values(["entra_na_metrica", "pessoas"], ascending=[False, False])
    )
    print("Composição de 'docentes considerados' (pessoas únicas por cargo):")
    print(composicao_docentes_v53.to_string(index=False))
    print(f"Total considerado: {total_docentes_banco_unicos} | Total em todos os cargos da base: {total_docentes_banco_unicos_todos_cargos}")
else:
    composicao_docentes_v53 = pd.DataFrame(columns=["cargo", "entra_na_metrica", "pessoas"])
total_docentes_alocados = int(docente_individual["alocado_no_registro"].sum()) if not docente_individual.empty else 0
total_docentes_sem_alocacao = total_docentes_banco - total_docentes_alocados
total_ch_letiva_ativos_docente = float(docente_individual["ch_letiva"].fillna(0).sum()) if not docente_individual.empty else 0.0
total_ch_letiva_alocavel_docente = float(docente_individual["ch_letiva_alocavel"].fillna(0).sum()) if not docente_individual.empty else 0.0
total_docentes = total_docentes_banco
capacidade_periodo_total = float(docente_individual["capacidade_periodo"].fillna(0).sum()) if not docente_individual.empty else 0.0
capacidade_modulo_total = float(docente_individual["capacidade_modulo_2_periodos"].fillna(0).sum()) if not docente_individual.empty else 0.0
horas_p1_doc_total = float(docente_individual["horas_p1"].fillna(0).sum()) if not docente_individual.empty else 0.0
horas_p2_doc_total = float(docente_individual["horas_p2"].fillna(0).sum()) if not docente_individual.empty else 0.0
horas_alocadas_total = horas_p1_doc_total + horas_p2_doc_total
utilizacao_p1_total = horas_p1_doc_total / capacidade_periodo_total * 100 if capacidade_periodo_total else np.nan
utilizacao_p2_total = horas_p2_doc_total / capacidade_periodo_total * 100 if capacidade_periodo_total else np.nan
utilizacao_docente_total = horas_alocadas_total / capacidade_modulo_total * 100 if capacidade_modulo_total else np.nan
docentes_acima_p1 = int(docente_individual["status_p1"].eq("Acima da capacidade").sum()) if not docente_individual.empty else 0
docentes_acima_p2 = int(docente_individual["status_p2"].eq("Acima da capacidade").sum()) if not docente_individual.empty else 0
docentes_acima = int(docente_individual["status_geral"].eq("Acima da capacidade").sum()) if not docente_individual.empty else 0
docentes_fora_parametros_ativos_total = int(docente_fora_parametros_ativos["professor_key"].nunique()) if isinstance(docente_fora_parametros_ativos, pd.DataFrame) and not docente_fora_parametros_ativos.empty else 0

demanda_p1_planejada = float(docente_delta_geral.loc[docente_delta_geral["periodo"].eq("Período 1"), "demanda_planejada_horas"].sum()) if not docente_delta_geral.empty else 0.0
demanda_p2_planejada = float(docente_delta_geral.loc[docente_delta_geral["periodo"].eq("Período 2"), "demanda_planejada_horas"].sum()) if not docente_delta_geral.empty else 0.0
delta_demanda_p1 = float(docente_delta_geral.loc[docente_delta_geral["periodo"].eq("Período 1"), "delta_demanda_capacidade"].sum()) if not docente_delta_geral.empty else 0.0
delta_demanda_p2 = float(docente_delta_geral.loc[docente_delta_geral["periodo"].eq("Período 2"), "delta_demanda_capacidade"].sum()) if not docente_delta_geral.empty else 0.0

kpi_cards([
    {"label": "Docentes na base de capacidade", "value": br_int(total_docentes_banco), "note": f"{br_int(total_docentes_banco_unicos)} professores únicos | {br_int(total_docentes_alocados)} alocados", "color": COLORS["horas"]},
    {"label": "CH letiva bruta / alocável", "value": f"{br_int(total_ch_letiva_ativos_docente)}h / {br_int(capacidade_periodo_total)}h", "note": "Ativos + alocados no módulo por regra | bruta / alocável", "color": COLORS["horas"]},
    {"label": "Delta P1", "value": f"{br_int(delta_demanda_p1)}h", "note": "Demanda planejada - CH letiva alocável", "color": COLORS["gap"] if delta_demanda_p1 > 0 else COLORS["ok"]},
    {"label": "Delta P2", "value": f"{br_int(delta_demanda_p2)}h", "note": "Demanda planejada - CH letiva alocável", "color": COLORS["gap"] if delta_demanda_p2 > 0 else COLORS["ok"]},
    {"label": "Acima da capacidade", "value": br_int(docentes_acima), "note": f"P1: {br_int(docentes_acima_p1)} | P2: {br_int(docentes_acima_p2)}", "color": COLORS["gap"] if docentes_acima else COLORS["ok"]},
])

insight_box(
    "Como ler o delta docente",
    "O delta principal compara a demanda planejada de horas das disciplinas com a CH letiva alocável dos docentes. Para manter a leitura didática, esta visão usa somente os mesmos três blocos do gráfico: Demanda planejada, CH letiva alocável e Alocado no Registro. O Alocado no Registro considera todas as linhas válidas registradas; duplicidades são exibidas em governança, mas não são excluídas dos cálculos. P1 usa 1ª + ESTENDIDA e P2 usa 2ª + ESTENDIDA. Valores positivos indicam demanda maior que a capacidade disponível; valores negativos indicam folga de capacidade.",
    COLORS["horas"]
)


# Guard rail final: garante que qualquer gráfico/tabela que fale em CH letiva alocável
# use a coluna alocável e não a CH letiva bruta.
def validar_consistencia_delta_docente(delta_df: pd.DataFrame) -> None:
    if delta_df is None or delta_df.empty:
        return
    obrigatorias = ["ch_letiva_bruta_horas", "ch_letiva_alocavel_horas", "capacidade_docente_horas", "capacidade_docente_alocavel_horas"]
    faltantes = [c for c in obrigatorias if c not in delta_df.columns]
    if faltantes:
        print(f"Aviso: colunas ausentes no delta docente: {faltantes}")
        return
    # A coluna legada capacidade_docente_horas é mantida apenas para compatibilidade e representa CH letiva bruta.
    # Para cálculos e gráficos de capacidade operacional, usar SEMPRE ch_letiva_alocavel_horas.
    bruto = float(delta_df["ch_letiva_bruta_horas"].iloc[0])
    aloc = float(delta_df["ch_letiva_alocavel_horas"].iloc[0])
    if bruto == aloc:
        print(f"Validação delta docente: CH bruta e CH alocável estão iguais ({bruto:.0f}h).")
    else:
        print(f"Validação delta docente: CH bruta = {bruto:.0f}h | CH alocável = {aloc:.0f}h. Gráficos usam CH alocável.")

validar_consistencia_delta_docente(docente_delta_geral)

display(Markdown("### Delta geral — demanda planejada das disciplinas × CH letiva alocável"))
cols_delta_exibir = [
    "periodo",
    "disciplinas_planejadas",
    "demanda_planejada_horas",
    "ch_letiva_bruta_horas",
    "ch_letiva_alocavel_horas",
    "delta_demanda_ch_alocavel",
    "horas_alocadas_registro",
    "delta_alocado_ch_alocavel",
    "cobertura_capacidade_%",
    "utilizacao_realizada_%",
]
style_delta = (
    docente_delta_geral[cols_delta_exibir]
    .rename(columns={
        "periodo": "Período",
        "disciplinas_planejadas": "Disciplinas planejadas",
        "demanda_planejada_horas": "Demanda planejada",
        "ch_letiva_bruta_horas": "CH letiva bruta",
        "ch_letiva_alocavel_horas": "CH letiva alocável",
        "delta_demanda_ch_alocavel": "Delta demanda - CH alocável",
        "horas_alocadas_registro": "Alocado no Registro (Ato 2)",
            "delta_alocado_ch_alocavel": "Delta realizado - CH alocável",
            "cobertura_capacidade_%": "Cobertura da capacidade",
        "utilizacao_realizada_%": "Utilização realizada",
        })
    .style.format({
        "Demanda planejada": "{:.0f}h",
        "CH letiva bruta": "{:.0f}h",
        "CH letiva alocável": "{:.0f}h",
        "Delta demanda - CH alocável": "{:.0f}h",
        "Alocado no Registro (Ato 2)": "{:.0f}h",
        "Delta realizado - CH alocável": "{:.0f}h",
        "Cobertura da capacidade": "{:.1f}%",
        "Utilização realizada": "{:.1f}%",
    })
    .set_properties(**{"text-align": "left"})
)
display(style_delta)
show_fig(fig_delta_demanda_capacidade(docente_delta_geral))

barra_progressao_docente_periodos(docente_individual, top_n=35)
show_fig(fig_docentes_periodos_lado_a_lado(docente_individual, top_n=40))
show_fig(fig_utilizacao_docente(docente_individual, top_n=40))
show_fig(fig_geral_cargo_ch_periodos(docente_geral))
show_fig(fig_delta_cargo_ch(docente_geral))
show_fig(fig_geral_ch_utilizacao(docente_por_ch))

sem_alocacao = docente_individual[docente_individual["alocado_no_registro"].eq(False)].copy()
if not sem_alocacao.empty:
    insight_box(
        "Docentes ativos sem alocação no módulo",
        f"A base docente atualizada possui {br_int(len(sem_alocacao))} linhas docentes com STATUS = ATIVO sem disciplina registrada no módulo {modulo_alvo}. Esses docentes entram no banco de capacidade e no delta geral, mas não apareceriam caso a análise usasse apenas PREENCHIMENTO|LETIVAS.",
        COLORS["neutro"]
    )
    display(Markdown("### Docentes ATIVOS na base docente atualizada sem alocação no Registro"))
    display(sem_alocacao[["professor", "cargo", "status", "ch_semanal", "ch_letiva", "ch_letiva_alocavel", "capacidade_periodo", "origem_capacidade"]].sort_values(["cargo", "professor"]).head(150))

display(Markdown("### Tabela geral — docentes ATIVOS da base docente atualizada (banco oficial de capacidade)"))
insight_box(
    "Leitura correta desta tabela",
    "A tabela abaixo é o banco ATIVO de capacidade docente usado no delta. Se um professor aparece com STATUS = ATIVO na base docente atualizada, ele está corretamente incluído na capacidade. Esta tabela não representa inconsistência; é a visão completa dos docentes ativos, alocados ou não no módulo.",
    COLORS["ok"]
)
cols_docente_display = [
    "professor", "cargo", "status", "origem_capacidade", "alocado_no_registro", "ch_semanal", "ch_letiva", "ch_letiva_alocavel", "capacidade_periodo",
    "qtd_1", "qtd_2", "qtd_estendida", "horas_p1", "horas_p2", "saldo_p1", "saldo_p2", "delta_alocacao_p1", "delta_alocacao_p2",
    "utilizacao_p1_%", "utilizacao_p2_%", "status_p1", "status_p2", "status_geral"
]
display(docente_individual.sort_values(["alocado_no_registro", "utilizacao_max_%"], ascending=[False, False])[cols_docente_display].head(220) if not docente_individual.empty else docente_individual)

if isinstance(docente_fora_parametros_ativos, pd.DataFrame) and not docente_fora_parametros_ativos.empty:
    qtd_em_parametros_nao_ativo = int(docente_fora_parametros_ativos.get("encontrado_parametros", pd.Series(False, index=docente_fora_parametros_ativos.index)).fillna(False).sum())
    qtd_sem_cadastro_parametros = int(len(docente_fora_parametros_ativos) - qtd_em_parametros_nao_ativo)
    insight_box(
        "Governança — alocados fora do STATUS ATIVO na base docente atualizada",
        f"Foram encontrados {br_int(docentes_fora_parametros_ativos_total)} docentes alocados no Registro sem STATUS = ATIVO na base docente atualizada. Desses, {br_int(qtd_em_parametros_nao_ativo)} constam na base docente atualizada com outro status e {br_int(qtd_sem_cadastro_parametros)} não foram localizados na base docente atualizada. Eles são incluídos nos cálculos por estarem alocados no módulo e permanecem listados para governança/RM.",
        COLORS["alerta"]
    )
    display(Markdown("### Docentes alocados no Registro sem STATUS = ATIVO na base docente atualizada"))
    cols_fora = [c for c in [
        "professor_realizado", "cargo_realizado", "status_parametros", "cargo_parametros", "ch_semanal_parametros", "ch_letiva_parametros",
        "ch_semanal_realizado", "ch_letiva_realizado", "horas_p1", "horas_p2", "incluido_nos_calculos", "motivo", "regra_v20"
    ] if c in docente_fora_parametros_ativos.columns]
    display(docente_fora_parametros_ativos[cols_fora].sort_values(["motivo", "cargo_realizado", "professor_realizado"]).head(150))
else:
    insight_box(
        "Governança docente validada",
        "Todas as alocações do Registro foram vinculadas a docentes com STATUS = ATIVO na base docente atualizada. Não há divergências cadastrais para levar ao RM.",
        COLORS["ok"]
    )

display(Markdown("### Tabela — visão geral por cargo, CH semanal e CH letiva"))
display(docente_geral.sort_values(["cargo", "ch_semanal", "ch_letiva"]) if not docente_geral.empty else docente_geral)

# ================================================================
# DOCENTES PRESENCIAL (CASO À PARTE) — não entra em nenhum total geral
# ================================================================
# Professores de cargo "Professor de Ensino Superior Presencial" não fazem parte da base
# de capacidade CLT EAD/Regente do SGA; quando aparecem alocados no Registro, entram em
# docente_individual com ch_semanal/ch_letiva/capacidade_periodo = 0 e status "NÃO LOCALIZADO"
# (governança). Isso é proposital e continua exatamente assim — nada aqui muda esse
# comportamento nem qualquer total geral (delta docente, capacidade, KPIs). Esta seção é
# só uma leitura complementar, cruzando a CH contratada da nova base de docentes
# presenciais, para os que foram efetivamente alocados neste módulo.
docentes_presencial_alocados = pd.DataFrame()
docentes_presencial_por_tipo = pd.DataFrame()
if not docente_individual.empty:
    mask_presencial_alocado = (
        docente_individual["cargo"].astype(str).str.upper().str.contains("PRESENCIAL", na=False)
        & docente_individual["alocado_no_registro"].fillna(False)
    )
    docentes_presencial_alocados = docente_individual.loc[mask_presencial_alocado].copy()

if not docentes_presencial_alocados.empty:
    if not docentes_presencial_base.empty:
        # Casamento por CHAPA direto (não por chave_docente): a coluna chave_docente do
        # docente_individual fica vazia para docentes "não localizados"/governança — a
        # chapa em si, porém, vem corretamente preenchida a partir do Registro.
        docentes_presencial_alocados["_chapa_match"] = docentes_presencial_alocados["chapa"].map(normalize_id)
        base_presencial_lookup = docentes_presencial_base.copy()
        base_presencial_lookup["_chapa_match"] = base_presencial_lookup["chapa"].map(normalize_id)
        docentes_presencial_alocados = docentes_presencial_alocados.merge(
            base_presencial_lookup[["_chapa_match", "tipo_contrato", "ch_contratada_semanal", "ch_contratada_mensal", "situacao_rh"]],
            on="_chapa_match", how="left",
        ).drop(columns=["_chapa_match"])
    else:
        docentes_presencial_alocados["tipo_contrato"] = np.nan
        docentes_presencial_alocados["ch_contratada_semanal"] = np.nan
        docentes_presencial_alocados["ch_contratada_mensal"] = np.nan
        docentes_presencial_alocados["situacao_rh"] = np.nan

    docentes_presencial_alocados["horas_alocadas_total"] = docentes_presencial_alocados["horas_p1"].fillna(0) + docentes_presencial_alocados["horas_p2"].fillna(0)
    docentes_presencial_alocados["localizado_na_base_presencial"] = docentes_presencial_alocados["ch_contratada_semanal"].notna()
    docentes_presencial_alocados["tipo_contrato"] = docentes_presencial_alocados["tipo_contrato"].fillna("NÃO LOCALIZADO NA BASE PRESENCIAL")

    docentes_presencial_por_tipo = docentes_presencial_alocados.groupby("tipo_contrato", dropna=False).agg(
        docentes=("professor_key", "nunique"),
        ch_contratada_semanal_total=("ch_contratada_semanal", "sum"),
        horas_p1_total=("horas_p1", "sum"),
        horas_p2_total=("horas_p2", "sum"),
        horas_alocadas_total=("horas_alocadas_total", "sum"),
    ).reset_index().sort_values("docentes", ascending=False)
    total_row_presencial = pd.DataFrame([{
        "tipo_contrato": "TOTAL GERAL",
        "docentes": docentes_presencial_por_tipo["docentes"].sum(),
        "ch_contratada_semanal_total": docentes_presencial_por_tipo["ch_contratada_semanal_total"].sum(),
        "horas_p1_total": docentes_presencial_por_tipo["horas_p1_total"].sum(),
        "horas_p2_total": docentes_presencial_por_tipo["horas_p2_total"].sum(),
        "horas_alocadas_total": docentes_presencial_por_tipo["horas_alocadas_total"].sum(),
    }])
    docentes_presencial_por_tipo = pd.concat([docentes_presencial_por_tipo, total_row_presencial], ignore_index=True)

    qtd_docentes_presencial_alocados = int(docentes_presencial_alocados["professor_key"].nunique())
    insight_box(
        "Docentes Presencial — caso à parte (não entra nos totais gerais)",
        f"{br_int(qtd_docentes_presencial_alocados)} docentes de cargo Professor de Ensino Superior Presencial foram alocados no "
        f"Registro deste módulo, sem constar na base de capacidade CLT EAD/Regente. Cruzando com a base de docentes presenciais, "
        f"a CH contratada (semanal) somada é {br_int(docentes_presencial_por_tipo.loc[docentes_presencial_por_tipo['tipo_contrato'].eq('TOTAL GERAL'), 'ch_contratada_semanal_total'].iloc[0])}h "
        f"e as horas efetivamente alocadas no Registro somam {br_int(docentes_presencial_por_tipo.loc[docentes_presencial_por_tipo['tipo_contrato'].eq('TOTAL GERAL'), 'horas_alocadas_total'].iloc[0])}h. "
        f"Ainda não existe regra validada de CH LETIVA para presencial, por isso essa coluna não aparece aqui. "
        f"Nenhum valor desta seção entra no delta docente, na capacidade geral ou em qualquer KPI do relatório — é só leitura complementar.",
        COLORS["neutro"]
    )
    display(Markdown("### Docentes Presencial — resumo por tipo de contrato"))
    display(docentes_presencial_por_tipo)
    display(Markdown("### Docentes Presencial — detalhe por docente"))
    cols_presencial_detalhe = [c for c in [
        "professor", "chapa", "tipo_contrato", "situacao_rh", "ch_contratada_semanal", "ch_contratada_mensal",
        "horas_p1", "horas_p2", "horas_alocadas_total", "localizado_na_base_presencial",
    ] if c in docentes_presencial_alocados.columns]
    display(docentes_presencial_alocados[cols_presencial_detalhe].sort_values("horas_alocadas_total", ascending=False))
elif not docente_individual.empty:
    print("Nenhum docente de cargo Professor de Ensino Superior Presencial alocado no Registro deste módulo.")

# ================================================================
# ATO 5 — RESULTADOS
# ================================================================
section("Ato 5 — Resultados", "Síntese executiva com diagnóstico, riscos e pontos de decisão.")

valor_mapa = float(rpa_comp.loc[rpa_comp["origem"].eq("Mapa"), "valor_rpa"].sum())
valor_reg = float(rpa_comp.loc[rpa_comp["origem"].eq("Registro"), "valor_rpa"].sum())
delta_rpa = valor_reg - valor_mapa
qtd_rpa_mapa = int(rpa_comp.loc[rpa_comp["origem"].eq("Mapa"), "qtd_rpa"].sum())
qtd_rpa_reg = int(rpa_comp.loc[rpa_comp["origem"].eq("Registro"), "qtd_rpa"].sum())
qtd_rpa_mapa_com_valor = int(rpa_comp.loc[rpa_comp["origem"].eq("Mapa"), "qtd_rpa_com_valor"].sum())
qtd_rpa_mapa_sem_valor = int(rpa_comp.loc[rpa_comp["origem"].eq("Mapa"), "qtd_rpa_sem_valor"].sum())
qtd_rpa_reg_com_valor = int(rpa_comp.loc[rpa_comp["origem"].eq("Registro"), "qtd_rpa_com_valor"].sum())
qtd_rpa_reg_sem_valor = int(rpa_comp.loc[rpa_comp["origem"].eq("Registro"), "qtd_rpa_sem_valor"].sum())

rpa_mapa_mod = rpa_by_module(rpa_mapa, "valor_rpa_mapa")
rpa_reg_mod = rpa_by_module(rpa_registro, "valor_rpa_registro")
rpa_mod_comp = rpa_mapa_mod.merge(rpa_reg_mod, on="modulo", how="outer", suffixes=("_mapa", "_registro")).fillna(0)

kpi_cards([
    {"label": "Valor RPA — Mapa", "value": br_money(valor_mapa), "note": f"{br_int(qtd_rpa_mapa)} RPAs • {br_int(qtd_rpa_mapa_com_valor)} com valor | {br_int(qtd_rpa_mapa_sem_valor)} sem valor", "color": COLORS["rpa"]},
    {"label": "Valor RPA — Registro", "value": br_money(valor_reg), "note": f"{br_int(qtd_rpa_reg)} RPAs • {br_int(qtd_rpa_reg_com_valor)} com valor | {br_int(qtd_rpa_reg_sem_valor)} sem valor", "color": COLORS["rpa"]},
    {"label": "Delta RPA", "value": br_money(delta_rpa), "note": "Registro - Mapa", "color": COLORS["alerta"] if abs(delta_rpa) else COLORS["ok"]},
    {"label": "Horas P1 — aderência", "value": br_pct(comparativo_horas.loc[comparativo_horas['periodo'].eq('Período 1'), 'aderencia_%'].sum()), "note": "1ª + ESTENDIDA", "color": COLORS["horas"]},
    {"label": "Horas P2 — aderência", "value": br_pct(comparativo_horas.loc[comparativo_horas['periodo'].eq('Período 2'), 'aderencia_%'].sum()), "note": "2ª + ESTENDIDA", "color": COLORS["horas"]},
    {"label": "Delta docente P1", "value": f"{br_int(delta_demanda_p1)}h", "note": "Demanda planejada - CH letiva alocável", "color": COLORS["gap"] if delta_demanda_p1 > 0 else COLORS["ok"]},
    {"label": "Delta docente P2", "value": f"{br_int(delta_demanda_p2)}h", "note": "Demanda planejada - CH letiva alocável", "color": COLORS["gap"] if delta_demanda_p2 > 0 else COLORS["ok"]},
])

resumo_texto = f"""
No módulo {modulo_alvo}, o Mapa apresenta {br_int(qtd_planejado)} disciplinas planejadas com chave completa de conferência, enquanto o Registro apresenta {br_int(qtd_realizado)} disciplinas realizadas.
A conformidade Mapa × Registro é de {br_pct(conformidade_pct)}, com {br_int(qtd_conforme)} disciplinas aderentes. Foram identificados {br_int(qtd_faltantes)} itens planejados sem correspondência no Registro e {br_int(qtd_extras)} itens registrados sem correspondência no Mapa.

Na visão de horas, o bloco 1ª ordem + ESTENDIDA considera as disciplinas da 1ª ordem somadas às ESTENDIDAS, e o bloco 2ª ordem + ESTENDIDA considera as disciplinas da 2ª ordem somadas às ESTENDIDAS, sempre multiplicando a quantidade de disciplinas por 2 horas.
O valor RPA sinalizado no Mapa é {br_money(valor_mapa)} em {br_int(qtd_rpa_mapa)} disciplinas RPA ({br_int(qtd_rpa_mapa_com_valor)} com valor e {br_int(qtd_rpa_mapa_sem_valor)} sem valor). No Registro, há {br_int(qtd_rpa_reg)} disciplinas RPA no total, sendo {br_int(qtd_rpa_reg_com_valor)} com valor preenchido e {br_int(qtd_rpa_reg_sem_valor)} sem valor. O valor financeiro registrado é {br_money(valor_reg)}, resultando em delta de {br_money(delta_rpa)}.

Na visão docente, a CH letiva alocável foi calculada a partir da base docente atualizada. A regra operacional aplicada é: docentes com STATUS = ATIVO entram na capacidade; além disso, docentes que aparecem alocados no Registro do módulo também entram nos cálculos, mesmo que ainda não estejam ativos na base, pois representam alocação real. Esses casos permanecem listados em governança cadastral para posterior atualização no RM. O banco de capacidade contempla {br_int(total_docentes_banco)} registros docentes ({br_int(total_docentes_banco_unicos)} professores únicos), com CH letiva bruta de {br_int(total_ch_letiva_ativos_docente)}h e CH letiva alocável de {br_int(capacidade_periodo_total)}h por período. O delta de demanda contra CH letiva alocável é {br_int(delta_demanda_p1)}h em 1ª ordem + ESTENDIDA e {br_int(delta_demanda_p2)}h em 2ª ordem + ESTENDIDA. Nesta execução, há {br_int(docentes_fora_parametros_ativos_total)} docentes alocados no Registro que não constam como ativos na base docente atualizada; eles foram considerados nos cálculos por refletirem a operação realizada e foram mantidos em governança para atualização cadastral.
"""
insight_box("Resumo executivo", resumo_texto.replace("\n", "<br>"), conformidade_cor)

prioridades = []
if qtd_faltantes:
    prioridades.append(f"{br_int(qtd_faltantes)} disciplinas planejadas não aparecem no Registro.")
if qtd_extras:
    prioridades.append(f"{br_int(qtd_extras)} disciplinas aparecem no Registro sem equivalência no Mapa.")
if len(professor_divergencias):
    prioridades.append(f"{br_int(len(professor_divergencias))} divergências de professor foram encontradas em disciplinas conformes.")
if qtd_duplicatas_total:
    prioridades.append(f"{br_int(qtd_duplicatas_total)} duplicatas operacionais precisam ser tratadas; no Mapa, sinergias entre cursos foram desconsideradas como problema.")
if abs(delta_rpa) > 0:
    prioridades.append(f"Há diferença financeira de RPA de {br_money(delta_rpa)} entre Registro e Mapa.")
if delta_demanda_p1 > 0:
    prioridades.append(f"No Período 1, a demanda planejada supera a CH letiva alocável em {br_int(delta_demanda_p1)}h.")
if delta_demanda_p2 > 0:
    prioridades.append(f"No Período 2, a demanda planejada supera a CH letiva alocável em {br_int(delta_demanda_p2)}h.")
if total_docentes_sem_alocacao > 0:
    prioridades.append(f"Há {br_int(total_docentes_sem_alocacao)} docentes na base docente atualizada sem alocação no Registro do módulo.")
if not prioridades:
    prioridades.append("Não foram identificados gaps críticos pela regra atual de auditoria.")
insight_box("Prioridades recomendadas", "<br>".join([f"• {p}" for p in prioridades]), COLORS["alerta"] if prioridades else COLORS["ok"])

if not rpa_mod_comp.empty:
    display(Markdown("### Valor RPA por módulo"))
    display(rpa_mod_comp)
    plot_mod = rpa_mod_comp.melt(id_vars="modulo", value_vars=["valor_rpa_mapa", "valor_rpa_registro"], var_name="origem", value_name="valor")
    plot_mod["origem"] = plot_mod["origem"].map({"valor_rpa_mapa": "Mapa", "valor_rpa_registro": "Registro"})
    fig = px.bar(plot_mod, x="modulo", y="valor", color="origem", barmode="group", text="valor", color_discrete_map={"Mapa": COLORS["planejado"], "Registro": COLORS["realizado"]}, labels={"modulo": "Módulo", "valor": "Valor RPA", "origem": "Origem"})
    fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
    show_fig(aplicar_layout(fig, "Valor RPA por módulo — Mapa × Registro", height=460))

quality_rows = []
for fonte, df_base in [("Mapa", planejado), ("Registro", realizado)]:
    for field, label in [("disciplina_key", "Disciplina"), ("ordem", "Ordem"), ("dia_semana", "Dia da semana"), ("horario", "Horário")]:
        quality_rows.append({"fonte": fonte, "campo": label, "faltantes": int(df_base[field].eq("").sum()), "total": int(len(df_base))})
quality_df = pd.DataFrame(quality_rows)
quality_df["% faltante"] = np.where(quality_df["total"].gt(0), quality_df["faltantes"] / quality_df["total"] * 100, 0)

display(Markdown("### Qualidade dos campos da chave — tabela separada para não poluir os gráficos"))
display(quality_df)

In [ ]:
# ================================================================
# COBERTURA ANALÍTICA, DASHBOARD HTML PREMIUM E PDF EXECUTIVO
# ================================================================
section("Cobertura analítica", "Checklist das análises disponíveis para garantir uso amplo das duas planilhas.")

coverage = pd.DataFrame([
    {"Dimensão": "Planejado", "Análise": "Disciplinas AO VIVO por ordem", "Status": "OK"},
    {"Dimensão": "Planejado", "Análise": "Horas por período: 1ª + ESTENDIDA e 2ª + ESTENDIDA", "Status": "OK"},
    {"Dimensão": "Planejado", "Análise": "RPA/NF: quantidade, valor sinalizado, professor e curso", "Status": "OK"},
    {"Dimensão": "Planejado", "Análise": "Distribuição por dia da semana e ordem", "Status": "OK"},
    {"Dimensão": "Planejado", "Análise": "Leitura de sinergias do Mapa para evitar falso positivo de duplicidade", "Status": "OK"},
    {"Dimensão": "Realizado", "Análise": "Disciplinas registradas por ordem", "Status": "OK"},
    {"Dimensão": "Realizado", "Análise": "Horas por período: 1ª + ESTENDIDA e 2ª + ESTENDIDA", "Status": "OK"},
    {"Dimensão": "Realizado", "Análise": "RPA: total, com valor, sem valor e valor registrado", "Status": "OK"},
    {"Dimensão": "Comparação", "Análise": "Conformidade pela chave Disciplina + Ordem + Dia + Horário", "Status": "OK"},
    {"Dimensão": "Comparação", "Análise": "Planejado sem realizado", "Status": "OK"},
    {"Dimensão": "Comparação", "Análise": "Realizado sem planejado", "Status": "OK"},
    {"Dimensão": "Comparação", "Análise": "Divergência de professor com cargo realizado", "Status": "OK"},
    {"Dimensão": "Comparação", "Análise": "Duplicata operacional: Mapa considera Curso + Código Disciplina para respeitar sinergias", "Status": "OK"},
    {"Dimensão": "Financeiro", "Análise": "RPA Mapa × Registro por quantidade e valor", "Status": "OK"},
    {"Dimensão": "Docentes", "Análise": "Banco de docentes ativos pela base docente atualizada, incluindo professores ativos sem alocação", "Status": "OK"},
    {"Dimensão": "Docentes", "Análise": "Alocações no Registro sem STATUS = ATIVO na base docente atualizada isoladas para governança", "Status": "OK"},
    {"Dimensão": "Docentes", "Análise": "Visão individual com professor + cargo, CH semanal, CH letiva, CH letiva alocável, P1, P2 e saldos", "Status": "OK"},
    {"Dimensão": "Docentes", "Análise": "Delta demanda planejada das disciplinas menos CH letiva alocável dos docentes ativos para P1 e P2", "Status": "OK"},
    {"Dimensão": "Resultado", "Análise": "Resumo executivo automático", "Status": "OK"},
    {"Dimensão": "Exportação", "Análise": "Excel consolidado + HTML executivo + PDF completo", "Status": "OK"},
])
display(coverage)

# ---------------------------------------------------------------
# Exportação Excel
# ---------------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
excel_out = OUTPUT_DIR / f"SGA_Auditoria_Pedagogica_Modulo_{modulo_alvo}_{timestamp}.xlsx"
html_out = OUTPUT_DIR / f"SGA_Dashboard_Executivo_Modulo_{modulo_alvo}_{timestamp}.html"
pdf_out = OUTPUT_DIR / f"SGA_Relatorio_Completo_Modulo_{modulo_alvo}_{timestamp}.pdf"

try:
    writer_context = pd.ExcelWriter(excel_out, engine="xlsxwriter")
except Exception:
    writer_context = pd.ExcelWriter(excel_out, engine="openpyxl")

with writer_context as writer:
    pd.DataFrame([{
        "modulo": modulo_alvo,
        "ano": ano_alvo,
        "qtd_planejado": qtd_planejado,
        "qtd_realizado": qtd_realizado,
        "qtd_conforme": qtd_conforme,
        "conformidade_%": conformidade_pct,
        "rpa_mapa_total": qtd_rpa_mapa,
        "rpa_mapa_com_valor": qtd_rpa_mapa_com_valor,
        "rpa_mapa_sem_valor": qtd_rpa_mapa_sem_valor,
        "valor_rpa_mapa": valor_mapa,
        "rpa_registro_total": qtd_rpa_reg,
        "rpa_registro_com_valor": qtd_rpa_reg_com_valor,
        "rpa_registro_sem_valor": qtd_rpa_reg_sem_valor,
        "valor_rpa_registro": valor_reg,
        "delta_rpa": delta_rpa,
        "registros_docentes_ativos_parametros": parametros_docentes_ativos,
        "professores_unicos_ativos_parametros": parametros_docentes_ativos_unicos,
        "registros_docentes_base_capacidade": total_docentes_banco,
        "professores_unicos_base_capacidade": total_docentes_banco_unicos,
        "ch_letiva_ativa_total": parametros_ch_letiva_ativos,
        "ch_letiva_base_capacidade_total": total_ch_letiva_ativos_docente,
        "ch_letiva_alocavel_total": capacidade_periodo_total,
        "docentes_alocados": total_docentes_alocados,
        "docentes_sem_alocacao": total_docentes_sem_alocacao,
        "docentes_alocados_fora_parametros_ativos": docentes_fora_parametros_ativos_total,
        "horas_realizadas_ato2_p1": float(horas_realizado.loc[horas_realizado["periodo"].eq("Período 1"), "horas"].sum()),
        "horas_realizadas_ato2_p2": float(horas_realizado.loc[horas_realizado["periodo"].eq("Período 2"), "horas"].sum()),
        "capacidade_docente_base": "Docentes com STATUS = ATIVO na base docente atualizada + docentes alocados no módulo mesmo sem ativo, pois representam alocação real; divergências ficam em governança para atualização cadastral.",
        "regra_ch_letiva_alocavel": "piso(CH letiva / 2) × 2",
        "delta_docente_p1_horas": delta_demanda_p1,
        "delta_docente_p2_horas": delta_demanda_p2,
        "duplicatas_operacionais": qtd_duplicatas_total,
        "registro_linhas_brutas": registro_linhas_brutas,
        "registro_linhas_pos_dedup_exata": registro_linhas_usadas,
        "registro_linhas_usadas_calculo": len(realizado),
        "registro_duplicatas_exatas_grupos_modulo": registro_duplicatas_exatas_grupos_modulo,
        "registro_duplicatas_exatas_linhas_modulo": registro_duplicatas_exatas_linhas_modulo,
        "registro_duplicatas_removidas_calculo_modulo": registro_duplicatas_removidas_linhas_modulo,
        "registro_duplicatas_chave_operacional_grupos_modulo": registro_duplicatas_chave_grupos_modulo,
        "registro_duplicatas_chave_operacional_linhas_modulo": registro_duplicatas_chave_linhas_modulo,
        "registro_duplicatas_chave_operacional_removidas_modulo": registro_duplicatas_chave_removidas_linhas_modulo,
        "regra_duplicata_exata_registro": "Linhas totalmente iguais no Registro: identificar para governança, sem remover dos cálculos.",
        "regra_duplicata_chave_operacional_registro": "Mesma Disciplina + Ordem + Dia + Horário no Registro: identificar para governança, sem remover dos cálculos.",
    }]).to_excel(writer, sheet_name="Resumo", index=False)
    planejado_ordem.to_excel(writer, sheet_name="Planejado Ordem", index=False)
    horas_planejado.to_excel(writer, sheet_name="Horas Planejado", index=False)
    mapa_ao_vivo_usado.to_excel(writer, sheet_name="Mapa AO VIVO Usado", index=False)
    planejado_key.to_excel(writer, sheet_name="Mapa Chaves Planejado", index=False)
    realizado.to_excel(writer, sheet_name="Registro Tratado Usado", index=False)
    if isinstance(registro_duplicatas_exatas_modulo, pd.DataFrame) and not registro_duplicatas_exatas_modulo.empty:
        registro_duplicatas_exatas_modulo.to_excel(writer, sheet_name="Registro Dup Exatas", index=False)
    if isinstance(registro_duplicatas_removidas_modulo, pd.DataFrame) and not registro_duplicatas_removidas_modulo.empty:
        registro_duplicatas_removidas_modulo.to_excel(writer, sheet_name="Registro Dup Removidas", index=False)
    if isinstance(registro_duplicatas_chave_modulo, pd.DataFrame) and not registro_duplicatas_chave_modulo.empty:
        registro_duplicatas_chave_modulo.to_excel(writer, sheet_name="Registro Dup Chave", index=False)
    if isinstance(registro_duplicatas_chave_removidas_modulo, pd.DataFrame) and not registro_duplicatas_chave_removidas_modulo.empty:
        registro_duplicatas_chave_removidas_modulo.to_excel(writer, sheet_name="Registro Chave Remov", index=False)
    realizado_ordem.to_excel(writer, sheet_name="Realizado Ordem", index=False)
    horas_realizado.to_excel(writer, sheet_name="Horas Realizado", index=False)
    comparativo_horas.to_excel(writer, sheet_name="Horas Comparativo", index=False)
    rpa_comp.to_excel(writer, sheet_name="RPA Comparativo", index=False)
    rpa_mapa.to_excel(writer, sheet_name="RPA Mapa", index=False)
    rpa_registro.to_excel(writer, sheet_name="RPA Registro", index=False)
    comparativo_chaves.to_excel(writer, sheet_name="Conformidade Chaves", index=False)
    planejado_sem_realizado.to_excel(writer, sheet_name="Planejado sem Realizado", index=False)
    realizado_sem_planejado.to_excel(writer, sheet_name="Realizado sem Planejado", index=False)
    professor_divergencias.to_excel(writer, sheet_name="Diverg Professor", index=False)
    duplicatas_totais.to_excel(writer, sheet_name="Duplicatas Operacionais", index=False)
    parametros_docentes.to_excel(writer, sheet_name="Base Docente Atualizada", index=False)
    docente_delta_geral[cols_delta_exibir].to_excel(writer, sheet_name="Delta Docente Geral", index=False)
    docente_individual.to_excel(writer, sheet_name="Docentes Individual", index=False)
    docente_geral.to_excel(writer, sheet_name="Docentes Cargo CH", index=False)
    docente_por_ch.to_excel(writer, sheet_name="Docentes Grupo CH", index=False)
    if isinstance(docente_fora_parametros_ativos, pd.DataFrame) and not docente_fora_parametros_ativos.empty:
        docente_fora_parametros_ativos.to_excel(writer, sheet_name="Governanca Docentes", index=False)
    if not docentes_presencial_alocados.empty:
        docentes_presencial_por_tipo.to_excel(writer, sheet_name="Docentes Presencial", index=False, startrow=0)
        startrow_detalhe_presencial = len(docentes_presencial_por_tipo) + 3
        docentes_presencial_alocados[cols_presencial_detalhe].to_excel(
            writer, sheet_name="Docentes Presencial", index=False, startrow=startrow_detalhe_presencial
        )
        ws_pres = writer.sheets["Docentes Presencial"]
        try:
            ws_pres.write(startrow_detalhe_presencial - 2, 0, "CASO À PARTE — não entra em nenhum total geral do relatório. Detalhe por docente abaixo:")
        except Exception:
            pass
    quality_df.to_excel(writer, sheet_name="Qualidade Chave", index=False)
    coverage.to_excel(writer, sheet_name="Cobertura Analitica", index=False)

    # Formatação executiva do Excel quando a engine xlsxwriter estiver disponível.
    try:
        workbook = writer.book
        if hasattr(workbook, "add_format"):
            header_fmt = workbook.add_format({"bold": True, "font_color": "white", "bg_color": "#0F172A", "border": 1})
            hour_fmt = workbook.add_format({"num_format": '#,##0"h"', "border": 1})
            pct_fmt = workbook.add_format({"num_format": "0.0%", "border": 1})
            money_fmt = workbook.add_format({"num_format": "R$ #,##0.00", "border": 1})
            text_fmt = workbook.add_format({"border": 1})
            for sheet_name, ws in writer.sheets.items():
                try:
                    ws.freeze_panes(1, 0)
                    ws.autofilter(0, 0, 0, 20)
                    ws.set_row(0, None, header_fmt)
                    ws.set_column(0, 20, 18, text_fmt)
                except Exception:
                    pass
            if "Delta Docente Geral" in writer.sheets:
                ws = writer.sheets["Delta Docente Geral"]
                # colunas: período, disciplinas, demanda, CH bruta, CH alocável, delta demanda, realizado total, realizado ativos, deltas, percentuais
                ws.set_column(0, 1, 24, text_fmt)
                ws.set_column(2, 9, 20, hour_fmt)
                ws.set_column(10, 12, 20, workbook.add_format({"num_format": '0.0"%"', "border": 1}))
    except Exception as e:
        print(f"Aviso: formatação avançada do Excel não aplicada: {e}")

# ---------------------------------------------------------------

print(f"Excel consolidado gerado em: {excel_out.resolve()}")


In [ ]:
# ================================================================
# V52 — Relatório executivo reestruturado e análises adicionais
# ================================================================

def periodo_exec_label(p):
    return {
        "Período 1": "1ª ordem + ESTENDIDA",
        "Período 2": "2ª ordem + ESTENDIDA",
        "1ª": "1ª ordem",
        "2ª": "2ª ordem",
        "ESTENDIDA": "ESTENDIDA",
    }.get(str(p), str(p))

def expandir_periodos_operacionais(df: pd.DataFrame, fonte: str) -> pd.DataFrame:
    if df is None or df.empty or "ordem" not in df.columns:
        return pd.DataFrame()
    linhas = []
    for _, r in df.iterrows():
        ordem = str(r.get("ordem", "")).strip()
        if ordem == "1ª":
            periodos = ["1ª ordem + ESTENDIDA"]
        elif ordem == "2ª":
            periodos = ["2ª ordem + ESTENDIDA"]
        elif ordem == "ESTENDIDA":
            periodos = ["1ª ordem + ESTENDIDA", "2ª ordem + ESTENDIDA"]
        else:
            periodos = []
        for p in periodos:
            rr = r.to_dict()
            rr["periodo_exec"] = p
            rr["horas_demandadas"] = 2
            rr["fonte_periodo"] = fonte
            linhas.append(rr)
    return pd.DataFrame(linhas)

def inferir_cluster_por_curso(curso):
    # Os rótulos devolvidos aqui precisam ser IDÊNTICOS (grafia e acentuação) aos rótulos
    # que já vêm preenchidos na coluna CLUSTER do Mapa. Caso contrário, o mesmo cluster
    # aparece duplicado na análise: uma vez com o rótulo original ("GESTÃO E NEGÓCIOS")
    # e outra com o rótulo inferido sem acento ("GESTAO E NEGOCIOS").
    s = normalize_key(curso)
    if "ENGENHARIA" in s or "ARQUITETURA" in s or "MANUTENCAO INDUSTRIAL" in s or "PRODUCAO INDUSTRIAL" in s:
        return "ENGENHARIAS E ARQUITETURA"
    if "LICENCIATURA" in s or "PEDAGOGIA" in s:
        return "EDUCAÇÃO"
    if any(k in s for k in ["FARMACIA", "BIOMEDICINA", "NUTRICAO", "FISIOTERAPIA", "ENFERMAGEM", "EDUCACAO FISICA", "GERONTOLOGIA"]):
        return "SAÚDE E BEM-ESTAR"
    if any(k in s for k in ["ADMINISTRACAO", "GESTAO", "CIENCIAS CONTABEIS", "ECONOMICAS", "COMERCIO", "LOGISTICA", "MARKETING"]):
        return "GESTÃO E NEGÓCIOS"
    if any(k in s for k in ["SOFTWARE", "SISTEMAS", "REDES", "TECNOLOGIA", "COMPUTACAO"]):
        return "TECNOLOGIA"
    return "OUTROS"

# Tokens de erro de fórmula do Excel (e variações comuns) que não representam um cluster
# de fato — precisam ser tratados como "sem cluster" para acionar a inferência por curso,
# em vez de aparecerem como um cluster literal "#N/D" no relatório.
CLUSTER_TOKENS_INVALIDOS_V52 = {
    "", "NAN", "NONE", "#N/D", "#N/A", "#NULO!", "#NUL!", "#DIV/0!",
    "#VALOR!", "#REF!", "#NOME?", "#NÚM!", "#NUM!", "N/D", "N/A",
}

def _cluster_valido_v52(v) -> bool:
    s = str(v).strip().upper()
    return s not in CLUSTER_TOKENS_INVALIDOS_V52

def preparar_cluster(df):
    d = df.copy()
    if "cluster" not in d.columns or not d["cluster"].map(_cluster_valido_v52).any():
        d["cluster"] = d.get("curso_nome", "").map(inferir_cluster_por_curso)
    else:
        d["cluster"] = d["cluster"].where(d["cluster"].map(_cluster_valido_v52), np.nan)
        d["cluster"] = d["cluster"].fillna(d.get("curso_nome", "").map(inferir_cluster_por_curso))
    # Padroniza maiúsculas/espaços para que o mesmo cluster nunca seja contado em duas
    # linhas por diferença pontual de grafia (ex.: espaço duplo, caixa mista).
    d["cluster"] = d["cluster"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
    return d

def resumo_ordem(df):
    if df is None or df.empty:
        return pd.DataFrame(columns=["ordem", "disciplinas"])
    return df.groupby("ordem", dropna=False).size().reindex(["1ª","2ª","ESTENDIDA"]).fillna(0).astype(int).reset_index(name="disciplinas")

def resumo_horas_periodo(df):
    exp = expandir_periodos_operacionais(df, "base")
    if exp.empty:
        return pd.DataFrame(columns=["periodo_exec","horas_demandadas"])
    return exp.groupby("periodo_exec", as_index=False)["horas_demandadas"].sum()

def is_rpa_df(df, origem):
    if df is None or df.empty:
        return df.copy()
    d = df.copy()
    if origem == "mapa":
        if "contrato" in d.columns:
            return d[d["contrato"].astype(str).str.upper().eq("RPA/NF")].copy()
    else:
        if "is_rpa_registro" in d.columns:
            return d[d["is_rpa_registro"].fillna(False).astype(bool)].copy()
        if "tem_rpa_registro" in d.columns:
            return d[d["tem_rpa_registro"].fillna(False).astype(bool)].copy()
        if "cargo" in d.columns:
            return d[d["cargo"].astype(str).str.upper().str.contains("RPA", na=False)].copy()
    return d.iloc[0:0].copy()

# Bases v52:
# - planejado_v52 usa chaves únicas do Mapa para evitar inflar disciplinas por sinergia de cursos;
# - planejado_curso_base_v52 preserva as linhas por curso para análises de demanda por curso/cluster;
# - realizado_v52 considera todas as linhas válidas do Registro, sem excluir duplicidades.
planejado_v52 = planejado_key.copy()
planejado_curso_base_v52 = planejado.copy()
realizado_v52 = realizado.copy()
planejado_periodos_v52 = expandir_periodos_operacionais(planejado_v52, "Planejado")
realizado_periodos_v52 = expandir_periodos_operacionais(realizado_v52, "Realizado")

# RPA
rpa_mapa_v52 = rpa_mapa.copy() if isinstance(rpa_mapa, pd.DataFrame) and not rpa_mapa.empty else is_rpa_df(planejado_v52, "mapa")
rpa_registro_v52 = is_rpa_df(realizado_v52, "registro")
rpa_mapa_periodos_v52 = expandir_periodos_operacionais(rpa_mapa_v52, "Planejado")
rpa_reg_periodos_v52 = expandir_periodos_operacionais(rpa_registro_v52, "Realizado")

# Ordens e horas
ordem_plan_v52 = resumo_ordem(planejado_v52).rename(columns={"disciplinas": "planejado"})
ordem_real_v52 = resumo_ordem(realizado_v52).rename(columns={"disciplinas": "realizado"})
ordem_comp_v52 = ordem_plan_v52.merge(ordem_real_v52, on="ordem", how="outer").fillna(0)
ordem_comp_v52["delta"] = ordem_comp_v52["realizado"] - ordem_comp_v52["planejado"]

horas_plan_v52 = resumo_horas_periodo(planejado_v52).rename(columns={"horas_demandadas": "planejado_horas"})
horas_real_v52 = resumo_horas_periodo(realizado_v52).rename(columns={"horas_demandadas": "realizado_horas"})
horas_periodo_v52 = horas_plan_v52.merge(horas_real_v52, on="periodo_exec", how="outer").fillna(0)
horas_periodo_v52["delta_realizado_menos_planejado"] = horas_periodo_v52["realizado_horas"] - horas_periodo_v52["planejado_horas"]
horas_periodo_v52["delta_realizado_planejado"] = horas_periodo_v52["delta_realizado_menos_planejado"]
horas_periodo_v52["ch_letiva_alocavel"] = capacidade_periodo_total
horas_periodo_v52["delta_planejado_mapa_menos_ch_alocavel"] = horas_periodo_v52["planejado_horas"] - capacidade_periodo_total
horas_periodo_v52["delta_realizado_registro_menos_ch_alocavel"] = horas_periodo_v52["realizado_horas"] - capacidade_periodo_total
horas_periodo_v52["delta_demanda_menos_ch_alocavel"] = horas_periodo_v52["delta_planejado_mapa_menos_ch_alocavel"]

# RPA por ordem e horas
rpa_ordem_plan_v52 = resumo_ordem(rpa_mapa_v52).rename(columns={"disciplinas": "rpa_planejado"})
rpa_ordem_real_v52 = resumo_ordem(rpa_registro_v52).rename(columns={"disciplinas": "rpa_realizado"})
rpa_ordem_comp_v52 = rpa_ordem_plan_v52.merge(rpa_ordem_real_v52, on="ordem", how="outer").fillna(0)
rpa_ordem_comp_v52["delta"] = rpa_ordem_comp_v52["rpa_realizado"] - rpa_ordem_comp_v52["rpa_planejado"]

rpa_horas_plan_v52 = rpa_mapa_periodos_v52.groupby("periodo_exec", as_index=False)["horas_demandadas"].sum().rename(columns={"horas_demandadas":"rpa_planejado_horas"}) if not rpa_mapa_periodos_v52.empty else pd.DataFrame(columns=["periodo_exec","rpa_planejado_horas"])
rpa_horas_real_v52 = rpa_reg_periodos_v52.groupby("periodo_exec", as_index=False)["horas_demandadas"].sum().rename(columns={"horas_demandadas":"rpa_realizado_horas"}) if not rpa_reg_periodos_v52.empty else pd.DataFrame(columns=["periodo_exec","rpa_realizado_horas"])
rpa_horas_v52 = rpa_horas_plan_v52.merge(rpa_horas_real_v52, on="periodo_exec", how="outer").fillna(0)
rpa_horas_v52["delta_realizado_menos_planejado_horas"] = rpa_horas_v52["rpa_realizado_horas"] - rpa_horas_v52["rpa_planejado_horas"]
rpa_horas_v52["delta_horas"] = rpa_horas_v52["delta_realizado_menos_planejado_horas"]

# Simulação RPA
valor_rpa_registro_total_v52 = pd.to_numeric(rpa_registro_v52.get("valor_rpa_registro", pd.Series(dtype=float)), errors="coerce").fillna(0).sum() if not rpa_registro_v52.empty else 0
rpa_com_valor_v52 = rpa_registro_v52[pd.to_numeric(rpa_registro_v52.get("valor_rpa_registro", pd.Series(dtype=float)), errors="coerce").fillna(0).gt(0)].copy() if not rpa_registro_v52.empty else pd.DataFrame()
rpa_sem_valor_v52 = rpa_registro_v52[pd.to_numeric(rpa_registro_v52.get("valor_rpa_registro", pd.Series(dtype=float)), errors="coerce").fillna(0).le(0)].copy() if not rpa_registro_v52.empty else pd.DataFrame()
media_rpa_por_disciplina_v52 = (pd.to_numeric(rpa_com_valor_v52.get("valor_rpa_registro", pd.Series(dtype=float)), errors="coerce").fillna(0).sum() / len(rpa_com_valor_v52)) if len(rpa_com_valor_v52) else 0
rpa_estimado_sem_valor_v52 = media_rpa_por_disciplina_v52 * len(rpa_sem_valor_v52)
rpa_total_estimado_v52 = valor_rpa_registro_total_v52 + rpa_estimado_sem_valor_v52
rpa_simulacao_v52 = pd.DataFrame([{
    "rpa_com_valor": len(rpa_com_valor_v52),
    "rpa_sem_valor": len(rpa_sem_valor_v52),
    "valor_registrado": valor_rpa_registro_total_v52,
    "media_por_disciplina_com_valor": media_rpa_por_disciplina_v52,
    "estimativa_sem_valor": rpa_estimado_sem_valor_v52,
    "total_estimado": rpa_total_estimado_v52,
}])

# RPA pendente: sinalizado no mapa, mas sem chave registrada como RPA no Registro
rpa_reg_keys = set(rpa_registro_v52.get("chave_conformidade", pd.Series(dtype=str)).dropna().astype(str))
rpa_pendente_v52 = rpa_mapa_v52[~rpa_mapa_v52["chave_conformidade"].astype(str).isin(rpa_reg_keys)].copy()
rpa_pendente_v52 = rpa_pendente_v52[["curso_nome","disciplina","ordem","dia_semana","horario","professor_planejado","valor_rpa_mapa","chave_conformidade"]].drop_duplicates() if not rpa_pendente_v52.empty else pd.DataFrame()

# Cursos com RPA
rpa_mapa_cursos_v52 = preparar_cluster(is_rpa_df(planejado_curso_base_v52, "mapa"))
top_cursos_rpa_v52 = (
    rpa_mapa_cursos_v52.assign(horas=2)
    .groupby(["curso_nome"], dropna=False)["horas"].sum()
    .reset_index()
    .sort_values("horas", ascending=False)
) if not rpa_mapa_cursos_v52.empty else pd.DataFrame(columns=["curso_nome","horas"])

# Realizado por dia: disciplinas e docentes
realizado_dia_v52 = realizado_v52.copy()
dia_disc_v52 = realizado_dia_v52.groupby("dia_semana", dropna=False).size().reset_index(name="qtd_disciplinas")

def _is_rpa_row_v52(df):
    if df is None or df.empty:
        return pd.Series(False, index=df.index if df is not None else [])
    mask = pd.Series(False, index=df.index)
    if "is_rpa_registro" in df.columns:
        mask = mask | df["is_rpa_registro"].fillna(False).astype(bool)
    if "tem_rpa_registro" in df.columns:
        mask = mask | df["tem_rpa_registro"].fillna(False).astype(bool)
    if "cargo" in df.columns:
        mask = mask | df["cargo"].astype(str).map(normalize_key).str.contains("RPA", na=False)
    if "modelo_contrato" in df.columns:
        mask = mask | df["modelo_contrato"].astype(str).map(normalize_key).str.contains("RPA", na=False)
    return mask

realizado_dia_v52["rpa_linha_v52"] = _is_rpa_row_v52(realizado_dia_v52)

# Regra v52:
# - Docentes CLT/Regente/etc. = professores únicos por dia;
# - RPA = quantidade de alocações/disciplinas RPA por dia, pois RPA não deve aparecer como "1 professor".
dia_rpa_qtd_v52 = realizado_dia_v52.groupby("dia_semana", dropna=False)["rpa_linha_v52"].sum().reset_index(name="qtd_rpa_dia")
dia_doc_nao_rpa_v52 = (
    realizado_dia_v52[~realizado_dia_v52["rpa_linha_v52"].fillna(False)]
    .groupby("dia_semana", dropna=False)["professor_realizado_key"]
    .nunique()
    .reset_index(name="qtd_docentes_sem_rpa")
)
dia_doc_v52 = dia_doc_nao_rpa_v52.merge(dia_rpa_qtd_v52, on="dia_semana", how="outer").fillna(0)
dia_doc_v52["qtd_docentes"] = dia_doc_v52["qtd_docentes_sem_rpa"] + dia_doc_v52["qtd_rpa_dia"]

# Visão por cargo para hover/tabela de apoio:
# - CLT/Regente/etc.: docentes únicos por cargo no dia;
# - RPA: quantidade de alocações/disciplinas RPA no dia.
cargo_parts_v52 = []
for (dia, cargo), dgrp in realizado_dia_v52.groupby(["dia_semana", "cargo"], dropna=False):
    cargo_norm = normalize_key(cargo)
    is_rpa_cargo = dgrp["rpa_linha_v52"].fillna(False).any() or ("RPA" in cargo_norm)
    if is_rpa_cargo:
        qtd = int(dgrp["rpa_linha_v52"].fillna(False).sum())
        if qtd == 0:
            qtd = len(dgrp)
        label = f"{cargo or 'RPA'}: {qtd} RPA(s)"
        tipo = "RPA"
    else:
        qtd = int(dgrp["professor_realizado_key"].nunique())
        label = f"{cargo or 'Sem cargo'}: {qtd} docente(s)"
        tipo = "Docente"
    cargo_parts_v52.append({"dia_semana": dia, "cargo": cargo, "qtd": qtd, "tipo_contagem": tipo, "label_hover": label})
dia_cargo_v52 = pd.DataFrame(cargo_parts_v52)
cargo_hover_v52 = (
    dia_cargo_v52.sort_values(["dia_semana", "tipo_contagem", "qtd"], ascending=[True, True, False])
    .groupby("dia_semana")["label_hover"]
    .apply(lambda s: "<br>".join(s.astype(str)))
    .reset_index(name="docentes_por_cargo")
)
dia_realizado_v52 = (
    dia_disc_v52
    .merge(dia_doc_v52[["dia_semana", "qtd_docentes_sem_rpa", "qtd_rpa_dia", "qtd_docentes"]], on="dia_semana", how="outer")
    .merge(cargo_hover_v52, on="dia_semana", how="left")
    .fillna(0)
)
dia_realizado_v52["dia_semana"] = pd.Categorical(dia_realizado_v52["dia_semana"], DAY_SORT, ordered=True)
dia_realizado_v52 = dia_realizado_v52.sort_values("dia_semana")

# Inconsistências docentes: choques de alocação no mesmo bloco temporal
doc_conf_base_v52 = expandir_periodos_operacionais(realizado_v52, "Realizado")
doc_conf_v52 = pd.DataFrame()
if not doc_conf_base_v52.empty:
    group_cols_conf = ["professor_realizado_key","professor_realizado","periodo_exec","dia_semana","horario"]
    agg_conf = (
        doc_conf_base_v52.groupby(group_cols_conf, dropna=False)
        .agg(
            qtd_linhas=("disciplina", "size"),
            qtd_disciplinas=("disciplina_key", "nunique"),
            disciplinas=("disciplina", lambda s: "; ".join(pd.Series(s).dropna().astype(str).drop_duplicates().head(8))),
            ordens=("ordem", lambda s: "; ".join(pd.Series(s).dropna().astype(str).drop_duplicates())),
        )
        .reset_index()
    )
    doc_conf_v52 = agg_conf[(agg_conf["qtd_linhas"] > 1) & (agg_conf["qtd_disciplinas"] > 1)].copy()
    doc_conf_v52["regra"] = "Mesmo docente no mesmo bloco temporal, dia e horário. 1ª cruza com ESTENDIDA nas primeiras 5 semanas; 2ª cruza com ESTENDIDA nas últimas 5 semanas."
    doc_conf_v52 = doc_conf_v52.sort_values(["periodo_exec","dia_semana","horario","professor_realizado"])

# Docentes ativos e considerados
# Quantidade oficial de docentes ativos na base atualizada.
# Usa a métrica calculada na leitura da base; fallback robusto por coluna docente_ativo/status.
if "parametros_docentes_ativos_unicos" in globals():
    qtd_docentes_ativos_base_v52 = int(parametros_docentes_ativos_unicos)
elif "parametros_docentes" in globals() and isinstance(parametros_docentes, pd.DataFrame) and not parametros_docentes.empty and "professor_key" in parametros_docentes.columns:
    qtd_docentes_ativos_base_v52 = int(parametros_docentes["professor_key"].nunique())
else:
    _base_ativa_tmp = base_docente.copy() if isinstance(base_docente, pd.DataFrame) else pd.DataFrame()
    if not _base_ativa_tmp.empty and "docente_ativo" in _base_ativa_tmp.columns:
        _mask_ativo = _base_ativa_tmp["docente_ativo"].astype(str).str.upper().isin(["TRUE","VERDADEIRO","SIM","1"])
    elif not _base_ativa_tmp.empty and "status" in _base_ativa_tmp.columns:
        _mask_ativo = _base_ativa_tmp["status"].astype(str).map(normalize_key).eq("ATIVO")
    else:
        _mask_ativo = pd.Series(False, index=_base_ativa_tmp.index)
    qtd_docentes_ativos_base_v52 = int(_base_ativa_tmp.loc[_mask_ativo, "professor_key"].nunique()) if "professor_key" in _base_ativa_tmp.columns else 0
docentes_ativos_base_v52 = pd.DataFrame([{"docentes_ativos_base": qtd_docentes_ativos_base_v52}])

# Docentes por cargo
doc_ind_v52 = docente_individual.copy()
for col in ["ch_semanal","ch_letiva","ch_letiva_alocavel","horas_p1","horas_p2"]:
    if col in doc_ind_v52.columns:
        doc_ind_v52[col] = pd.to_numeric(doc_ind_v52[col], errors="coerce").fillna(0)
if "horas_total_periodos" not in doc_ind_v52.columns:
    doc_ind_v52["horas_total_periodos"] = doc_ind_v52.get("horas_p1", 0) + doc_ind_v52.get("horas_p2", 0)
doc_ind_v52["utilizacao_1_ordem_estendida_%"] = np.where(doc_ind_v52["ch_letiva_alocavel"].gt(0), doc_ind_v52["horas_p1"]/doc_ind_v52["ch_letiva_alocavel"]*100, 0)
doc_ind_v52["utilizacao_2_ordem_estendida_%"] = np.where(doc_ind_v52["ch_letiva_alocavel"].gt(0), doc_ind_v52["horas_p2"]/doc_ind_v52["ch_letiva_alocavel"]*100, 0)
doc_ind_v52["acima_limite"] = (doc_ind_v52["horas_p1"] > doc_ind_v52["ch_letiva_alocavel"]) | (doc_ind_v52["horas_p2"] > doc_ind_v52["ch_letiva_alocavel"])
doc_ind_v52["sem_alocacao"] = doc_ind_v52["horas_total_periodos"].eq(0)

doc_cargo_v52 = (
    doc_ind_v52.groupby("cargo", dropna=False)
    .agg(
        headcount=("professor","nunique"),
        ch_contratada=("ch_semanal","sum"),
        ch_letiva=("ch_letiva","sum"),
        ch_letiva_alocavel=("ch_letiva_alocavel","sum"),
        horas_1_ordem_estendida=("horas_p1","sum"),
        horas_2_ordem_estendida=("horas_p2","sum"),
        docentes_acima_limite=("acima_limite","sum"),
        docentes_sem_alocacao=("sem_alocacao","sum"),
    )
    .reset_index()
)
doc_cargo_v52["utilizacao_1_ordem_estendida_%"] = np.where(doc_cargo_v52["ch_letiva_alocavel"].gt(0), doc_cargo_v52["horas_1_ordem_estendida"]/doc_cargo_v52["ch_letiva_alocavel"]*100, 0)
doc_cargo_v52["utilizacao_2_ordem_estendida_%"] = np.where(doc_cargo_v52["ch_letiva_alocavel"].gt(0), doc_cargo_v52["horas_2_ordem_estendida"]/doc_cargo_v52["ch_letiva_alocavel"]*100, 0)

# v52 — Headcount RPA não pode ser professor único.
# Para RPA, o headcount representa a quantidade total de contratações/alocações RPA no Registro.
rpa_headcount_contratado_v52 = int(len(rpa_registro_v52)) if isinstance(rpa_registro_v52, pd.DataFrame) else 0
mask_cargo_rpa_v52 = doc_cargo_v52["cargo"].astype(str).map(normalize_key).str.contains("RPA", na=False)
if mask_cargo_rpa_v52.any():
    doc_cargo_v52.loc[mask_cargo_rpa_v52, "headcount"] = rpa_headcount_contratado_v52

# v52 — Regra de governança docente:
# Professor de Ensino Superior Presencial e Tutor Facilitador Online não entram
# nas contagens das colunas "Acima limite" e "Sem alocação" do quadro executivo.
mask_cargos_sem_alerta_v52 = (
    doc_cargo_v52["cargo"].astype(str).map(normalize_key).isin([
        "PROFESSOR DE ENSINO SUPERIOR PRESENCIAL",
        "TUTOR FACILITADOR ONLINE",
    ])
    | doc_cargo_v52["cargo"].astype(str).map(normalize_key).str.contains("RPA", na=False)
)
doc_cargo_v52.loc[mask_cargos_sem_alerta_v52, ["docentes_acima_limite", "docentes_sem_alocacao"]] = 0

total_row_doc = pd.DataFrame([{
    "cargo": "TOTAL GERAL",
    "headcount": doc_cargo_v52["headcount"].sum(),
    "ch_contratada": doc_cargo_v52["ch_contratada"].sum(),
    "ch_letiva": doc_cargo_v52["ch_letiva"].sum(),
    "ch_letiva_alocavel": doc_cargo_v52["ch_letiva_alocavel"].sum(),
    "horas_1_ordem_estendida": doc_cargo_v52["horas_1_ordem_estendida"].sum(),
    "horas_2_ordem_estendida": doc_cargo_v52["horas_2_ordem_estendida"].sum(),
    "docentes_acima_limite": doc_cargo_v52["docentes_acima_limite"].sum(),
    "docentes_sem_alocacao": doc_cargo_v52["docentes_sem_alocacao"].sum(),
}])
total_row_doc["utilizacao_1_ordem_estendida_%"] = np.where(total_row_doc["ch_letiva_alocavel"].gt(0), total_row_doc["horas_1_ordem_estendida"]/total_row_doc["ch_letiva_alocavel"]*100, 0)
total_row_doc["utilizacao_2_ordem_estendida_%"] = np.where(total_row_doc["ch_letiva_alocavel"].gt(0), total_row_doc["horas_2_ordem_estendida"]/total_row_doc["ch_letiva_alocavel"]*100, 0)
doc_cargo_total_v52 = pd.concat([doc_cargo_v52, total_row_doc], ignore_index=True)

# Cluster demand semanal: planejado e realizado vinculando realizado ao mapa por chave
# Correção: uma mesma aula (chave_conformidade) pode aparecer em mais de uma linha do Mapa
# quando a disciplina é compartilhada por sinergia entre cursos (planejado_curso_base_v52
# preserva 1 linha por curso). Se somarmos horas direto dessa base "por curso", cada aula
# compartilhada é contada uma vez por curso, inflando o total de horas planejadas.
# A correção usa a base já deduplicada por aula (planejado_v52) para as horas, e usa a
# visão "por curso" apenas para descobrir o cluster de cada aula (1 cluster por chave).
planejado_cluster_curso_v52 = preparar_cluster(planejado_curso_base_v52)
mapa_cluster_lookup = planejado_cluster_curso_v52[["chave_conformidade","cluster"]].drop_duplicates("chave_conformidade")
# Fallback por disciplina: quando o Registro não bate a chave exata (disciplina+ordem+dia+
# horário) com o Mapa — aula executada em outro horário/dia do planejado, por exemplo —
# ainda é possível recuperar o cluster pela disciplina, já que o cluster é uma propriedade
# do curso que oferece a disciplina, não do horário em que ela é dada.
mapa_cluster_lookup_disciplina = planejado_cluster_curso_v52[["disciplina_key","cluster"]].drop_duplicates("disciplina_key")

planejado_cluster_v52 = planejado_v52.drop(columns=["cluster"], errors="ignore").merge(mapa_cluster_lookup, on="chave_conformidade", how="left")
planejado_cluster_v52["cluster"] = planejado_cluster_v52["cluster"].fillna("SEM CLUSTER IDENTIFICADO")
planejado_cluster_periodos_v52 = expandir_periodos_operacionais(planejado_cluster_v52, "Planejado")
if not realizado_periodos_v52.empty:
    realizado_cluster_periodos_v52 = realizado_periodos_v52.merge(mapa_cluster_lookup, on="chave_conformidade", how="left")
    sem_cluster_mask = realizado_cluster_periodos_v52["cluster"].isna()
    if sem_cluster_mask.any() and "disciplina_key" in realizado_cluster_periodos_v52.columns:
        cluster_por_disciplina = realizado_cluster_periodos_v52.loc[sem_cluster_mask, "disciplina_key"].map(
            mapa_cluster_lookup_disciplina.set_index("disciplina_key")["cluster"]
        )
        realizado_cluster_periodos_v52.loc[sem_cluster_mask, "cluster"] = cluster_por_disciplina
    realizado_cluster_periodos_v52["cluster"] = realizado_cluster_periodos_v52["cluster"].fillna("SEM CORRESPONDÊNCIA NO MAPA")
else:
    realizado_cluster_periodos_v52 = pd.DataFrame()

cluster_plan_v52 = planejado_cluster_periodos_v52.groupby(["periodo_exec","cluster"], dropna=False)["horas_demandadas"].sum().reset_index(name="horas_planejado") if not planejado_cluster_periodos_v52.empty else pd.DataFrame()
cluster_real_v52 = realizado_cluster_periodos_v52.groupby(["periodo_exec","cluster"], dropna=False)["horas_demandadas"].sum().reset_index(name="horas_realizado") if not realizado_cluster_periodos_v52.empty else pd.DataFrame()
cluster_v52 = cluster_plan_v52.merge(cluster_real_v52, on=["periodo_exec","cluster"], how="outer").fillna(0)

# Heatmaps planejado e realizado por período
def heatmap_base(df_periodos):
    if df_periodos is None or df_periodos.empty:
        return pd.DataFrame()
    h = df_periodos.groupby(["periodo_exec","horario","dia_semana"], dropna=False).size().reset_index(name="disciplinas")
    h["periodo_horario"] = h["periodo_exec"].astype(str) + " | " + h["horario"].astype(str)
    pvt = h.pivot_table(index="periodo_horario", columns="dia_semana", values="disciplinas", aggfunc="sum", fill_value=0)
    cols = [c for c in DAY_SORT if c in pvt.columns and pd.to_numeric(pvt[c], errors="coerce").fillna(0).sum() > 0]
    return pvt[cols] if cols else pvt

heat_plan_v52 = heatmap_base(planejado_periodos_v52)
heat_real_v52 = heatmap_base(realizado_periodos_v52)

# ---------------------------
# Figuras Plotly interativas
# ---------------------------
fig_v52_horas = go.Figure()
fig_v52_horas.add_bar(x=horas_periodo_v52["periodo_exec"], y=horas_periodo_v52["planejado_horas"], name="Planejado", marker_color=COLORS["planejado"], text=horas_periodo_v52["planejado_horas"])
fig_v52_horas.add_bar(x=horas_periodo_v52["periodo_exec"], y=horas_periodo_v52["realizado_horas"], name="Realizado", marker_color=COLORS["realizado"], text=horas_periodo_v52["realizado_horas"])
fig_v52_horas.add_scatter(x=horas_periodo_v52["periodo_exec"], y=horas_periodo_v52["ch_letiva_alocavel"], mode="lines+markers+text", name="CH letiva alocável", line=dict(color=COLORS["horas"], width=3), text=[f"{int(v)}h" for v in horas_periodo_v52["ch_letiva_alocavel"]], textposition="top center")
fig_v52_horas.update_traces(texttemplate="%{text:.0f}h", textposition="outside", selector=dict(type="bar"))
fig_v52_horas.update_layout(barmode="group", title="Horas demandadas - Planejado x Realizado x CH letiva alocável", yaxis_title="Horas")
for _, row in horas_periodo_v52.iterrows():
    fig_v52_horas.add_annotation(
        x=row["periodo_exec"],
        y=max(row["planejado_horas"], row["realizado_horas"], row["ch_letiva_alocavel"]) * 1.08,
        text=(
            f"Delta mapa-CH: {int(row['delta_planejado_mapa_menos_ch_alocavel'])}h<br>"
            f"Delta registro-CH: {int(row['delta_realizado_registro_menos_ch_alocavel'])}h"
        ),
        showarrow=False,
        font=dict(size=11, color="#0F172A"),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#CBD5E1",
    )

dia_long_v52 = dia_realizado_v52.melt(id_vars=["dia_semana","docentes_por_cargo","qtd_rpa_dia"], value_vars=["qtd_disciplinas","qtd_docentes"], var_name="indicador", value_name="quantidade")
dia_long_v52["indicador"] = dia_long_v52["indicador"].map({"qtd_disciplinas":"Disciplinas realizadas", "qtd_docentes":"Docentes alocados"})
fig_v52_dia = px.bar(
    dia_long_v52, x="dia_semana", y="quantidade", color="indicador", barmode="group",
    custom_data=["docentes_por_cargo", "qtd_rpa_dia"],
    color_discrete_map={"Disciplinas realizadas": COLORS["planejado"], "Docentes alocados": COLORS["realizado"]},
    labels={"dia_semana":"Dia da semana","quantidade":"Quantidade","indicador":"Indicador"}
)
fig_v52_dia.update_traces(hovertemplate="%{x}<br>%{fullData.name}: %{y}<br><br>Docentes por cargo<br>%{customdata[0]}<br><br>Quantidade de RPA no dia: %{customdata[1]}<extra></extra>")
fig_v52_dia.update_layout(title="Realizado - disciplinas e docentes por dia da semana")

fig_v52_rpa_valor = go.Figure()
fig_v52_rpa_valor.add_bar(x=["Valor registrado", "Estimativa sem valor", "Total estimado"], y=[valor_rpa_registro_total_v52, rpa_estimado_sem_valor_v52, rpa_total_estimado_v52], marker_color=[COLORS["realizado"], COLORS["alerta"], COLORS["rpa"]], text=[valor_rpa_registro_total_v52, rpa_estimado_sem_valor_v52, rpa_total_estimado_v52])
fig_v52_rpa_valor.update_traces(texttemplate="R$ %{text:,.0f}", textposition="outside")
fig_v52_rpa_valor.update_layout(title="RPA - valor registrado e simulação para itens sem valor", yaxis_title="R$")

rpa_horas_plot = rpa_horas_v52.melt(id_vars=["periodo_exec"], value_vars=["rpa_planejado_horas","rpa_realizado_horas"], var_name="origem", value_name="horas")
rpa_horas_plot["origem"] = rpa_horas_plot["origem"].map({"rpa_planejado_horas":"Planejado", "rpa_realizado_horas":"Realizado"})
fig_v52_rpa_horas = px.bar(rpa_horas_plot, x="periodo_exec", y="horas", color="origem", barmode="group", text="horas", color_discrete_map={"Planejado":COLORS["planejado"],"Realizado":COLORS["realizado"]}, labels={"periodo_exec":"Bloco","horas":"Horas RPA","origem":"Origem"})
fig_v52_rpa_horas.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
fig_v52_rpa_horas.update_layout(title="RPA - horas demandadas por bloco")

fig_v52_top_rpa_cursos = px.bar(top_cursos_rpa_v52.head(15).sort_values("horas"), x="horas", y="curso_nome", orientation="h", text="horas", color_discrete_sequence=[COLORS["rpa"]], labels={"curso_nome":"Curso","horas":"Horas RPA no Mapa"}) if not top_cursos_rpa_v52.empty else go.Figure()
fig_v52_top_rpa_cursos.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
fig_v52_top_rpa_cursos.update_layout(title="Top cursos com maior demanda RPA sinalizada no Mapa")

fig_v52_doc_alertas = px.bar(
    doc_cargo_v52.melt(id_vars=["cargo"], value_vars=["docentes_acima_limite","docentes_sem_alocacao"], var_name="indicador", value_name="docentes"),
    x="cargo", y="docentes", color="indicador", barmode="group", text="docentes",
    color_discrete_map={"docentes_acima_limite": COLORS["gap"], "docentes_sem_alocacao": COLORS["neutro"]},
    labels={"cargo":"Cargo","docentes":"Docentes","indicador":"Indicador"}
)
fig_v52_doc_alertas.update_layout(title="Docentes - acima do limite e sem disciplina por cargo")
fig_v52_doc_alertas.update_xaxes(tickangle=-35)

fig_v52_cluster = px.bar(
    cluster_v52.melt(id_vars=["periodo_exec","cluster"], value_vars=["horas_planejado","horas_realizado"], var_name="origem", value_name="horas"),
    x="cluster", y="horas", color="origem", facet_col="periodo_exec", barmode="group",
    color_discrete_map={"horas_planejado": COLORS["planejado"], "horas_realizado": COLORS["realizado"]},
    labels={"cluster":"Cluster","horas":"Horas semanais","origem":"Origem","periodo_exec":"Bloco"}
)
fig_v52_cluster.update_layout(title="Demanda semanal por cluster - Planejado x Realizado")
fig_v52_cluster.update_xaxes(tickangle=-35)

_heat_plan_vmin_v52, _heat_plan_vmax_v52 = limites_heatmap(heat_plan_v52.values if isinstance(heat_plan_v52, pd.DataFrame) and not heat_plan_v52.empty else [0])
fig_v52_heat_plan = px.imshow(heat_plan_v52, text_auto=True, aspect="auto", color_continuous_scale=escala_heatmap_gamma(HEATMAP_PALETAS["planejado"]), zmin=_heat_plan_vmin_v52, zmax=_heat_plan_vmax_v52, title="Previsto - disciplinas por dia/horário")
fig_v52_heat_plan.update_traces(textfont=dict(color="#0F172A"))
_heat_real_vmin_v52, _heat_real_vmax_v52 = limites_heatmap(heat_real_v52.values if isinstance(heat_real_v52, pd.DataFrame) and not heat_real_v52.empty else [0])
fig_v52_heat_real = px.imshow(heat_real_v52, text_auto=True, aspect="auto", color_continuous_scale=escala_heatmap_gamma(HEATMAP_PALETAS["realizado"]), zmin=_heat_real_vmin_v52, zmax=_heat_real_vmax_v52, title="Realizado - disciplinas por dia/horário")
fig_v52_heat_real.update_traces(textfont=dict(color="#0F172A"))

# ---------------------------
# Exportar abas v52 ao Excel existente
# ---------------------------
# Reaproveita o mesmo Excel consolidado gerado anteriormente (evita duplicar arquivo de saída).
excel_v52_out = excel_out
try:
    with pd.ExcelWriter(excel_v52_out, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        ordem_comp_v52.to_excel(writer, sheet_name="V52 Ordem PR", index=False)
        horas_periodo_v52.to_excel(writer, sheet_name="V52 Horas Periodo", index=False)
        rpa_ordem_comp_v52.to_excel(writer, sheet_name="V52 RPA Ordem", index=False)
        rpa_horas_v52.to_excel(writer, sheet_name="V52 RPA Horas", index=False)
        rpa_simulacao_v52.to_excel(writer, sheet_name="V52 RPA Simulacao", index=False)
        rpa_pendente_v52.to_excel(writer, sheet_name="V52 RPA Pendente", index=False)
        top_cursos_rpa_v52.to_excel(writer, sheet_name="V52 Top Cursos RPA", index=False)
        dia_realizado_v52.to_excel(writer, sheet_name="V52 Realizado Dia", index=False)
        doc_conf_v52.to_excel(writer, sheet_name="V52 Conflitos Doc", index=False)
        doc_cargo_total_v52.to_excel(writer, sheet_name="V52 Doc Cargo Geral", index=False)
        cluster_v52.to_excel(writer, sheet_name="V52 Cluster PR", index=False)
        pd.DataFrame([{
            "docentes_ativos_base": qtd_docentes_ativos_base_v52,
            "docentes_considerados_modulo": total_docentes_banco_unicos,
            "ch_letiva_alocavel_por_bloco": capacidade_periodo_total,
            "regra": "Docentes ativos da base + docentes alocados no módulo; Registro integral sem exclusão automática de duplicidades."
        }]).to_excel(writer, sheet_name="V52 Resumo Docentes", index=False)
        heat_plan_v52.reset_index().to_excel(writer, sheet_name="V52 Heat Plan", index=False)
        heat_real_v52.reset_index().to_excel(writer, sheet_name="V52 Heat Real", index=False)
except Exception as exc:
    print(f"Falha ao anexar abas analíticas ao Excel consolidado: {exc}")

# ---------------------------
# HTML executivo v52
# ---------------------------
def plotly_div(fig):
    return fig.to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": True, "responsive": True})

COL_LABELS_V52 = {
    "ordem": "Ordem",
    "planejado": "Planejado",
    "realizado": "Executado",
    "delta": "Delta",
    "periodo_exec": "Bloco executivo",
    "planejado_horas": "Horas planejadas",
    "realizado_horas": "Horas executadas",
    "delta_realizado_menos_planejado": "Delta executado - planejado",
    "delta_realizado_planejado": "Delta executado - planejado",
    "ch_letiva_alocavel": "CH letiva alocável",
    "delta_planejado_mapa_menos_ch_alocavel": "Delta mapa - CH alocável",
    "delta_realizado_registro_menos_ch_alocavel": "Delta registro - CH alocável",
    "rpa_planejado": "RPA planejado",
    "rpa_realizado": "RPA executado",
    "rpa_planejado_horas": "Horas RPA planejadas",
    "rpa_realizado_horas": "Horas RPA executadas",
    "delta_realizado_menos_planejado_horas": "Delta RPA executado - planejado",
    "delta_horas": "Delta horas",
    "cargo": "Cargo",
    "headcount": "Headcount",
    "ch_contratada": "CH contratada",
    "ch_letiva": "CH letiva",
    "horas_1_ordem_estendida": "Horas 1ª ordem + ESTENDIDA",
    "horas_2_ordem_estendida": "Horas 2ª ordem + ESTENDIDA",
    "docentes_acima_limite": "Docentes acima do limite",
    "docentes_sem_alocacao": "Docentes sem alocação",
    "utilizacao_1_ordem_estendida_%": "% utilizado 1ª ordem + ESTENDIDA",
    "utilizacao_2_ordem_estendida_%": "% utilizado 2ª ordem + ESTENDIDA",
    "dia_semana": "Dia da semana",
    "qtd_disciplinas": "Qtd. disciplinas",
    "qtd_docentes": "Qtd. docentes ajustado",
    "qtd_docentes_sem_rpa": "Docentes sem RPA",
    "qtd_rpa_dia": "Qtd. RPA no dia",
    "docentes_por_cargo": "Docentes por cargo",
    "professor_realizado": "Professor",
    "qtd_linhas": "Qtd. linhas",
    "disciplinas": "Disciplinas",
    "ordens": "Ordens",
    "horario": "Horário",
    "regra": "Regra",
    "curso_nome": "Curso",
    "disciplina": "Disciplina",
    "valor_rpa_mapa": "Valor RPA mapa",
}

def fmt_num_v52(v, col=""):
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    col_l = str(col).lower()
    if isinstance(v, (int, float, np.integer, np.floating)):
        if "%" in col_l or "percent" in col_l or "aderencia" in col_l or "utilizacao" in col_l or "cobertura" in col_l:
            return f"{float(v):,.2f}%".replace(",", "X").replace(".", ",").replace("X", ".")
        if "r$" in col_l or "valor_rpa" in col_l or "valor registrado" in col_l or "media" in col_l or "média" in col_l or "estimativa" in col_l:
            return br_money(float(v))
        is_hour_col = (
            "hora" in col_l or
            col_l.endswith("_horas") or
            "ch_" in col_l or
            "ch letiva" in col_l or
            "alocavel" in col_l or
            "alocável" in col_l
        )
        if is_hour_col:
            return f"{float(v):,.0f}h".replace(",", ".")
        if abs(float(v) - int(float(v))) < 1e-9:
            return f"{int(float(v)):,}".replace(",", ".")
        return f"{float(v):,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return str(v)

def format_df_v52(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(lambda v, col=c: fmt_num_v52(v, col))
    d = d.rename(columns={c: COL_LABELS_V52.get(c, c.replace("_", " ").title()) for c in d.columns})
    return d

def df_html_v52(df, max_rows=20):
    if df is None or df.empty:
        return "<p class='muted'>Sem registros.</p>"
    return format_df_v52(df.head(max_rows)).to_html(index=False, classes="tbl", border=0)

def v52_get(df, filtro_col, filtro_val, valor_col):
    try:
        if df is None or df.empty:
            return 0
        d = df[df[filtro_col].astype(str).eq(str(filtro_val))]
        if d.empty or valor_col not in d.columns:
            return 0
        return float(pd.to_numeric(d[valor_col], errors="coerce").fillna(0).sum())
    except Exception:
        return 0

def kpi_card_v52(label, value, note="", color="#2563EB"):
    return f"""
    <div class="card" style="border-left-color:{color}">
      <div class="label">{label}</div>
      <div class="value">{value}</div>
      <div class="muted">{note}</div>
    </div>
    """

kpis_v52_html = "".join([
    kpi_card_v52("Disciplinas previstas realizadas", br_pct(conformidade_pct), f"{br_int(qtd_conforme)} de {br_int(qtd_planejado)} disciplinas previstas", COLORS["planejado"]),
    kpi_card_v52("Planejado 1ª ordem", br_int(v52_get(ordem_comp_v52, "ordem", "1ª", "planejado")), "Disciplinas no Mapa", ORDER_COLORS["1ª"]),
    kpi_card_v52("Planejado 2ª ordem", br_int(v52_get(ordem_comp_v52, "ordem", "2ª", "planejado")), "Disciplinas no Mapa", ORDER_COLORS["2ª"]),
    kpi_card_v52("Planejado ESTENDIDA", br_int(v52_get(ordem_comp_v52, "ordem", "ESTENDIDA", "planejado")), "Disciplinas no Mapa", ORDER_COLORS["ESTENDIDA"]),
    kpi_card_v52("Executado 1ª ordem", br_int(v52_get(ordem_comp_v52, "ordem", "1ª", "realizado")), "Disciplinas no Registro", COLORS["realizado"]),
    kpi_card_v52("Executado 2ª ordem", br_int(v52_get(ordem_comp_v52, "ordem", "2ª", "realizado")), "Disciplinas no Registro", COLORS["realizado"]),
    kpi_card_v52("Executado ESTENDIDA", br_int(v52_get(ordem_comp_v52, "ordem", "ESTENDIDA", "realizado")), "Disciplinas no Registro", COLORS["realizado"]),
    kpi_card_v52("CH letiva alocável", f"{br_int(capacidade_periodo_total)}h", "Capacidade por bloco operacional", COLORS["horas"]),

    kpi_card_v52("Mapa — 1ª ordem + ESTENDIDA", f"{br_int(v52_get(horas_periodo_v52, 'periodo_exec', '1ª ordem + ESTENDIDA', 'planejado_horas'))}h", "Horas demandadas planejadas", COLORS["planejado"]),
    kpi_card_v52("Registro — 1ª ordem + ESTENDIDA", f"{br_int(v52_get(horas_periodo_v52, 'periodo_exec', '1ª ordem + ESTENDIDA', 'realizado_horas'))}h", "Horas demandadas executadas", COLORS["realizado"]),
    kpi_card_v52("Delta mapa - CH — 1ª + EST.", f"{br_int(v52_get(horas_periodo_v52, 'periodo_exec', '1ª ordem + ESTENDIDA', 'delta_planejado_mapa_menos_ch_alocavel'))}h", "Planejado - CH alocável", COLORS["alerta"]),
    kpi_card_v52("Delta registro - CH — 1ª + EST.", f"{br_int(v52_get(horas_periodo_v52, 'periodo_exec', '1ª ordem + ESTENDIDA', 'delta_realizado_registro_menos_ch_alocavel'))}h", "Executado - CH alocável", COLORS["alerta"]),

    kpi_card_v52("Mapa — 2ª ordem + ESTENDIDA", f"{br_int(v52_get(horas_periodo_v52, 'periodo_exec', '2ª ordem + ESTENDIDA', 'planejado_horas'))}h", "Horas demandadas planejadas", COLORS["planejado"]),
    kpi_card_v52("Registro — 2ª ordem + ESTENDIDA", f"{br_int(v52_get(horas_periodo_v52, 'periodo_exec', '2ª ordem + ESTENDIDA', 'realizado_horas'))}h", "Horas demandadas executadas", COLORS["realizado"]),
    kpi_card_v52("Delta mapa - CH — 2ª + EST.", f"{br_int(v52_get(horas_periodo_v52, 'periodo_exec', '2ª ordem + ESTENDIDA', 'delta_planejado_mapa_menos_ch_alocavel'))}h", "Planejado - CH alocável", COLORS["alerta"]),
    kpi_card_v52("Delta registro - CH — 2ª + EST.", f"{br_int(v52_get(horas_periodo_v52, 'periodo_exec', '2ª ordem + ESTENDIDA', 'delta_realizado_registro_menos_ch_alocavel'))}h", "Executado - CH alocável", COLORS["alerta"]),

    kpi_card_v52("RPA planejado", br_int(rpa_ordem_comp_v52["rpa_planejado"].sum() if not rpa_ordem_comp_v52.empty else 0), "Disciplinas RPA no Mapa", COLORS["rpa"]),
    kpi_card_v52("RPA executado", br_int(rpa_ordem_comp_v52["rpa_realizado"].sum() if not rpa_ordem_comp_v52.empty else 0), "Disciplinas RPA no Registro", COLORS["rpa"]),
    kpi_card_v52("RPA registrado", br_money(valor_rpa_registro_total_v52), "Valor preenchido no Registro", COLORS["rpa"]),
    kpi_card_v52("RPA estimado total", br_money(rpa_total_estimado_v52), "Registrado + simulação sem valor", COLORS["rpa"]),
    kpi_card_v52("Docentes ativos", br_int(qtd_docentes_ativos_considerados_v53), "Ativos na base, dentro do escopo docente", COLORS["ok"]),
    kpi_card_v52("Docentes considerados", br_int(total_docentes_banco_unicos), "EAD, presencial e regentes | RPA à parte", COLORS["ok"]),
    kpi_card_v52("Conflitos docentes", br_int(len(doc_conf_v52)), "Mesmo docente/bloco/dia/horário", COLORS["gap"]),
])


html_v52 = f"""
<!DOCTYPE html>
<html lang="pt-br">
<head>
<meta charset="UTF-8">
<title>SGA - Relatório Executivo Módulo {modulo_alvo}</title>
<script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>
<style>
body{{font-family:Inter,Arial,sans-serif;background:#F8FAFC;color:#0F172A;margin:0}}
.hero{{background:linear-gradient(135deg,#0F172A,#1E3A8A);color:white;padding:34px 46px}}
.hero h1{{margin:0;font-size:34px}}
.container{{max-width:1320px;margin:0 auto;padding:28px 26px 70px}}
.section{{background:white;border:1px solid #E2E8F0;border-radius:18px;padding:22px;margin:22px 0;box-shadow:0 8px 24px rgba(15,23,42,.06)}}
.grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:14px}}
.card{{background:white;border:1px solid #E2E8F0;border-radius:16px;padding:18px;box-shadow:0 8px 22px rgba(15,23,42,.05);border-left:6px solid #2563EB}}
.label{{font-size:12px;text-transform:uppercase;color:#64748B;font-weight:800;letter-spacing:.04em}}
.value{{font-size:27px;font-weight:900;margin-top:8px;letter-spacing:-.02em}}
.muted{{color:#64748B}}
.tbl{{border-collapse:collapse;width:100%;font-size:13px}}
.tbl th{{background:#0F172A;color:white;padding:8px;text-align:left}}
.tbl td{{border-bottom:1px solid #E2E8F0;padding:7px;vertical-align:top}}
</style>
</head>
<body>
<div class="hero">
<h1>Relatório: Planejado versus Executado do Módulo {modulo_alvo} {RELATORIO_ORGANIZACAO}</h1>\n<p>Elaborado por: {RELATORIO_AUTOR_NOME} — {RELATORIO_AUTOR_CARGO} | Atualizado em: {carimbo_atualizacao("%d/%m/%Y %H:%M")}</p>
<p>Planejado x Realizado | 1ª ordem + ESTENDIDA | 2ª ordem + ESTENDIDA | Registro integral considerado nos cálculos</p>
</div>
<div class="container">
<div class="grid kpi-grid">
{kpis_v52_html}
</div>

<div class="section"><h2>Resumo geral</h2><p>{html_escape(resumo_texto.strip()).replace(chr(10), '<br>')}</p><p>Regra v52: as duplicidades do Registro são sinalizadas em governança, mas todas as linhas válidas entram nos cálculos para refletir a informação registrada.</p></div>

<div class="section"><h2>1. Horas demandadas - Planejado x Realizado x CH alocável</h2>{plotly_div(fig_v52_horas)}{df_html_v52(horas_periodo_v52)}</div>
<div class="section"><h2>2. Realizado - disciplinas e docentes por dia</h2>{plotly_div(fig_v52_dia)}{df_html_v52(dia_realizado_v52)}</div>
<div class="section"><h2>3. RPA - financeiro, horas e pendências</h2>{plotly_div(fig_v52_rpa_valor)}{plotly_div(fig_v52_rpa_horas)}{plotly_div(fig_v52_top_rpa_cursos)}<h3>RPA sinalizado no Mapa sem efetivação no Registro</h3>{df_html_v52(rpa_pendente_v52, 30)}</div>
<div class="section"><h2>4. Docentes - capacidade, utilização e alertas</h2>{df_html_v52(doc_cargo_total_v52, 30)}<p class="muted">Critério das colunas Acima limite e Sem alocação: Acima limite conta docentes cuja alocação registrada supera a CH letiva alocável em pelo menos um bloco executivo; Sem alocação conta docentes considerados na base de capacidade com 0h alocadas no Registro nos dois blocos. Para RPA, o headcount representa contratações/alocações e a leitura completa fica na seção RPA.</p>{plotly_div(fig_v52_doc_alertas)}<h3>Conflitos impossíveis de agenda docente</h3>{df_html_v52(doc_conf_v52, 40)}</div>
<div class="section"><h2>5. Macro - heatmaps por dia e horário</h2>{plotly_div(fig_v52_heat_plan)}{plotly_div(fig_v52_heat_real)}</div>
</div>
</body>
</html>
"""
html_v52_out = OUTPUT_DIR / f"SGA_Dashboard_Executivo_Modulo_{modulo_alvo}_{timestamp}.html"
html_v52_out.write_text(html_v52, encoding="utf-8")

# ---------------------------
# PDF v52 - estrutura solicitada
# ---------------------------
def gerar_pdf_v52(pdf_path: Path):
    import matplotlib.pyplot as plt
    from reportlab.lib import colors as rl_colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, Image as RLImage

    assets = OUTPUT_DIR / f"pdf_v52_assets_{timestamp}"
    assets.mkdir(exist_ok=True)

    def save_bar_horas(path):
        d = horas_periodo_v52.copy()
        fig, ax = plt.subplots(figsize=(10,4.4))
        x = np.arange(len(d)); w=.26
        ax.bar(x-w, d["planejado_horas"], w, color=COLORS["planejado"], label="Planejado")
        ax.bar(x, d["realizado_horas"], w, color=COLORS["realizado"], label="Realizado")
        ax.bar(x+w, d["ch_letiva_alocavel"], w, color=COLORS["horas"], label="CH alocável")
        ax.set_xticks(x); ax.set_xticklabels(d["periodo_exec"])
        ax.set_ylabel("Horas")
        ax.set_title("Horas demandadas - Planejado x Realizado x CH alocável")
        ax.grid(axis="y", alpha=.25); ax.legend(frameon=False)
        for i,row in d.iterrows():
            for off, val in [(-w,row["planejado_horas"]),(0,row["realizado_horas"]),(w,row["ch_letiva_alocavel"])]:
                ax.text(i+off, val, f"{int(val)}h", ha="center", va="bottom", fontsize=8)
        fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)

    def save_bar_dia(path):
        d = dia_realizado_v52.copy()
        fig, ax = plt.subplots(figsize=(10.5,4.4))
        x = np.arange(len(d)); w=.34
        bars1 = ax.bar(x-w/2, d["qtd_disciplinas"], w, color=COLORS["planejado"], label="Disciplinas")
        bars2 = ax.bar(x+w/2, d["qtd_docentes"], w, color=COLORS["realizado"], label="Docentes/RPA ajustado")
        ax.set_xticks(x); ax.set_xticklabels(d["dia_semana"].astype(str), rotation=0)
        ax.set_title("Realizado - disciplinas e docentes por dia")
        ax.grid(axis="y", alpha=.25); ax.legend(frameon=False)
        max_val = max(float(d[["qtd_disciplinas","qtd_docentes"]].max().max()), 1)
        ax.set_ylim(0, max_val * 1.18)
        for bars in [bars1, bars2]:
            for b in bars:
                val = b.get_height()
                ax.text(b.get_x() + b.get_width()/2, val, f"{int(val)}", ha="center", va="bottom", fontsize=9)
        fig.tight_layout(); fig.savefig(path, dpi=170, bbox_inches="tight"); plt.close(fig)

    def save_bar_rpa(path):
        d = rpa_horas_v52.copy()
        fig, ax = plt.subplots(figsize=(10,4.0))
        x=np.arange(len(d)); w=.34
        bars1 = ax.bar(x-w/2, d["rpa_planejado_horas"], w, color=COLORS["planejado"], label="Planejado")
        bars2 = ax.bar(x+w/2, d["rpa_realizado_horas"], w, color=COLORS["realizado"], label="Realizado")
        ax.set_xticks(x); ax.set_xticklabels(d["periodo_exec"])
        ax.set_title("RPA - horas demandadas")
        ax.grid(axis="y", alpha=.25); ax.legend(frameon=False)
        for bars in [bars1, bars2]:
            for b in bars:
                val = b.get_height()
                ax.text(b.get_x() + b.get_width()/2, val, f"{int(val)}h", ha="center", va="bottom", fontsize=8)
        ax.set_ylim(0, max(d[["rpa_planejado_horas","rpa_realizado_horas"]].max().max() * 1.18, 1))
        fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)

    def save_rpa_valor(path):
        valores = [valor_rpa_registro_total_v52, rpa_estimado_sem_valor_v52, rpa_total_estimado_v52]
        labels = ["Valor registrado", "Estimativa sem valor", "Total estimado"]
        fig, ax = plt.subplots(figsize=(10,4.0))
        bars = ax.bar(labels, valores, color=[COLORS["realizado"], COLORS["alerta"], COLORS["rpa"]])
        ax.set_title("RPA - valor registrado e simulação financeira")
        ax.set_ylabel("R$")
        ax.grid(axis="y", alpha=.25)
        for b, val in zip(bars, valores):
            ax.text(b.get_x() + b.get_width()/2, val, br_money(val), ha="center", va="bottom", fontsize=8)
        ax.set_ylim(0, max(max(valores) * 1.22, 1))
        fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)

    def save_doc_alertas(path):
        d = doc_cargo_v52.copy()
        fig, ax = plt.subplots(figsize=(10,4.3))
        x=np.arange(len(d)); w=.34
        ax.bar(x-w/2, d["docentes_acima_limite"], w, color=COLORS["gap"], label="Acima do limite")
        ax.bar(x+w/2, d["docentes_sem_alocacao"], w, color=COLORS["neutro"], label="Sem disciplina")
        ax.set_xticks(x); ax.set_xticklabels(d["cargo"].astype(str), rotation=25, ha="right", fontsize=8)
        ax.set_title("Docentes - alertas por cargo")
        ax.grid(axis="y", alpha=.25); ax.legend(frameon=False)
        fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)

    def save_top_rpa_cursos(path):
        d = top_cursos_rpa_v52.head(12).sort_values("horas") if isinstance(top_cursos_rpa_v52, pd.DataFrame) and not top_cursos_rpa_v52.empty else pd.DataFrame()
        fig, ax = plt.subplots(figsize=(10,5.0))
        if d.empty:
            ax.text(.5,.5,"Sem dados de RPA por curso",ha="center",va="center")
            ax.axis("off")
        else:
            ax.barh(d["curso_nome"], d["horas"], color=COLORS["rpa"])
            ax.set_xlabel("Horas RPA")
            ax.set_title("Top cursos com maior demanda RPA")
            for i, v in enumerate(d["horas"]):
                ax.text(v, i, f" {int(v)}h", va="center", fontsize=8)
        fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)

    def save_cluster_bar(path):
        d = cluster_v52.copy() if isinstance(cluster_v52, pd.DataFrame) else pd.DataFrame()
        fig, ax = plt.subplots(figsize=(11,5.0))
        if d.empty:
            ax.text(.5,.5,"Sem dados de cluster",ha="center",va="center")
            ax.axis("off")
        else:
            pivot = d.pivot_table(index="cluster", columns="periodo_exec", values="horas_planejado", aggfunc="sum", fill_value=0)
            pivot = pivot.sort_values(pivot.columns[0] if len(pivot.columns) else "cluster", ascending=True)
            pivot.plot(kind="barh", ax=ax, color=[ORDER_COLORS["1ª"], ORDER_COLORS["2ª"]][:len(pivot.columns)])
            ax.set_xlabel("Horas semanais planejadas")
            ax.set_ylabel("Cluster")
            ax.set_title("Horas demandadas na semana por cluster")
            ax.grid(axis="x", alpha=.25)
            ax.legend(title="Bloco")
        fig.tight_layout(); fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig)

    def save_heatmap_df_v52(df, path, title, palette="planejado"):
        """Mapa de calor com realce de potência (gamma) e limites robustos.

        Antes a escala era linear sobre uma paleta clara: com dados de amplitude
        pequena, todas as células saíam praticamente no mesmo tom. Agora:
          - células vazias (0) ficam brancas e não entram no cálculo da escala;
          - vmin/vmax vêm dos percentis dos valores > 0 (limites_heatmap);
          - PowerNorm(gamma < 1) distribui as cores de forma não linear, abrindo
            o contraste justamente na faixa onde os dados se concentram;
          - o rótulo troca para branco nas células escuras, mantendo leitura.
        """
        from matplotlib.colors import LinearSegmentedColormap, PowerNorm
        fig, ax = plt.subplots(figsize=(11,4.9))
        if df is None or df.empty:
            ax.text(.5,.5,"Sem dados para heatmap",ha="center",va="center")
            ax.axis("off")
        else:
            d = df.copy()
            arr = d.values.astype(float)
            cores = HEATMAP_PALETAS.get(palette, HEATMAP_PALETAS["planejado"])
            cmap = LinearSegmentedColormap.from_list(f"siap_{palette}_gamma", cores)
            cmap.set_bad("#FFFFFF")
            vmin, vmax = limites_heatmap(arr)
            norm = PowerNorm(gamma=HEATMAP_GAMMA, vmin=vmin, vmax=vmax, clip=True)
            arr_plot = np.ma.masked_where(arr <= 0, arr)
            im = ax.imshow(arr_plot, aspect="auto", cmap=cmap, norm=norm)
            ax.set_xticks(np.arange(len(d.columns))); ax.set_xticklabels(d.columns, rotation=0, fontsize=9)
            ax.set_yticks(np.arange(len(d.index))); ax.set_yticklabels(d.index, fontsize=8)
            ax.set_title(title, fontsize=12, fontweight="bold", color="#0F172A")
            # Grade branca separando as células: facilita a leitura célula a célula.
            ax.set_xticks(np.arange(-.5, arr.shape[1], 1), minor=True)
            ax.set_yticks(np.arange(-.5, arr.shape[0], 1), minor=True)
            ax.grid(which="minor", color="white", linewidth=1.1)
            ax.tick_params(which="minor", length=0)
            for spine in ax.spines.values():
                spine.set_edgecolor("#CBD5E1")
            for i in range(arr.shape[0]):
                for j in range(arr.shape[1]):
                    val = arr[i, j]
                    if val <= 0:
                        cor_texto = "#94A3B8"
                    else:
                        intensidade = float(norm(min(max(val, vmin), vmax)))
                        cor_texto = "#FFFFFF" if intensidade >= 0.60 else "#0F172A"
                    ax.text(
                        j, i, f"{int(val)}",
                        ha="center", va="center",
                        fontsize=8.6,
                        color=cor_texto,
                        fontweight="bold",
                    )
            ticks = np.unique(np.linspace(vmin, vmax, 5).round(0))
            cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, ticks=ticks)
            cbar.ax.tick_params(labelsize=8)
            cbar.set_label("Disciplinas", fontsize=8, color="#334155")
            cbar.outline.set_edgecolor("#CBD5E1")
        fig.tight_layout(); fig.savefig(path, dpi=170, bbox_inches="tight"); plt.close(fig)

    chart1 = assets / "horas.png"; save_bar_horas(chart1)
    chart2 = assets / "dia.png"; save_bar_dia(chart2)
    chart3 = assets / "rpa.png"; save_bar_rpa(chart3)
    chart5 = assets / "top_rpa_cursos.png"; save_top_rpa_cursos(chart5)
    chart9 = assets / "rpa_valor.png"; save_rpa_valor(chart9)
    chart7 = assets / "heat_plan.png"; save_heatmap_df_v52(heat_plan_v52, chart7, "Disciplinas previstas por dia e horário", palette="planejado")
    chart8 = assets / "heat_real.png"; save_heatmap_df_v52(heat_real_v52, chart8, "Disciplinas realizadas por dia e horário", palette="realizado")
    chart10 = assets / "cluster_bar.png"; save_cluster_bar(chart10)

    doc = SimpleDocTemplate(str(pdf_path), pagesize=landscape(A4), rightMargin=1.2*cm, leftMargin=1.2*cm, topMargin=1.65*cm, bottomMargin=1.15*cm)
    page_w_v52, page_h_v52 = landscape(A4)

    def header_footer_v52(canvas, doc_obj):
        """Cabeçalho e rodapé do documento executivo.

        A marca da ferramenta saiu do cabeçalho: o documento é institucional e
        carrega apenas contexto (organização, módulo/ano) no topo e a
        responsabilidade (nome, cargo e data/hora da atualização) no rodapé.
        """
        canvas.saveState()
        canvas.setFillColor(rl_colors.HexColor("#0F172A"))
        canvas.rect(0, page_h_v52 - 22, page_w_v52, 22, fill=1, stroke=0)
        canvas.setFillColor(rl_colors.white)
        canvas.setFont("Helvetica-Bold", 8)
        canvas.drawString(1.2*cm, page_h_v52 - 14.5, f"{RELATORIO_ORGANIZACAO} — Planejado versus Executado")
        canvas.drawRightString(page_w_v52 - 1.2*cm, page_h_v52 - 14.5, f"Módulo {modulo_alvo} | {ano_alvo}")
        canvas.setStrokeColor(rl_colors.HexColor("#E2E8F0"))
        canvas.setLineWidth(0.5)
        canvas.line(1.2*cm, 0.95*cm, page_w_v52 - 1.2*cm, 0.95*cm)
        canvas.setFillColor(rl_colors.HexColor("#475569"))
        canvas.setFont("Helvetica-Bold", 7)
        canvas.drawString(1.2*cm, 0.55*cm, assinatura_relatorio())
        canvas.setFillColor(rl_colors.HexColor("#64748B"))
        canvas.setFont("Helvetica", 7)
        canvas.drawCentredString(page_w_v52 / 2.0, 0.55*cm, f"Atualizado em {carimbo_atualizacao('%d/%m/%Y %H:%M')}")
        canvas.drawRightString(page_w_v52 - 1.2*cm, 0.55*cm, f"Página {doc_obj.page}")
        canvas.restoreState()

    styles = getSampleStyleSheet()
    title = ParagraphStyle("title", parent=styles["Title"], fontSize=24, alignment=TA_CENTER, textColor=rl_colors.HexColor("#0F172A"))
    h2 = ParagraphStyle("h2", parent=styles["Heading2"], fontSize=14, textColor=rl_colors.HexColor("#0F172A"))
    body = ParagraphStyle("body", parent=styles["BodyText"], fontSize=9.5, leading=13)
    small = ParagraphStyle("small", parent=styles["BodyText"], fontSize=8.2, leading=10)

    def P(txt, st=body):
        return Paragraph(str(txt).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;").replace("\n","<br/>"), st)

    def tbl(df, max_rows=12):
        if df is None or df.empty:
            return P("Sem registros.", small)
        d = format_df_v52(df.head(max_rows).copy())
        data = [list(d.columns)] + d.astype(str).values.tolist()
        t = Table(data, repeatRows=1)
        t.setStyle(TableStyle([
            ("BACKGROUND",(0,0),(-1,0),rl_colors.HexColor("#0F172A")),
            ("TEXTCOLOR",(0,0),(-1,0),rl_colors.white),
            ("GRID",(0,0),(-1,-1),0.25,rl_colors.HexColor("#CBD5E1")),
            ("FONTSIZE",(0,0),(-1,-1),7),
            ("VALIGN",(0,0),(-1,-1),"TOP"),
            ("ROWBACKGROUNDS",(0,1),(-1,-1),[rl_colors.white, rl_colors.HexColor("#F8FAFC")]),
        ]))
        return t

    def tbl_compact(df, max_rows=12, font_size=5.8):
        if df is None or df.empty:
            return P("Sem registros.", small)
        d = format_df_v52(df.head(max_rows).copy())
        data = [list(d.columns)] + d.astype(str).values.tolist()
        t = Table(data, repeatRows=1)
        t.setStyle(TableStyle([
            ("BACKGROUND",(0,0),(-1,0),rl_colors.HexColor("#0F172A")),
            ("TEXTCOLOR",(0,0),(-1,0),rl_colors.white),
            ("GRID",(0,0),(-1,-1),0.20,rl_colors.HexColor("#CBD5E1")),
            ("FONTSIZE",(0,0),(-1,-1),font_size),
            ("LEFTPADDING",(0,0),(-1,-1),3),
            ("RIGHTPADDING",(0,0),(-1,-1),3),
            ("TOPPADDING",(0,0),(-1,-1),2),
            ("BOTTOMPADDING",(0,0),(-1,-1),2),
            ("VALIGN",(0,0),(-1,-1),"TOP"),
            ("ROWBACKGROUNDS",(0,1),(-1,-1),[rl_colors.white, rl_colors.HexColor("#F8FAFC")]),
        ]))
        return t

    def tbl_wrapped(df, max_rows=12, font_size=6.2, col_widths=None):
        if df is None or df.empty:
            return P("Sem registros.", small)
        d = format_df_v52(df.head(max_rows).copy())
        ncols = max(1, len(d.columns))
        if col_widths is None:
            col_widths = [26.4*cm/ncols] * ncols

        cell_style = ParagraphStyle(
            "cellWrap",
            parent=small,
            fontSize=font_size,
            leading=font_size + 1.2,
            wordWrap="CJK",
        )
        header_style = ParagraphStyle(
            "headerWrap",
            parent=cell_style,
            textColor=rl_colors.white,
            fontSize=font_size,
            leading=font_size + 1.2,
            alignment=1,
        )

        def _par(v, style):
            txt = str(v).replace("<br>", "\n").replace("<br/>", "\n")
            txt = txt.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br/>")
            return Paragraph(txt, style)

        data = [[_par(c, header_style) for c in d.columns]]
        for _, row in d.iterrows():
            data.append([_par(row[c], cell_style) for c in d.columns])

        t = Table(data, colWidths=col_widths, repeatRows=1)
        t.setStyle(TableStyle([
            ("BACKGROUND",(0,0),(-1,0),rl_colors.HexColor("#0F172A")),
            ("TEXTCOLOR",(0,0),(-1,0),rl_colors.white),
            ("GRID",(0,0),(-1,-1),0.20,rl_colors.HexColor("#CBD5E1")),
            ("FONTSIZE",(0,0),(-1,-1),font_size),
            ("LEFTPADDING",(0,0),(-1,-1),3),
            ("RIGHTPADDING",(0,0),(-1,-1),3),
            ("TOPPADDING",(0,0),(-1,-1),2),
            ("BOTTOMPADDING",(0,0),(-1,-1),2),
            ("VALIGN",(0,0),(-1,-1),"TOP"),
            ("ROWBACKGROUNDS",(0,1),(-1,-1),[rl_colors.white, rl_colors.HexColor("#F8FAFC")]),
        ]))
        return t

    def box_title(txt):
        return Paragraph(f"{str(txt).replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')}", ParagraphStyle("boxTitle", parent=small, fontSize=8.5, leading=10, textColor=rl_colors.HexColor("#0F172A")))

    def metric_card_v52(label, value, note, color_hex):
        accent = Table([[""]], colWidths=[4], rowHeights=[44])
        accent.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,-1), rl_colors.HexColor(color_hex))]))
        content = Table([
            [Paragraph(str(label).upper(), ParagraphStyle("cardLabel", parent=small, fontSize=6.8, leading=8.2, textColor=rl_colors.HexColor("#64748B"), fontName="Helvetica-Bold"))],
            [Paragraph(str(value), ParagraphStyle("cardValue", parent=small, fontSize=13.5, leading=16, textColor=rl_colors.HexColor("#0F172A"), fontName="Helvetica-Bold"))],
            [Paragraph(str(note), ParagraphStyle("cardNote", parent=small, fontSize=6.6, leading=8.2, textColor=rl_colors.HexColor("#475569")))],
        ], colWidths=[150], rowHeights=[11, 17, 15])
        content.setStyle(TableStyle([
            ("VALIGN", (0,0), (-1,-1), "TOP"),
            ("LEFTPADDING", (0,0), (-1,-1), 7), ("RIGHTPADDING", (0,0), (-1,-1), 6),
            ("TOPPADDING", (0,0), (-1,-1), 2), ("BOTTOMPADDING", (0,0), (-1,-1), 1),
        ]))
        card = Table([[accent, content]], colWidths=[4, 150], rowHeights=[46])
        card.setStyle(TableStyle([
            ("BOX", (0,0), (-1,-1), 0.6, rl_colors.HexColor("#E2E8F0")),
            ("VALIGN", (0,0), (-1,-1), "TOP"),
            ("LEFTPADDING", (0,0), (-1,-1), 0), ("RIGHTPADDING", (0,0), (-1,-1), 0),
            ("TOPPADDING", (0,0), (-1,-1), 0), ("BOTTOMPADDING", (0,0), (-1,-1), 0),
        ]))
        return card

    def card_grid_v52(cards, per_row=5):
        rows = []
        for i in range(0, len(cards), per_row):
            row_cards = [metric_card_v52(*c) for c in cards[i:i+per_row]]
            while len(row_cards) < per_row:
                row_cards.append("")
            rows.append(row_cards)
        grid = Table(rows, colWidths=[doc.width/per_row - 5] * per_row, hAlign="LEFT")
        grid.setStyle(TableStyle([
            ("VALIGN", (0,0), (-1,-1), "TOP"),
            ("LEFTPADDING", (0,0), (-1,-1), 3), ("RIGHTPADDING", (0,0), (-1,-1), 3),
            ("TOPPADDING", (0,0), (-1,-1), 4), ("BOTTOMPADDING", (0,0), (-1,-1), 6),
        ]))
        return grid

    story = []
    # Página 1: capa executiva com banner colorido, cards de indicadores e leitura textual.
    titulo_relatorio_v52 = f"Planejado versus Executado — Módulo {modulo_alvo} | {RELATORIO_ORGANIZACAO}"
    subtitulo_relatorio_v52 = f"Ano {ano_alvo} | Atualizado em {carimbo_atualizacao('%d/%m/%Y %H:%M')}"
    # Terceira linha da capa: responsabilidade pelo documento (nome, cargo e área).
    responsavel_capa_v52 = f"Responsável: {RELATORIO_AUTOR_NOME} — {RELATORIO_AUTOR_CARGO}"
    if str(RELATORIO_AUTOR_AREA).strip():
        responsavel_capa_v52 += f" | {RELATORIO_AUTOR_AREA}"
    banner = Table(
        [[Paragraph(titulo_relatorio_v52, ParagraphStyle("bannerTitle", parent=title, fontSize=19, leading=23, alignment=TA_CENTER, textColor=rl_colors.white, fontName="Helvetica-Bold"))],
         [Paragraph(subtitulo_relatorio_v52, ParagraphStyle("bannerSub", parent=styles["BodyText"], fontSize=10.5, alignment=TA_CENTER, textColor=rl_colors.HexColor("#DBEAFE"), leading=14))],
         [Paragraph(responsavel_capa_v52, ParagraphStyle("bannerAssin", parent=styles["BodyText"], fontSize=9.5, alignment=TA_CENTER, textColor=rl_colors.HexColor("#94A3B8"), leading=12))]],
        colWidths=[doc.width],
    )
    banner.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,-1), rl_colors.HexColor("#0F172A")),
        ("TOPPADDING", (0,0), (-1,0), 14), ("BOTTOMPADDING", (0,0), (-1,0), 4),
        ("TOPPADDING", (0,1), (-1,1), 2), ("BOTTOMPADDING", (0,1), (-1,1), 2),
        ("TOPPADDING", (0,2), (-1,2), 2), ("BOTTOMPADDING", (0,2), (-1,2), 13),
    ]))
    story.append(banner)
    story.append(Spacer(1, 12))
    story.append(card_grid_v52([
        ("Disciplinas previstas realizadas", br_pct(conformidade_pct), f"{br_int(qtd_conforme)} de {br_int(qtd_planejado)} disciplinas previstas", COLORS["planejado"]),
        ("Delta docente 1ª + EST.", f"{br_int(delta_demanda_p1)}h", "Demanda - CH alocável", COLORS["horas"] if delta_demanda_p1 <= 0 else COLORS["gap"]),
        ("Delta docente 2ª + EST.", f"{br_int(delta_demanda_p2)}h", "Demanda - CH alocável", COLORS["horas"] if delta_demanda_p2 <= 0 else COLORS["gap"]),
        ("RPA total estimado", br_money(rpa_total_estimado_v52), "Registrado + simulação", COLORS["rpa"]),
        ("Docentes considerados", br_int(total_docentes_banco_unicos), f"{br_int(qtd_docentes_ativos_considerados_v53)} ativos | RPA à parte", COLORS["ok"]),
    ], per_row=5))
    story.append(Spacer(1, 16))
    story.append(P("Regras de negócio, insights e conclusões executivas", h2))
    page1_text = f"""
<b>Regras de negócio aplicadas:</b> A auditoria compara o Mapa Pedagógico planejado com o Registro executado, considerando apenas aulas ao vivo e preservando a rastreabilidade entre disciplina, ordem, dia da semana e horário. O Registro é tratado como base operacional de execução: todas as linhas válidas registradas entram nos cálculos, enquanto duplicidades e inconsistências são mantidas em governança para acompanhamento e correção pela área responsável.

<b>Leitura de carga horária:</b> Toda análise de horas demandadas é realizada exclusivamente por dois blocos executivos: 1ª ordem + ESTENDIDA e 2ª ordem + ESTENDIDA. Cada disciplina ativa no bloco representa 2 horas de demanda semanal. As disciplinas de 1ª e 2ª ordem compõem os blocos específicos do módulo, enquanto as disciplinas ESTENDIDAS atravessam os dois blocos e por isso são consideradas nas duas leituras.

<b>Capacidade docente:</b> A capacidade utiliza a base docente atualizada. Docentes ativos entram na composição da CH letiva alocável e docentes que aparecem alocados no Registro também são considerados, mesmo quando ainda não constam como ativos na base, pois representam operação efetivamente realizada. Esses casos permanecem separados para governança cadastral e eventual atualização no RM.

<b>RPA:</b> As disciplinas sinalizadas como RPA são acompanhadas em duas frentes: efetivação operacional no Registro e impacto financeiro. Quando há RPA registrado sem valor preenchido, a estimativa financeira utiliza a média das contratações com valor informado para projetar o montante aproximado ainda não preenchido.

<b>Insights do módulo:</b> Das {br_int(qtd_planejado)} disciplinas previstas para o módulo, {br_int(qtd_conforme)} aconteceram exatamente como previsto ({br_pct(conformidade_pct)} de aderência). {br_int(qtd_faltantes)} disciplinas previstas não foram executadas na forma planejada e {br_int(qtd_extras)} disciplinas foram executadas sem previsão correspondente no Mapa. A demanda planejada contra a CH letiva alocável apresenta delta de {br_int(delta_demanda_p1)}h em 1ª ordem + ESTENDIDA e {br_int(delta_demanda_p2)}h em 2ª ordem + ESTENDIDA, indicando a necessidade de leitura por bloco para avaliar suficiência de capacidade.

<b>Conclusões executivas:</b> O relatório demonstra que a tomada de decisão deve priorizar três pontos: fechamento das pendências entre Mapa e Registro, acompanhamento dos RPAs ainda não efetivados ou sem valor informado, e governança contínua da base docente. A estrutura do relatório permite acompanhar a aderência do executado ao planejado, a pressão de carga horária por bloco e a necessidade de ajustes operacionais antes do próximo ciclo.
"""
    # Page1 usa Paragraph diretamente para permitir negrito nos rótulos dos tópicos.
    story.append(Paragraph(str(page1_text).replace("\n", "<br/>"), body))
    story.append(PageBreak())

    # Página 2: tabelas executivas em uma única página
    story.append(P("Página 2 - Quantitativos executivos", title))
    story.append(Spacer(1, 4))

    def _conf_str_v52(num, den):
        try:
            den = float(den)
            if den == 0:
                return "0,00%"
            return br_pct(float(num) / den * 100)
        except Exception:
            return "0,00%"

    horas_pdf_v52 = horas_periodo_v52[[
        "periodo_exec","planejado_horas","realizado_horas",
        "delta_realizado_menos_planejado","ch_letiva_alocavel",
        "delta_planejado_mapa_menos_ch_alocavel",
        "delta_realizado_registro_menos_ch_alocavel"
    ]].copy()
    horas_pdf_v52["Conformidade"] = horas_pdf_v52.apply(lambda r: _conf_str_v52(r.get("realizado_horas", 0), r.get("planejado_horas", 0)), axis=1)

    rpa_horas_pdf_v52 = rpa_horas_v52.copy()
    if not rpa_horas_pdf_v52.empty:
        rpa_horas_pdf_v52["Conformidade"] = rpa_horas_pdf_v52.apply(lambda r: _conf_str_v52(r.get("rpa_realizado_horas", 0), r.get("rpa_planejado_horas", 0)), axis=1)

    ordem_pdf_v52 = ordem_comp_v52.copy()
    if not ordem_pdf_v52.empty:
        ordem_pdf_v52["Conformidade"] = ordem_pdf_v52.apply(lambda r: _conf_str_v52(r.get("realizado", 0), r.get("planejado", 0)), axis=1)

    rpa_ordem_pdf_v52 = rpa_ordem_comp_v52.copy()
    if not rpa_ordem_pdf_v52.empty:
        rpa_ordem_pdf_v52["Conformidade"] = rpa_ordem_pdf_v52.apply(lambda r: _conf_str_v52(r.get("rpa_realizado", 0), r.get("rpa_planejado", 0)), axis=1)

    docentes_pdf_v52 = pd.DataFrame([
        {"Indicador": "Professores considerados no módulo (EAD, presencial e regentes)", "Valor": br_int(total_docentes_banco_unicos)},
        {"Indicador": "Desses, ativos na base docente", "Valor": br_int(qtd_docentes_ativos_considerados_v53)},
        {"Indicador": "Desses, alocados no Registro sem status ativo na base", "Valor": br_int(docentes_considerados_fora_base_ativa_v53)},
        {"Indicador": "Contratações RPA no Registro (não entram na contagem de docentes)", "Valor": br_int(rpa_headcount_contratado_v52)},
        {"Indicador": "CH letiva alocável por bloco", "Valor": f"{br_int(capacidade_periodo_total)}h"},
    ])

    section_compact = ParagraphStyle("sectionCompact", parent=h2, fontSize=9.8, leading=11, spaceBefore=0, spaceAfter=1, textColor=rl_colors.HexColor("#0F172A"))

    story.append(P("Horas letivas demandadas por bloco executivo", section_compact))
    story.append(tbl_wrapped(horas_pdf_v52, 4, 5.2, col_widths=[4.8*cm,2.9*cm,2.9*cm,3.3*cm,2.8*cm,3.7*cm,3.7*cm,2.7*cm]))
    story.append(Spacer(1, 4))

    story.append(P("RPA — horas demandadas por bloco", section_compact))
    story.append(tbl_wrapped(rpa_horas_pdf_v52, 4, 5.4, col_widths=[5.2*cm,3.1*cm,3.1*cm,3.6*cm,2.7*cm,3.0*cm]))
    story.append(Spacer(1, 4))

    story.append(P("Disciplinas por ordem — previstas x realizadas", section_compact))
    story.append(tbl_wrapped(ordem_pdf_v52, 5, 5.4, col_widths=[4.8*cm,3.5*cm,3.5*cm,3.0*cm,3.6*cm]))
    story.append(Spacer(1, 4))

    story.append(P("RPA — quantidade de disciplinas por ordem", section_compact))
    story.append(tbl_wrapped(rpa_ordem_pdf_v52, 5, 5.4, col_widths=[4.8*cm,3.6*cm,3.6*cm,3.0*cm,3.6*cm]))
    story.append(Spacer(1, 4))

    story.append(P("Docentes e capacidade", section_compact))
    story.append(tbl_wrapped(docentes_pdf_v52, 5, 5.8, col_widths=[9.5*cm,3.8*cm]))
    story.append(Spacer(1, 3))
    story.append(P("Leitura: horas demandadas são lidas sempre por 1ª ordem + ESTENDIDA e 2ª ordem + ESTENDIDA. Conformidade = executado ÷ planejado. Os valores financeiros RPA aparecem no final da seção RPA.", small))
    story.append(PageBreak())

    # Páginas de gráficos
    story.append(P("Detalhamento 1 - Horas demandadas e delta de capacidade", title))
    story.append(RLImage(str(chart1), width=24*cm, height=10.5*cm))
    story.append(tbl(horas_periodo_v52[["periodo_exec","planejado_horas","realizado_horas","ch_letiva_alocavel","delta_planejado_mapa_menos_ch_alocavel","delta_realizado_registro_menos_ch_alocavel"]], 10))
    story.append(PageBreak())

    story.append(P("Detalhamento 1.1 - Demanda por cluster de curso", title))
    story.append(RLImage(str(chart10), width=22.5*cm, height=8.6*cm))
    story.append(Spacer(1, 5))
    cluster_pdf_v52 = cluster_v52.pivot_table(
        index="cluster", columns="periodo_exec", values=["horas_planejado", "horas_realizado"], aggfunc="sum", fill_value=0
    )
    cluster_pdf_v52.columns = [f"Horas {origem.replace('horas_', '')} - {periodo}" for origem, periodo in cluster_pdf_v52.columns]
    cluster_pdf_v52 = cluster_pdf_v52.reset_index().rename(columns={"cluster": "Cluster"}).sort_values("Cluster")
    cluster_total_row_v52 = pd.DataFrame([{
        "Cluster": "TOTAL GERAL",
        **{c: cluster_pdf_v52[c].sum() for c in cluster_pdf_v52.columns if c != "Cluster"},
    }])
    cluster_pdf_v52 = pd.concat([cluster_pdf_v52, cluster_total_row_v52], ignore_index=True)
    cluster_col_widths = [6.2*cm] + [4.85*cm] * (len(cluster_pdf_v52.columns) - 1)
    story.append(tbl_wrapped(cluster_pdf_v52, 20, 6.4, col_widths=cluster_col_widths))
    story.append(Spacer(1, 4))
    story.append(P("Leitura: horas semanais demandadas por cluster de curso, no mesmo critério de blocos executivos usado no restante do relatório (1ª ordem + ESTENDIDA e 2ª ordem + ESTENDIDA). O cluster é atribuído pela disciplina no Mapa: primeiro pela aula exata (disciplina+ordem+dia+horário); se o Registro não bate a chave exata, usa o cluster da mesma disciplina em qualquer outro horário do Mapa. Só cai em \"SEM CORRESPONDÊNCIA NO MAPA\" quando a disciplina realizada não existe no Mapa em nenhuma combinação.", small))
    story.append(PageBreak())

    story.append(P("Detalhamento 2 - Realizado por dia da semana", title))
    story.append(RLImage(str(chart2), width=23.5*cm, height=9.2*cm))
    dia_pdf_v52 = dia_realizado_v52[["dia_semana","qtd_disciplinas","qtd_docentes","qtd_rpa_dia","docentes_por_cargo"]].copy()
    dia_pdf_v52["docentes_por_cargo"] = dia_pdf_v52["docentes_por_cargo"].astype(str).str.replace("<br>", "\n", regex=False)
    story.append(P("Detalhamento complementar do gráfico: para RPA, a contagem representa a quantidade de alocações/disciplinas RPA no dia.", small))
    story.append(tbl_wrapped(dia_pdf_v52, 10, 5.7, col_widths=[2.4*cm,2.2*cm,2.2*cm,2.2*cm,17.0*cm]))
    story.append(PageBreak())

    story.append(P("Detalhamento 3 - RPA sinalizado no Mapa e ainda não efetivado no Registro", title))
    rpa_pendente_pdf_v52 = rpa_pendente_v52[["curso_nome","disciplina","ordem","dia_semana","horario","valor_rpa_mapa"]].copy() if isinstance(rpa_pendente_v52, pd.DataFrame) and not rpa_pendente_v52.empty else pd.DataFrame()

    if isinstance(rpa_pendente_v52, pd.DataFrame) and not rpa_pendente_v52.empty:
        rpa_pendente_resumo_ordem_v52 = (
            rpa_pendente_v52.groupby("ordem", dropna=False).size()
            .reindex(["1ª", "2ª", "ESTENDIDA"]).fillna(0).astype(int)
            .reset_index(name="qtd_rpa_pendente")
        )
        _pend_periodos_v52 = expandir_periodos_operacionais(rpa_pendente_v52, "RPA pendente")
        rpa_pendente_resumo_horas_v52 = (
            _pend_periodos_v52.groupby("periodo_exec", as_index=False)["horas_demandadas"].sum()
            if not _pend_periodos_v52.empty else pd.DataFrame(columns=["periodo_exec","horas_demandadas"])
        )
    else:
        rpa_pendente_resumo_ordem_v52 = pd.DataFrame({"ordem": ["1ª", "2ª", "ESTENDIDA"], "qtd_rpa_pendente": [0,0,0]})
        rpa_pendente_resumo_horas_v52 = pd.DataFrame(columns=["periodo_exec","horas_demandadas"])

    rpa_pendente_resumo_ordem_v52 = rpa_pendente_resumo_ordem_v52.rename(columns={"ordem": "Ordem", "qtd_rpa_pendente": "Qtd. RPA pendente"})
    rpa_pendente_resumo_horas_v52 = rpa_pendente_resumo_horas_v52.rename(columns={"periodo_exec": "Bloco executivo", "horas_demandadas": "Horas demandadas pendentes"})

    resumo_pendente_grid_v52 = Table([
        [
            [box_title("Resumo por ordem"), tbl_wrapped(rpa_pendente_resumo_ordem_v52, 5, 6.2, col_widths=[3.4*cm,4.0*cm])],
            [box_title("Horas demandadas pendentes por bloco"), tbl_wrapped(rpa_pendente_resumo_horas_v52, 4, 6.2, col_widths=[6.4*cm,4.0*cm])],
        ]
    ], colWidths=[10.5*cm, 15.9*cm])
    resumo_pendente_grid_v52.setStyle(TableStyle([
        ("VALIGN",(0,0),(-1,-1),"TOP"),
        ("BOX",(0,0),(-1,-1),0.25,rl_colors.HexColor("#CBD5E1")),
        ("INNERGRID",(0,0),(-1,-1),0.20,rl_colors.HexColor("#E2E8F0")),
        ("LEFTPADDING",(0,0),(-1,-1),5),
        ("RIGHTPADDING",(0,0),(-1,-1),5),
        ("TOPPADDING",(0,0),(-1,-1),4),
        ("BOTTOMPADDING",(0,0),(-1,-1),4),
    ]))
    story.append(resumo_pendente_grid_v52)
    story.append(Spacer(1, 8))
    story.append(P("Lista detalhada das disciplinas/cursos pendentes", h2))
    story.append(tbl_wrapped(rpa_pendente_pdf_v52, 12, 5.3, col_widths=[6.4*cm,8.8*cm,2.0*cm,2.3*cm,2.2*cm,3.2*cm]))
    story.append(PageBreak())

    story.append(P("Detalhamento 3.1 - RPA horas demandadas e top cursos", title))
    story.append(Spacer(1, 4))
    story.append(RLImage(str(chart3), width=23.8*cm, height=6.3*cm))
    story.append(Spacer(1, 4))
    story.append(RLImage(str(chart5), width=23.8*cm, height=7.0*cm))
    story.append(Spacer(1, 4))
    story.append(P("As barras exibem os valores exatos, evitando leitura aproximada pelo eixo. A lista completa permanece exportada no Excel analítico.", small))
    story.append(PageBreak())

    story.append(P("Detalhamento 3.2 - Valores RPA", title))
    rpa_financeiro_pdf_v52 = pd.DataFrame([
        {"Indicador": "RPA com valor", "Valor": br_int(rpa_simulacao_v52.loc[0, "rpa_com_valor"]) if not rpa_simulacao_v52.empty else "0"},
        {"Indicador": "RPA sem valor", "Valor": br_int(rpa_simulacao_v52.loc[0, "rpa_sem_valor"]) if not rpa_simulacao_v52.empty else "0"},
        {"Indicador": "Valor registrado", "Valor": br_money(valor_rpa_registro_total_v52)},
        {"Indicador": "Média por disciplina", "Valor": br_money(media_rpa_por_disciplina_v52)},
        {"Indicador": "Estimativa sem valor", "Valor": br_money(rpa_estimado_sem_valor_v52)},
        {"Indicador": "Total estimado RPA", "Valor": br_money(rpa_total_estimado_v52)},
    ])
    story.append(RLImage(str(chart9), width=23*cm, height=8.0*cm))
    story.append(P(f"Resumo financeiro RPA: valor registrado {br_money(valor_rpa_registro_total_v52)} | média por disciplina {br_money(media_rpa_por_disciplina_v52)} | estimativa sem valor {br_money(rpa_estimado_sem_valor_v52)} | total estimado {br_money(rpa_total_estimado_v52)}.", h2))
    story.append(tbl_wrapped(rpa_financeiro_pdf_v52, 8, 6.2, col_widths=[10.0*cm, 6.0*cm]))
    story.append(PageBreak())

    story.append(P("Detalhamento 4 - Docentes", title))
    doc_pdf_v52 = doc_cargo_total_v52[[
        "cargo", "headcount", "ch_contratada", "ch_letiva_alocavel",
        "utilizacao_1_ordem_estendida_%", "utilizacao_2_ordem_estendida_%",
        "docentes_acima_limite", "docentes_sem_alocacao"
    ]].copy()
    doc_pdf_v52 = doc_pdf_v52.rename(columns={
        "cargo": "Cargo",
        "headcount": "Headcount",
        "ch_contratada": "CH contratada",
        "ch_letiva_alocavel": "CH letiva alocável",
        "utilizacao_1_ordem_estendida_%": "Uso 1ª+EST %",
        "utilizacao_2_ordem_estendida_%": "Uso 2ª+EST %",
        "docentes_acima_limite": "Acima limite",
        "docentes_sem_alocacao": "Sem alocação",
    })
    story.append(P(
        f"Quadro geral dos docentes por cargo. A métrica 'docentes considerados' ({br_int(total_docentes_banco_unicos)} pessoas) soma apenas "
        f"professor de ensino superior EAD, professor de ensino superior presencial e professor regente. "
        f"A linha de RPA traz {br_int(rpa_headcount_contratado_v52)} contratações registradas no módulo e não entra nessa contagem, "
        f"porque representa contratação e não pessoa do quadro. {_v53_diagnostico_headcount()}", small))
    story.append(tbl_wrapped(doc_pdf_v52, 20, 5.2, col_widths=[7.0*cm,2.2*cm,2.8*cm,3.0*cm,3.0*cm,3.0*cm,2.6*cm,2.6*cm]))
    story.append(Spacer(1, 6))
    story.append(P("Critério das duas últimas colunas: Acima limite conta, por cargo, quantos docentes tiveram horas alocadas no Registro acima da própria CH letiva alocável em pelo menos um dos blocos executivos (1ª ordem + ESTENDIDA ou 2ª ordem + ESTENDIDA). Sem alocação conta, por cargo, quantos docentes considerados na base de capacidade ficaram com 0h alocadas no Registro nos dois blocos do módulo. Professor de Ensino Superior Presencial, Tutor Facilitador Online e RPA são mantidos no quadro, mas não entram nas contagens de Acima limite e Sem alocação. Para RPA, o headcount representa a quantidade de contratações/alocações RPA; como RPA não compõe capacidade letiva interna, sua análise operacional e financeira deve ser lida principalmente na seção RPA do relatório.", small))

    if not docentes_presencial_alocados.empty:
        story.append(Spacer(1, 10))
        story.append(P("Detalhamento 4.1 - Docentes Presencial (caso à parte)", ParagraphStyle("h2b", parent=h2, fontSize=13)))
        _total_pres = docentes_presencial_por_tipo.loc[docentes_presencial_por_tipo["tipo_contrato"].eq("TOTAL GERAL")].iloc[0]
        story.append(P(
            f"{br_int(int(docentes_presencial_alocados['professor_key'].nunique()))} docentes de cargo Professor de Ensino Superior Presencial foram "
            f"alocados no Registro deste módulo sem constar na base de capacidade CLT EAD/Regente. CH contratada (semanal) somada: "
            f"{br_int(_total_pres['ch_contratada_semanal_total'])}h. Horas alocadas no Registro: {br_int(_total_pres['horas_alocadas_total'])}h. "
            f"Ainda não há regra validada de CH letiva para presencial, por isso essa coluna não aparece. "
            f"Nada nesta seção entra no delta docente, na capacidade geral ou em qualquer outro total do relatório.", small))
        doc_pres_pdf_v52 = docentes_presencial_por_tipo.rename(columns={
            "tipo_contrato": "Tipo de contrato", "docentes": "Docentes",
            "ch_contratada_semanal_total": "CH contratada (semanal)",
            "horas_p1_total": "Horas 1ª+EST", "horas_p2_total": "Horas 2ª+EST",
            "horas_alocadas_total": "Horas alocadas (total)",
        })[["Tipo de contrato", "Docentes", "CH contratada (semanal)", "Horas 1ª+EST", "Horas 2ª+EST", "Horas alocadas (total)"]]
        story.append(tbl_wrapped(doc_pres_pdf_v52, 10, 6.0, col_widths=[6.5*cm, 2.6*cm, 4.2*cm, 3.4*cm, 3.4*cm, 4.0*cm]))

    story.append(PageBreak())

    story.append(P("Detalhamento 5 - Mapa de calor das disciplinas previstas por dia e horário", title))
    story.append(Spacer(1, 6))
    story.append(RLImage(str(chart7), width=24.2*cm, height=10.0*cm))
    story.append(PageBreak())

    story.append(P("Detalhamento 6 - Mapa de calor das disciplinas realizadas por dia e horário", title))
    story.append(Spacer(1, 6))
    story.append(RLImage(str(chart8), width=24.2*cm, height=10.0*cm))

    # v52: Inconsistências de agenda docente permanecem calculadas e exportadas para governança
    # na aba "V52 Conflitos Doc" do Excel analítico, mas não aparecem mais no relatório PDF final.
    # Página final: validação de consistência entre os números do relatório.
    story.append(PageBreak())
    story.append(P("Validação de consistência dos números", title))
    story.append(Spacer(1, 6))
    try:
        checagens_pdf_v53 = checagens_consistencia_v53()
    except Exception as exc:
        checagens_pdf_v53 = pd.DataFrame([{
            "Verificação": "Checagem não executada", "Valor A": "-", "Valor B": "-",
            "Diferença": "-", "Status": "NÃO AVALIADO", "Observação": str(exc),
        }])
    total_check_v53 = len(checagens_pdf_v53)
    div_check_v53 = int((checagens_pdf_v53["Status"] == "DIVERGENTE").sum()) if total_check_v53 else 0
    info_check_v53 = int((checagens_pdf_v53["Status"] == "INFORMATIVO").sum()) if total_check_v53 else 0
    story.append(P(
        f"{total_check_v53} verificações automáticas executadas | {total_check_v53 - div_check_v53 - info_check_v53} consistentes | "
        f"{div_check_v53} com divergência | {info_check_v53} informativas.",
        h2))
    story.append(Spacer(1, 5))

    def tbl_checagens_v53(df):
        t = tbl_wrapped(df, max_rows=40, font_size=6.6,
                        col_widths=[7.4*cm, 3.1*cm, 3.1*cm, 2.2*cm, 2.4*cm, 8.2*cm])
        try:
            estilos = []
            for i, status in enumerate(df["Status"].astype(str).tolist(), start=1):
                st = status.upper()
                if st == "OK":
                    cor = rl_colors.HexColor("#DCFCE7")
                elif st == "INFORMATIVO":
                    cor = rl_colors.HexColor("#E2E8F0")
                else:
                    cor = rl_colors.HexColor("#FEE2E2")
                estilos.append(("BACKGROUND", (4, i), (4, i), cor))
            t.setStyle(TableStyle(estilos))
        except Exception:
            pass
        return t

    story.append(tbl_checagens_v53(checagens_pdf_v53))
    story.append(Spacer(1, 6))
    story.append(P(
        "Como ler: cada linha compara dois caminhos de cálculo independentes sobre o mesmo indicador — "
        "por exemplo, o total de docentes considerados no início do relatório contra a soma do quadro por cargo. "
        "Status OK significa que os dois caminhos fecham. Status INFORMATIVO traz uma diferença esperada por regra de "
        "negócio, útil como contexto. Status DIVERGENTE indica diferença a tratar: a coluna "
        "Observação traz a causa provável e a diferença deve ser tratada antes da divulgação do documento. "
        "Diferenças esperadas por regra de negócio (como disciplinas ESTENDIDAS entrando nos dois blocos executivos) "
        "estão explicitadas na própria observação.", small))

    doc.build(story, onFirstPage=header_footer_v52, onLaterPages=header_footer_v52)
    return pdf_path


# ================================================================
# CHECAGEM DE CONSISTÊNCIA DOS NÚMEROS DO RELATÓRIO
# ================================================================
# Motivação: indicadores do topo do relatório (ex.: "docentes considerados")
# são calculados por um caminho e os quadros detalhados por outro. Quando os
# dois caminhos não fecham, o leitor executivo perde confiança no documento.
# Cada verificação abaixo compara dois cálculos independentes e classifica em
# OK ou DIVERGENTE, com observação explicando a causa provável quando possível.

TOL_CHECAGEM_QTD = 0.0      # contagens têm de fechar exatamente
TOL_CHECAGEM_HORAS = 0.5    # horas admitem arredondamento
TOL_CHECAGEM_VALOR = 1.0    # valores financeiros admitem centavos

def _v53_num(x, default=0.0):
    try:
        v = float(x)
        return v if np.isfinite(v) else default
    except Exception:
        return default

def _v53_soma(df, coluna, excluir_total=True):
    """Soma uma coluna do quadro ignorando a linha TOTAL GERAL."""
    if not isinstance(df, pd.DataFrame) or df.empty or coluna not in df.columns:
        return 0.0
    d = df.copy()
    if excluir_total:
        # A linha de total é identificada pelo texto em qualquer coluna, sem depender
        # do dtype (pandas recente usa dtype "str" em vez de "object").
        mask_total = pd.Series(False, index=d.index)
        for c in d.columns:
            try:
                mask_total = mask_total | d[c].astype(str).str.upper().str.strip().eq("TOTAL GERAL")
            except Exception:
                continue
        d = d[~mask_total]
    return float(pd.to_numeric(d[coluna], errors="coerce").fillna(0).sum())

def _v53_soma_cargos_docentes(df, coluna):
    """Soma a coluna apenas nos cargos que compõem a métrica de docentes."""
    if not isinstance(df, pd.DataFrame) or df.empty or coluna not in df.columns or "cargo" not in df.columns:
        return 0.0
    d = df[mascara_cargos_considerados(df["cargo"]).values]
    return float(pd.to_numeric(d[coluna], errors="coerce").fillna(0).sum())

def _v53_soma_cargos_rpa(df, coluna):
    """Soma a coluna apenas nas linhas de cargo RPA."""
    if not isinstance(df, pd.DataFrame) or df.empty or coluna not in df.columns or "cargo" not in df.columns:
        return 0.0
    mask = df["cargo"].astype(str).map(_cargo_normalizado_v53).str.contains("RPA", na=False)
    return float(pd.to_numeric(df.loc[mask, coluna], errors="coerce").fillna(0).sum())

def _v53_diagnostico_headcount():
    """Explica por que a soma do headcount por cargo pode não bater com o total de pessoas."""
    notas = []
    try:
        d = doc_ind_v52 if isinstance(doc_ind_v52, pd.DataFrame) else pd.DataFrame()
        if not d.empty and {"professor_key", "cargo"}.issubset(d.columns):
            escopo = d[mascara_cargos_considerados(d["cargo"]).values]
            cargos_por_pessoa = escopo.groupby("professor_key")["cargo"].nunique() if not escopo.empty else pd.Series(dtype=int)
            multi = int((cargos_por_pessoa > 1).sum()) if len(cargos_por_pessoa) else 0
            excedente = int((cargos_por_pessoa - 1).clip(lower=0).sum()) if len(cargos_por_pessoa) else 0
            if excedente:
                notas.append(f"{br_int(multi)} docente(s) aparecem em mais de um cargo docente, somando +{br_int(excedente)} no quadro por cargo")
    except Exception as exc:
        notas.append(f"diagnóstico indisponível ({exc})")
    if not notas:
        notas.append("Escopo: professor de ensino superior EAD, presencial e regente. RPA é verificado na linha própria.")
    return "; ".join(notas)

def checagens_consistencia_v53() -> pd.DataFrame:
    """Executa as verificações cruzadas e devolve o quadro de validação."""
    g = globals()
    linhas = []

    def add(verificacao, valor_a, valor_b, rotulo_a="Valor A", rotulo_b="Valor B",
            tolerancia=TOL_CHECAGEM_QTD, observacao="", formato="int", informativo=False):
        # informativo=True: a diferença é esperada por regra de negócio e serve
        # como leitura de contexto, não como erro a corrigir.
        a, b = _v53_num(valor_a), _v53_num(valor_b)
        dif = a - b
        if formato == "money":
            fmt = br_money
        elif formato == "horas":
            fmt = lambda v: f"{br_int(v)}h"
        else:
            fmt = br_int
        linhas.append({
            "Verificação": verificacao,
            "Valor A": f"{rotulo_a}: {fmt(a)}",
            "Valor B": f"{rotulo_b}: {fmt(b)}",
            "Diferença": fmt(dif) if abs(dif) > 0 else fmt(0),
            "Status": "INFORMATIVO" if informativo else ("OK" if abs(dif) <= tolerancia else "DIVERGENTE"),
            "Observação": observacao,
        })

    def tenta(fn):
        try:
            fn()
        except Exception as exc:
            linhas.append({
                "Verificação": "Verificação não executada",
                "Valor A": "-", "Valor B": "-", "Diferença": "-",
                "Status": "NÃO AVALIADO", "Observação": f"Falha ao calcular: {exc}",
            })

    # --- Disciplinas previstas x realizadas ---------------------------------
    tenta(lambda: add(
        "Disciplinas previstas no Mapa fecham com realizadas + não realizadas",
        g.get("qtd_planejado"), _v53_num(g.get("qtd_conforme")) + _v53_num(g.get("qtd_faltantes")),
        "Total previsto", "Realizadas + não realizadas",
        observacao="Se divergir, há disciplina prevista classificada fora das duas situações."))

    tenta(lambda: add(
        "Disciplinas realizadas (aulas distintas) fecham com aderentes + sem previsão",
        g.get("realizado_key")["chave_conformidade"].nunique() if isinstance(g.get("realizado_key"), pd.DataFrame) and "chave_conformidade" in g.get("realizado_key").columns else g.get("qtd_realizado"),
        _v53_num(g.get("qtd_conforme")) + _v53_num(g.get("qtd_extras")),
        "Aulas distintas realizadas", "Aderentes + sem previsão",
        observacao="Comparação por aula distinta (disciplina + ordem + dia + horário), mesmo critério usado na conformidade."))

    tenta(lambda: add(
        "Registro: linhas lançadas x aulas distintas (repetições de chave)",
        len(g.get("realizado_key")) if isinstance(g.get("realizado_key"), pd.DataFrame) else 0,
        g.get("realizado_key")["chave_conformidade"].nunique() if isinstance(g.get("realizado_key"), pd.DataFrame) and "chave_conformidade" in g.get("realizado_key").columns else 0,
        "Linhas do Registro", "Aulas distintas",
        informativo=True,
        observacao="Diferença = linhas que repetem a mesma aula (sinergia entre cursos ou lançamento duplicado). Por regra do módulo todas as linhas entram nos cálculos e as duplicidades ficam em governança."))

    tenta(lambda: add(
        "Quadro por ordem (previsto) fecha com o total de disciplinas previstas",
        _v53_soma(g.get("ordem_comp_v52"), "planejado"), g.get("qtd_planejado"),
        "Soma do quadro", "Total previsto",
        observacao="Confere a página de quantitativos executivos contra a capa."))

    tenta(lambda: add(
        "Quadro por ordem (realizado) fecha com o total de disciplinas realizadas",
        _v53_soma(g.get("ordem_comp_v52"), "realizado"), g.get("qtd_realizado"),
        "Soma do quadro", "Total realizado",
        observacao="Confere a página de quantitativos executivos contra o Registro."))

    tenta(lambda: add(
        "Realizado por dia da semana fecha com o total de disciplinas realizadas",
        _v53_soma(g.get("dia_realizado_v52"), "qtd_disciplinas"), g.get("qtd_realizado"),
        "Soma por dia", "Total realizado",
        observacao="Detalhamento 2 contra o total do Registro."))

    # --- Mapas de calor -----------------------------------------------------
    tenta(lambda: add(
        "Mapa de calor do previsto fecha com as ocorrências por bloco executivo",
        float(np.nansum(g.get("heat_plan_v52").values)) if isinstance(g.get("heat_plan_v52"), pd.DataFrame) and not g.get("heat_plan_v52").empty else 0,
        len(g.get("planejado_periodos_v52")) if isinstance(g.get("planejado_periodos_v52"), pd.DataFrame) else 0,
        "Soma das células", "Ocorrências por bloco",
        observacao="ESTENDIDA entra nos dois blocos executivos, por isso a soma supera o total de disciplinas."))

    tenta(lambda: add(
        "Mapa de calor do realizado fecha com as ocorrências por bloco executivo",
        float(np.nansum(g.get("heat_real_v52").values)) if isinstance(g.get("heat_real_v52"), pd.DataFrame) and not g.get("heat_real_v52").empty else 0,
        len(g.get("realizado_periodos_v52")) if isinstance(g.get("realizado_periodos_v52"), pd.DataFrame) else 0,
        "Soma das células", "Ocorrências por bloco",
        observacao="ESTENDIDA entra nos dois blocos executivos, por isso a soma supera o total de disciplinas."))

    # --- Docentes -----------------------------------------------------------
    tenta(lambda: add(
        "Docentes considerados fecham com ativos na base + alocados fora da base ativa",
        g.get("total_docentes_banco_unicos"),
        _v53_num(g.get("qtd_docentes_ativos_considerados_v53")) + _v53_num(g.get("docentes_considerados_fora_base_ativa_v53")),
        "Considerados", "Ativos + alocados fora",
        observacao="Ambos os lados usam o mesmo escopo de cargos docentes (RPA fora)."))

    tenta(lambda: add(
        "Docentes considerados fecham com o headcount dos cargos docentes (Detalhamento 4)",
        g.get("total_docentes_banco_unicos"), _v53_soma_cargos_docentes(g.get("doc_cargo_total_v52"), "headcount"),
        "Pessoas únicas", "Headcount dos cargos docentes",
        observacao=_v53_diagnostico_headcount()))

    tenta(lambda: add(
        "RPA: headcount do quadro corresponde às contratações registradas",
        _v53_soma_cargos_rpa(g.get("doc_cargo_total_v52"), "headcount"),
        len(g.get("rpa_registro_v52")) if isinstance(g.get("rpa_registro_v52"), pd.DataFrame) else 0,
        "Headcount RPA no quadro", "Contratações RPA no Registro",
        observacao="RPA é contratação, não pessoa do quadro: fica fora da métrica de docentes considerados por regra de negócio."))

    tenta(lambda: add(
        "Quadro de docentes: linha TOTAL GERAL fecha com a soma das linhas por cargo",
        float(pd.to_numeric(g.get("doc_cargo_total_v52").loc[g.get("doc_cargo_total_v52")["cargo"].astype(str).str.upper().eq("TOTAL GERAL"), "ch_letiva_alocavel"], errors="coerce").fillna(0).sum()),
        _v53_soma(g.get("doc_cargo_total_v52"), "ch_letiva_alocavel"),
        "TOTAL GERAL", "Soma das linhas",
        tolerancia=TOL_CHECAGEM_HORAS, formato="horas",
        observacao="Valida a linha de total do quadro de capacidade docente."))

    # --- Horas --------------------------------------------------------------
    tenta(lambda: add(
        "Horas previstas: quadro por bloco fecha com o quadro por cluster",
        _v53_soma(g.get("horas_periodo_v52"), "planejado_horas", excluir_total=False),
        _v53_soma(g.get("cluster_v52"), "horas_planejado", excluir_total=False),
        "Por bloco", "Por cluster",
        tolerancia=TOL_CHECAGEM_HORAS, formato="horas",
        observacao="Detalhamento 1 contra Detalhamento 1.1."))

    tenta(lambda: add(
        "Horas realizadas: quadro por bloco fecha com o quadro por cluster",
        _v53_soma(g.get("horas_periodo_v52"), "realizado_horas", excluir_total=False),
        _v53_soma(g.get("cluster_v52"), "horas_realizado", excluir_total=False),
        "Por bloco", "Por cluster",
        tolerancia=TOL_CHECAGEM_HORAS, formato="horas",
        observacao="Detalhamento 1 contra Detalhamento 1.1."))

    # --- RPA ----------------------------------------------------------------
    tenta(lambda: add(
        "RPA: total estimado fecha com valor registrado + estimativa sem valor",
        g.get("rpa_total_estimado_v52"),
        _v53_num(g.get("valor_rpa_registro_total_v52")) + _v53_num(g.get("rpa_estimado_sem_valor_v52")),
        "Total estimado", "Registrado + estimado",
        tolerancia=TOL_CHECAGEM_VALOR, formato="money",
        observacao="Card da capa contra o Detalhamento 3.2."))

    tenta(lambda: add(
        "RPA previsto por ordem fecha com o RPA sinalizado no Mapa",
        _v53_soma(g.get("rpa_ordem_comp_v52"), "rpa_planejado"),
        len(g.get("rpa_mapa_v52")) if isinstance(g.get("rpa_mapa_v52"), pd.DataFrame) else 0,
        "Soma por ordem", "RPA no Mapa",
        observacao="Quantitativos executivos contra a base de RPA planejado."))

    tenta(lambda: add(
        "RPA realizado por ordem fecha com o RPA no Registro",
        _v53_soma(g.get("rpa_ordem_comp_v52"), "rpa_realizado"),
        len(g.get("rpa_registro_v52")) if isinstance(g.get("rpa_registro_v52"), pd.DataFrame) else 0,
        "Soma por ordem", "RPA no Registro",
        observacao="Quantitativos executivos contra a base de RPA executado."))

    tenta(lambda: add(
        "RPA com valor + sem valor fecha com o total de RPA no Registro",
        _v53_num(g.get("rpa_simulacao_v52").loc[0, "rpa_com_valor"]) + _v53_num(g.get("rpa_simulacao_v52").loc[0, "rpa_sem_valor"]) if isinstance(g.get("rpa_simulacao_v52"), pd.DataFrame) and not g.get("rpa_simulacao_v52").empty else 0,
        len(g.get("rpa_registro_v52")) if isinstance(g.get("rpa_registro_v52"), pd.DataFrame) else 0,
        "Com valor + sem valor", "RPA no Registro",
        observacao="Detalhamento 3.2 contra a base de RPA executado."))

    return pd.DataFrame(linhas, columns=["Verificação", "Valor A", "Valor B", "Diferença", "Status", "Observação"])

checagens_v53 = checagens_consistencia_v53()
divergencias_v53 = int((checagens_v53["Status"] == "DIVERGENTE").sum()) if not checagens_v53.empty else 0
print("=" * 80)
print(f"Checagem de consistência: {len(checagens_v53)} verificações | {divergencias_v53} fora do esperado.")
if divergencias_v53:
    for _, _r in checagens_v53[checagens_v53["Status"] == "DIVERGENTE"].iterrows():
        print(f"  [{_r['Status']}] {_r['Verificação']} -> {_r['Valor A']} x {_r['Valor B']} (dif. {_r['Diferença']}) | {_r['Observação']}")
print("=" * 80)

pdf_v52_out = OUTPUT_DIR / f"SGA_Relatorio_Executivo_Modulo_{modulo_alvo}_{timestamp}.pdf"
try:
    gerar_pdf_v52(pdf_v52_out)
    print(f"PDF executivo gerado em: {pdf_v52_out.resolve()}")
except Exception as exc:
    print(f"Falha ao gerar PDF executivo: {exc}")

print(f"Abas analíticas anexadas em: {excel_v52_out.resolve()}")
print(f"HTML executivo gerado em: {html_v52_out.resolve()}")

In [ ]:
# ================================================================
# Manifesto de arquivos gerados
# ================================================================
print("\n" + "="*80)
print("SGA concluído. Arquivos gerados em:", OUTPUT_DIR.resolve())
for _p in [locals().get("excel_out"), locals().get("html_v52_out"), locals().get("pdf_v52_out")]:
    try:
        if _p and Path(_p).exists():
            print(" -", Path(_p).resolve())
    except Exception:
        pass
print("="*80 + "\n")
